# BTP v5 — Hierarchical Optical Transient Classification
## TNS vs ALeRCE ground truth, compared with the de Soto et al. (2024) agreement framework

Two-stage hierarchical classification of ZTF optical transients —
**Stage 1:** AGN / SNe / TDE / Stellar Flare, **Stage 2 (SNe):** Ia / Ib / Ic / II / SLSN —
run twice, once against **TNS spectroscopic labels** and once against **ALeRCE's own
classifier output**, then compared with the expected-vs-actual agreement framework of
de Soto et al. (2024, Superphot+, arXiv:2403.07975). Three models at both stages, both
tracks: Logistic Regression, Random Forest, Bagging Classifier.

**Preserved from v4:** TNS/BTS acquisition, the Villar (2019) fit, the quality gate, the
retry-until-target loop, the feature vocabulary, CV / bootstrap CIs, all diagnostics.

**New:** a live-verified ALeRCE crosswalk; an ALeRCE-native acquisition so the three
pools are genuinely distinct; host offset; Townsend et al. (2026) augmentation; agreement
at both stages; the diagnostics on both tracks; calibration.

---

### Findings from checking ALeRCE live, stated before anything depends on them

1. **TDE lives only in the 2025 classifier.** `lc_classifier` (hierarchical_rf_1.1.0) has
   15 classes and **no TDE**. `lc_classifier_BHRF_forced_phot` 2.1.0 has 21, including
   TDE. Section 3.1 wants ALeRCE's TDE output cross-checked, so BHRF is the primary source.
2. **Probabilities are stacked across versions.** `query_probabilities` returns every
   version it has run for an object in one frame (one classifier summed to 4.0 across four
   versions on ZTF18abtmbaz). Every lookup is keyed on (name, version).
3. **The brief's stated reason for merging SN IIn is not what the live API shows.** The
   brief says ALeRCE's 2025 SN output has no separate IIn. The live BHRF transient branch
   is `[SNIa, SESN, SNII, SNIIn, SLSN, TDE]` — it **does** separate IIn, and names the
   stripped-envelope class `SESN`. The merge stands on its own reason (sample size), and
   ALeRCE's SNIIn is merged symmetrically so both tracks collapse identically.
4. **ALeRCE's "stochastic" branch is not our AGN class**, as the brief warned: it also
   holds YSO and CV/Nova, which are Galactic. Only QSO/AGN/Blazar map to AGN.
5. **ALeRCE returns the least confident objects first.** At p ≥ 0.5, `query_objects`
   answers 0.500, 0.500, 0.501… and large classes outgrow a page (AGN+QSO+Blazar: 62,645
   objects). The native pool is a uniform random sample over the whole set above the cut.
6. **ALeRCE's confident set is tiny for the rare classes:** 33 SESN, 43 SLSN and 23 TDE in
   its entire BHRF catalogue at p ≥ 0.5 (checked 2026-09-29). That bounds pool 2.
7. **The ALeRCE track cannot contain our stellar flares.** They are TESS objects; ALeRCE
   classifies ZTF alerts. The overlap pool has no flare class.

**On ordering.** The brief's Section 9 puts the ALeRCE label step (its Phase 2B) before
preprocessing and acquisition. It runs after acquisition here: ALeRCE is queried per ZTF
object id, and those ids only exist once acquisition has resolved them. Every step cites
its phase in the brief.

# STEP 1 — Setup and configuration  *(brief Phase 1)*

In [ ]:
# v5 adds astro-ghost (host offset, Section 4.1) and sncosmo (template
# K-corrections for augmentation, Section 5).
!pip install -q alerce lightkurve astroquery pandas numpy matplotlib tqdm requests scikit-learn astro-ghost sncosmo tabulate

import os, io, json, time, zipfile, requests, gc, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from alerce.core import Alerce
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

BASE_DIR    = '/content/drive/MyDrive/BTP_optical_transients/data/raw'
FEATURE_DIR = '/content/drive/MyDrive/BTP_optical_transients/data/features'
for sub in ['sn_ia','sn_ib','sn_ic','sn_ii','slsn','agn','tde','stellar_flares','tns_catalog','bts']:
    os.makedirs(os.path.join(BASE_DIR, sub), exist_ok=True)
os.makedirs(FEATURE_DIR, exist_ok=True)

alerce = Alerce()

# ----------------------------- targets -----------------------------
N_PER_CLASS      = 150   # AGN / stellar flares

# TDE gets its own, smaller target. Not a concession - arithmetic. TNS lists ~153 TDEs
# in total and the ZTF BTS TDE page lists 78 carrying ZTF object ids; the union is
# ~160 objects in the entire public record, and not all survive a quality gate. v3
# asked for 150 and reached 73. Asking for a number the universe can supply, and
# handling the residual imbalance with class_weight='balanced', is the honest option.
N_TDE_TARGET     = 90

# Two numbers, deliberately separated:
#
#   N_HEADLINE_SN_SUBTYPE  the balanced set the headline Stage 2 result is reported on.
#                          Fixed at 30, as specified for the project.
#   N_ACQUIRE_SN_SUBTYPE   how many are actually collected per subtype. Anything above
#                          the headline number is not wasted: it is what makes the
#                          learning curve, the ablation and the class-scheme comparison
#                          in Phase 6 measurable, and it is the one knob that moves the
#                          5-way result towards 85-90%.
#
# Measured on the PLAsTiCC benchmark under this exact pipeline: a balanced 30/class
# 5-way split tops out near 0.66 no matter what; the same features on ~640 objects at
# natural class proportions reach 0.86 accuracy (macro-F1 0.69), and 0.91 / 0.85 once
# Ib and Ic are merged. Collecting more is the only thing that closes that gap.
#
# 30  -> fastest, reproduces the project spec exactly, curve is a single point
# 150 -> recommended, a few hours, makes every Phase 6 diagnostic meaningful
# 400 -> overnight, gets the 5-way scheme into the target band at natural proportions
N_HEADLINE_SN_SUBTYPE = 30
N_ACQUIRE_SN_SUBTYPE  = 30     # <-- raise this to 150 for the recommended run

N_PER_SN_SUBTYPE = N_ACQUIRE_SN_SUBTYPE   # what the acquisition loop targets

# --------------------------- quality gate ---------------------------
# Tightened in v4. Each threshold below came from a measured ablation, not from taste.
# Together they were worth +8.4 accuracy points at unchanged sample size:
#   v3 gate 0.635 -> +span>=60d 0.674 -> +both bands 0.679 -> +converged fit 0.719
#
# The cost is a higher rejection rate, which the retry loop absorbs by consuming more
# candidates. The Phase 2 pools are 12x-40x the target, so there is headroom.
QG = dict(
    min_det_total     = 15,     # detections across g+r
    min_det_best_band = 8,      # detections in the better-sampled band
    min_peak_snr      = 8.0,    # peak flux / its uncertainty
    max_chisq_dof     = 20.0,   # the Villar fit must actually describe the data
    require_redshift  = True,   # needed for peak absolute magnitude
    require_bracket   = True,   # points before AND after peak, not a one-sided ramp
    min_days_prepeak  = 5.0,    # v3 used 2.0
    min_days_postpeak = 30.0,   # v3 used 5.0 - the single loosest thing in v3
    min_obs_span      = 60.0,   # NEW: total baseline. A IIP plateau runs ~100 days;
                                #      5 days of post-peak data cannot see one.
    require_two_bands = True,   # NEW: no colour -> no colour-evolution features
)

# How many candidates to hold in the pool per class. Deep pools are what make the
# retry loop able to reach the target under a strict quality gate.
POOL_OVERSAMPLE = 40    # pool size = target * this, capped by what exists

RANDOM_STATE = 42
SN_SUBTYPES  = ['SN_Ia', 'SN_Ib', 'SN_Ic', 'SN_II', 'SLSN']

print('data root  :', BASE_DIR)
print('acquire    :', N_ACQUIRE_SN_SUBTYPE, 'per SN subtype /', N_PER_CLASS, 'per coarse class')
print('headline   :', N_HEADLINE_SN_SUBTYPE, 'per SN subtype (balanced, as specified)')
print('TDE target :', N_TDE_TARGET, '(TNS+BTS union is ~160 objects in total)')
print('quality gate:', QG)

### TNS credentials

v4 defined a `_get_secret()` helper and then overrode it with the bot id and API key as
literal strings, so the helper never ran and the key travelled with every copy of the
notebook. This cell uses the helper. Put `TNS_BOT_ID`, `TNS_BOT_NAME` and `TNS_API_KEY`
in Colab's Secrets panel (key icon, left sidebar).

In [ ]:
# TNS credentials — read at run time, never hard-coded.
#
# v2 had the bot id and API key written into this cell in plain text. That key is now
# public in every copy of that notebook; rotate it on the TNS website before running
# this. In Colab put the new values in the Secrets panel (key icon, left sidebar) as
# TNS_BOT_ID, TNS_BOT_NAME and TNS_API_KEY with notebook access enabled.
import os

def _get_secret(name, prompt=None):
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass
    val = os.environ.get(name)
    if val:
        return val
    import getpass
    return getpass.getpass(prompt or f'{name}: ')


TNS_BOT_ID   = _get_secret('TNS_BOT_ID')
TNS_BOT_NAME = _get_secret('TNS_BOT_NAME')
TNS_API_KEY  = _get_secret('TNS_API_KEY')

TNS_MARKER = ('tns_marker{"tns_id":' + str(TNS_BOT_ID) +
              ',"type": "bot", "name":"' + TNS_BOT_NAME + '"}')
HEADERS = {'user-agent': TNS_MARKER}

assert TNS_API_KEY and TNS_BOT_ID, 'TNS credentials are not set'
print(f'TNS credentials loaded for bot id {TNS_BOT_ID} (key not shown)')

### alerce client version — checked, not assumed

v1.x and v2.x of the client differ: v2.x needs an explicit `survey='ztf'`, v1.x rejects
it. The version is printed so a re-run cannot silently change call semantics.

In [ ]:
# v5 configuration
# ---------------------------------------------------------------
# Inlined verbatim from btp5/config.py, which the repository's test suite
# (tests5/) exercises. Edit it there and rebuild this notebook with
# tools/build_notebook_v5.py rather than editing it here.
# ---------------------------------------------------------------

"""
Constants for v5. v4's values are carried over unchanged; v5 additions are marked.
"""

# ---------------------------------------------------------------- v4, unchanged
N_PER_CLASS = 150          # AGN / stellar flares
N_TDE_TARGET = 90          # TNS lists ~153 TDEs total; ZTF BTS ~78 with ZTF ids.
                           # Asking for what the universe can supply, not 150.
N_HEADLINE_SN_SUBTYPE = 30
N_ACQUIRE_SN_SUBTYPE = 30  # raise to 150 for the recommended run
N_PER_SN_SUBTYPE = N_ACQUIRE_SN_SUBTYPE

QG = dict(
    min_det_total=15,
    min_det_best_band=8,
    min_peak_snr=8.0,
    max_chisq_dof=20.0,
    require_redshift=True,
    require_bracket=True,
    min_days_prepeak=5.0,
    min_days_postpeak=30.0,
    min_obs_span=60.0,
    require_two_bands=True,
)

POOL_OVERSAMPLE = 40
RANDOM_STATE = 42

SN_SUBTYPES = ['SN_Ia', 'SN_Ib', 'SN_Ic', 'SN_II', 'SLSN']
COARSE_CLASSES = ['AGN', 'SNe', 'TDE', 'stellar_flare']

LIGHTCURVE_COLS = ['log_A', 'beta', 'log_gamma', 'log_tau_rise', 'log_tau_fall', 'log_chisq',
                   'gr_log_A', 'gr_dt0', 'gr_log_tau_rise', 'gr_log_tau_fall', 'gr_dbeta',
                   'gr_log_gamma', 'has_color',
                   'duration_half', 'rise_half', 'fade_half', 'n_det', 'max_snr', 'obs_span']
PHYSICAL_COLS = ['redshift', 'peak_abs_mag']
CATALOGUE_COLS = ['bts_peakabs', 'bts_rise', 'bts_fade', 'bts_duration', 'from_bts']

# ---------------------------------------------------------------- v5 additions
# Section 4.1: host offset via astro-ghost. Same missing-data convention as
# color_g_r/has_color in v4 — NaN plus an explicit presence flag.
HOST_COLS = ['host_offset_arcsec', 'host_offset_norm', 'has_host_match']

FEATURE_COLS = LIGHTCURVE_COLS + PHYSICAL_COLS + CATALOGUE_COLS + HOST_COLS

# Section 7: three models in the headline comparison. HistGradientBoosting and
# Bagging(SVM) stay available for diagnostics only — v4 measured all five converging
# to within ~1 point at Stage 1, so the extra two add noise to the headline table.
HEADLINE_MODELS = ['Logistic Regression', 'Random Forest', 'Bagging Classifier']
DIAGNOSTIC_ONLY_MODELS = ['Bagging (SVM)', 'HistGradientBoosting']

# Section 3.2: ALeRCE top-1 confidence cut. 0.5 matches the threshold Superphot+
# (de Soto et al. 2024) used when building their own ALeRCE-labelled comparison set.
ALERCE_CONF_THRESHOLD = 0.5

# Section 3.3 pool 2: objects acquired FROM ALeRCE, per group (AGN, SN_Ia, SN_Ibc,
# SN_II, SLSN, TDE, stellar_flare). A ceiling applies regardless: at p>=0.5 the live
# BHRF catalogue held only 33 SESN, 43 SLSN and 23 TDE in total (checked 2026-09-29).
N_ALERCE_NATIVE_PER_GROUP = 30

# Section 3.2/3.1: which ALeRCE classifier supplies the labels.
#
# Verified live against query_classifiers() on 2026-09-25 — NOT hardcoded from
# memory, and re-verified at run time by alerce_labels.fetch_taxonomy():
#
#   lc_classifier (hierarchical_rf_1.1.0)      15 classes, NO TDE
#   lc_classifier_BHRF_forced_phot (2.1.0)     21 classes, INCLUDING TDE
#
# The brief's Section 3.1 wants ALeRCE's 2025 TDE output cross-checked, which only
# the BHRF family provides, so that is the primary. The classic lc_classifier is kept
# as a sensitivity check because it is the longer-established model.
ALERCE_PRIMARY_CLASSIFIER = 'lc_classifier_BHRF_forced_phot'
ALERCE_PRIMARY_VERSION = '2.1.0'
ALERCE_SECONDARY_CLASSIFIER = 'lc_classifier'

# Section 5: ZTF empirical noise model, Townsend et al. 2026 (arXiv:2602.13036),
# their published fitted constants.
NOISE_MODEL = dict(e_b=18.0, m=0.04, c=4.7, delta=-0.006)   # in microjansky (see augment.py)
AUGMENT_CLASSES = ['SLSN', 'SN_Ib', 'SN_Ic', 'TDE']
AUGMENT_COPIES_PER_PARENT = 2   # Townsend's 'unbalanced' strategy: constant copies per
                                # object, applied to the four small classes only
AUGMENT_DZ_MAX = 0.1            # Townsend Sect. 3.3: z_sim in [z_true, z_true + 0.1]
AUGMENT_Z_SCALE = 2             # p(z) ~ z^2, constant volumetric rate
AUGMENT_SUBSAMPLING_RATE = 1.0  # off. Townsend tested 0.7/0.9/1.0; their chosen value
                                # (Table 3) was not verified here, so none is assumed

In [ ]:
# Section 3.2 — ALeRCE label track
# ---------------------------------------------------------------
# Inlined verbatim from btp5/alerce_labels.py, which the repository's test suite
# (tests5/) exercises. Edit it there and rebuild this notebook with
# tools/build_notebook_v5.py rather than editing it here.
# ---------------------------------------------------------------

"""
Section 3.2 — the ALeRCE-labelled ground-truth track.

Three things here are deliberate, and each exists because the live API does not
behave the way a reasonable person would assume:

1. **The taxonomy is fetched at run time, never hardcoded.** The brief says so, and
   checking proved it right: ALeRCE's TDE class does NOT exist in the classic
   `lc_classifier` (hierarchical_rf_1.1.0, 15 classes). It lives only in the 2025
   `lc_classifier_BHRF_forced_phot` (2.1.0, 21 classes), whose SN branch is
   [SNIa, SESN, SNII, SNIIn, SLSN, TDE] — note `SESN`, not `SNIbc`, and note that a
   separate `SNIIn` DOES exist there.

2. **Probabilities must be keyed on (classifier_name, classifier_version), never name
   alone.** `query_probabilities` returns every version it has ever run for an object
   stacked in one frame. Measured on ZTF18abtmbaz: grouping by name alone makes
   `LC_classifier_ATAT_forced_phot(beta)` sum to 4.0 across four versions, and the
   top-1 row is then whichever version happened to score highest — a silent label
   corruption, not an error.

3. **The crosswalk is constructed against the fetched class list and printed**, and a
   class the live taxonomy contains but the mapping does not is reported rather than
   silently dropped.
"""

import numpy as np
import pandas as pd


# --------------------------------------------------------------------------
# Live taxonomy
# --------------------------------------------------------------------------

def client_version():
    """Which alerce client is installed. v1.x and v2.x differ: v2.x requires the
    explicit survey='ztf' argument or warns/raises."""
    try:
        import importlib.metadata as md
        return md.version('alerce')
    except Exception:  # pragma: no cover
        try:
            import alerce
            return getattr(alerce, '__version__', 'unknown')
        except Exception:
            return 'not installed'


def _survey_kwargs(version=None):
    """v2.x wants survey='ztf'; v1.x does not accept it."""
    v = version or client_version()
    major = str(v).split('.')[0]
    return {'survey': 'ztf'} if major.isdigit() and int(major) >= 2 else {}


def fetch_taxonomy(alerce, verbose=True):
    """
    Live classifier catalogue. Returns a DataFrame with one row per
    (classifier_name, classifier_version) and its class list.

    Never assume this matches what any document says — ALeRCE's taxonomy has changed
    at least once already (the 2025 TDE addition) and the version strings reported
    here do not always match the ones attached to per-object probabilities.
    """
    kw = _survey_kwargs()
    df = alerce.query_classifiers(format='pandas', **kw)
    if verbose:
        print(f'alerce client {client_version()} | {len(df)} classifier entries live')
        for _, r in df.iterrows():
            classes = list(r['classes'])
            print(f"  {r['classifier_name']:<42} {str(r['classifier_version']):<32} "
                  f"{len(classes):>2} classes")
    return df


def fetch_classes(alerce, classifier_name, classifier_version, verbose=True):
    """The class list for one classifier, straight from the API."""
    kw = _survey_kwargs()
    try:
        res = alerce.query_classes(classifier_name, classifier_version,
                                   format='pandas', **kw)
        # The column is 'name' on client 2.3.1 and 'class_name' on some other
        # versions; accept either rather than guessing, and fail loudly if neither
        # is present instead of silently returning the column headers as classes.
        col = next((c for c in ('name', 'class_name', 'class') if c in res.columns), None)
        if col is None:
            raise KeyError(f'no class-name column in query_classes output: '
                           f'{list(res.columns)}')
        classes = sorted(res[col].astype(str).unique())
    except Exception as e:
        if verbose:
            print(f'query_classes failed ({type(e).__name__}); falling back to the '
                  f'catalogue listing')
        tax = fetch_taxonomy(alerce, verbose=False)
        row = tax[(tax['classifier_name'] == classifier_name)
                  & (tax['classifier_version'].astype(str) == str(classifier_version))]
        classes = sorted(list(row.iloc[0]['classes'])) if len(row) else []
    if verbose:
        print(f'{classifier_name} [{classifier_version}]: {len(classes)} classes')
        print('  ' + ', '.join(classes))
    return classes


# --------------------------------------------------------------------------
# The crosswalk
# --------------------------------------------------------------------------

# ALeRCE native class -> our Stage 1 coarse class. Built against the LIVE class list
# (see build_crosswalk, which refuses to run against classes it has not seen).
#
# The brief warns specifically: do not assume ALeRCE's "stochastic" branch equals our
# AGN class. It does not. The BHRF stochastic branch is
# [Microlensing, QSO, AGN, Blazar, YSO, CV/Nova] — of which only QSO, AGN and Blazar
# are accreting supermassive black holes. YSO (young stellar objects) and CV/Nova
# (cataclysmic variables) are Galactic stellar systems; CV/Nova in particular is the
# closest ALeRCE analogue to our stellar-flare class, not to AGN. Microlensing is
# neither and is dropped.
COARSE_CROSSWALK = {
    # --- AGN family: accretion onto a supermassive black hole ---
    'AGN': 'AGN',
    'QSO': 'AGN',          # a quasar is a luminous AGN
    'Blazar': 'AGN',       # an AGN with its jet toward us

    # --- supernovae ---
    'SNIa': 'SNe',
    'SNIbc': 'SNe',        # classic lc_classifier naming
    'SESN': 'SNe',         # BHRF 2.1.0 naming for the same stripped-envelope class
    'SNIb': 'SNe',
    'SNIc': 'SNe',
    'SNII': 'SNe',
    'SNIIb': 'SNe',
    'SNIIn': 'SNe',
    'SLSN': 'SNe',

    # --- tidal disruption events ---
    'TDE': 'TDE',

    # --- Galactic stellar variability; CV/Nova is the flare-like analogue ---
    'CV/Nova': 'stellar_flare',
    'YSO': 'stellar_flare',
}

# Deliberately unmapped: everything periodic (LPV, E, EA, EB/EW, RRL, RRLab, RRLc,
# CEP, DSCT, RSCVn, Periodic-Other) and Microlensing. These have no counterpart in our
# 4-class scheme; objects whose top-1 lands here are EXCLUDED from the ALeRCE pool
# rather than forced into a class they do not belong to.
UNMAPPED_IS_EXCLUSION = True

# ALeRCE native SN class -> our Stage 2 fine class.
#
# Two asymmetries are stated rather than papered over:
#  * ALeRCE only ever emits a merged stripped-envelope class (SNIbc in the classic
#    model, SESN in BHRF 2.1.0). Our TNS scheme keeps Ib and Ic apart. So the
#    ALeRCE-labelled Stage 2 is necessarily coarser. The brief says not to merge
#    TNS's Ib+Ic to match, and that is respected.
#  * BHRF 2.1.0 DOES have a separate SNIIn, which our scheme merges into SN_II. We
#    merge ALeRCE's SNIIn into SN_II too, so both tracks apply the same merge and the
#    agreement matrices compare like with like.
FINE_CROSSWALK = {
    'SNIa': 'SN_Ia',
    'SNIbc': 'SN_Ibc',
    'SESN': 'SN_Ibc',
    'SNII': 'SN_II',
    'SNIIn': 'SN_II',      # merged, symmetrically with our TNS-side IIn -> SN_II
    'SNIIb': 'SN_II',
    'SLSN': 'SLSN',
}

# How our TNS fine labels collapse when compared against ALeRCE's coarser SN scheme.
TNS_TO_ALERCE_FINE = {
    'SN_Ia': 'SN_Ia', 'SN_Ib': 'SN_Ibc', 'SN_Ic': 'SN_Ibc',
    'SN_II': 'SN_II', 'SLSN': 'SLSN',
}


def build_crosswalk(live_classes, verbose=True):
    """
    Map the LIVE class list onto our schemes and print the result.

    Required, non-optional step per the brief: this must run and be inspected before
    any ALeRCE label is used downstream. Returns (table, unmapped).
    """
    rows = []
    for c in sorted(live_classes):
        coarse = COARSE_CROSSWALK.get(c)
        fine = FINE_CROSSWALK.get(c)
        rows.append({'alerce_class': c,
                     'our_coarse_class': coarse if coarse else '(excluded)',
                     'our_fine_class': fine if fine else '',
                     'used': bool(coarse)})
    table = pd.DataFrame(rows)
    unmapped = table.loc[~table['used'], 'alerce_class'].tolist()

    if verbose:
        print('ALeRCE -> our-scheme crosswalk, built against the LIVE class list:')
        print(table.to_string(index=False))
        print(f'\nMapped: {int(table["used"].sum())} / {len(table)} live classes.')
        if unmapped:
            print('Excluded (no counterpart in our 4-class scheme) — objects whose '
                  'top-1 lands here are dropped, not reassigned:')
            print('  ' + ', '.join(unmapped))
        known = set(COARSE_CROSSWALK) | set(FINE_CROSSWALK)
        stale = sorted(known - set(live_classes))
        if stale:
            print('\nNOTE: the crosswalk names classes the live taxonomy does NOT '
                  'contain (harmless, but means the mapping predates this taxonomy):')
            print('  ' + ', '.join(stale))
    return table, unmapped


# --------------------------------------------------------------------------
# Per-object labels
# --------------------------------------------------------------------------

def top_class(prob_df, classifier_name, classifier_version=None):
    """
    Top-1 class and probability for ONE classifier, keyed on name AND version.

    Keying on name alone is a silent bug: query_probabilities stacks every version it
    has run for the object, so the frame contains several complete probability
    distributions and the argmax crosses between them.
    """
    if prob_df is None or len(prob_df) == 0:
        return None
    sub = prob_df[prob_df['classifier_name'] == classifier_name]
    if classifier_version is not None:
        sub = sub[sub['classifier_version'].astype(str) == str(classifier_version)]
    if len(sub) == 0:
        return None
    if classifier_version is None and sub['classifier_version'].nunique() > 1:
        # Pick one version deterministically rather than mixing distributions.
        chosen = sorted(sub['classifier_version'].astype(str).unique())[-1]
        sub = sub[sub['classifier_version'].astype(str) == chosen]
    total = float(sub['probability'].sum())
    best = sub.loc[sub['probability'].idxmax()]
    return {'class_name': str(best['class_name']),
            'probability': float(best['probability']),
            'classifier_version': str(best['classifier_version']),
            'prob_sum': total,
            'n_classes': int(len(sub))}


def label_from_probabilities(prob_df, classifier_name, classifier_version=None,
                             threshold=0.5):
    """
    One object's ALeRCE-derived labels, or None if it does not qualify.

    Returns coarse/fine labels plus the raw native class and confidence, so every
    downstream table can show what the mapping actually did.
    """
    top = top_class(prob_df, classifier_name, classifier_version)
    if top is None:
        return None
    if top['probability'] < threshold:
        return None
    coarse = COARSE_CROSSWALK.get(top['class_name'])
    if coarse is None:
        return None                      # unmapped -> excluded, never reassigned
    return {'alerce_class': top['class_name'],
            'alerce_prob': top['probability'],
            'alerce_version': top['classifier_version'],
            'alerce_coarse': coarse,
            'alerce_fine': FINE_CROSSWALK.get(top['class_name'])}


def collapse_tns_fine(label):
    """Our TNS fine label in ALeRCE's coarser SN vocabulary (Ib/Ic -> Ibc)."""
    return TNS_TO_ALERCE_FINE.get(label, label)


def describe_probabilities(prob_df, classifier_name, classifier_version=None,
                           threshold=0.5):
    """
    Everything the notebook records per object: the raw top-1 (always, even below the
    cut — the Section 3.1 TDE cross-check needs what ALeRCE thinks of every TNS TDE,
    not just the confident ones) plus the thresholded, crosswalked labels (None when
    the object does not qualify for the ALeRCE-confident pool).
    """
    top = top_class(prob_df, classifier_name, classifier_version)
    if top is None:
        return None
    out = {'alerce_top1_class': top['class_name'], 'alerce_top1_prob': top['probability'],
           'alerce_version': top['classifier_version'],
           'alerce_class': None, 'alerce_prob': np.nan,
           'alerce_coarse': None, 'alerce_fine': None}
    lab = label_from_probabilities(prob_df, classifier_name, classifier_version, threshold)
    if lab is not None:
        out.update(lab)
    return out

In [ ]:
print('alerce client version:', client_version())
print('survey kwargs for this client:', _survey_kwargs())
print('Primary label source :', ALERCE_PRIMARY_CLASSIFIER, ALERCE_PRIMARY_VERSION)
print('Sensitivity check    :', ALERCE_SECONDARY_CLASSIFIER)
print('Confidence threshold :', ALERCE_CONF_THRESHOLD,
      '(the cut Superphot+ used for their own ALeRCE-labelled comparison set)')

# STEP 2 — TNS candidate pools  *(brief Phase 2A)*

In [ ]:
# TNS bulk public objects CSV — unchanged from v2, this part was right.
TNS_CSV_URL = "https://www.wis-tns.org/system/files/tns_public_objects/tns_public_objects.csv.zip"

resp = requests.post(TNS_CSV_URL, headers=HEADERS, data={'api_key': TNS_API_KEY}, timeout=180)
resp.raise_for_status()

zip_path = os.path.join(BASE_DIR, 'tns_catalog', 'tns_public_objects.csv.zip')
with open(zip_path, 'wb') as f:
    f.write(resp.content)
with zipfile.ZipFile(zip_path) as z:
    csv_name = z.namelist()[0]
    z.extractall(os.path.join(BASE_DIR, 'tns_catalog'))

tns_full = pd.read_csv(os.path.join(BASE_DIR, 'tns_catalog', csv_name), skiprows=1)
print(tns_full.shape)
tns_full['type'].value_counts().head(20)

### Why SN IIn stays merged into SN II

`classify_sn_subtype` (below, unchanged from v4) folds IIn into SN_II. The brief resolves
this and it is kept. **The justification is sample size**: TNS lists too few spectroscopic
IIn to support a class once the quality gate has run.

The brief also cites ALeRCE's 2025 taxonomy as having no separate IIn. Checked live, that
is not so: `lc_classifier_BHRF_forced_phot` 2.1.0's transient branch is
`[SNIa, SESN, SNII, SNIIn, SLSN, TDE]`. So the merge is not "what ALeRCE also does" — and
to keep the two tracks comparable, ALeRCE's `SNIIn` is merged into `SN_II` as well
(`FINE_CROSSWALK`, Step 1).

`SN Ib/c` (undecided between Ib and Ic) is excluded, as in v4: training on it teaches the
model that the two overlap, which is the confusion Stage 2 is trying to resolve.

In [ ]:
def classify_sn_subtype(t):
    """
    Raw TNS `type` string -> one of five SN subtype buckets, or None.
    Unchanged from v2. Sub-subtypes fold into their parent: 'SN Ia-91bg', 'SN Ia-pec',
    'SN Iax' -> SN_Ia; 'SN Ic-BL', 'SN Icn' -> SN_Ic; 'SN IIb/IIn/IIP/IIL' -> SN_II.
    SLSN-I/II/R are TNS top-level strings, so they are checked before the 'SN ' prefix.

    One deliberate exclusion added in v3: 'SN Ib/c' means the classifier could not
    decide between Ib and Ic. Training on it teaches the model that the two classes
    overlap, which is the exact confusion we are trying to resolve. v2 silently sent
    those 86 objects into SN_Ib.
    """
    t = str(t)
    if 'SLSN' in t:
        return 'SLSN'
    if not t.startswith('SN '):
        return None
    rest = t[3:]
    if rest.startswith('Ib/c') or rest.startswith('Ibc'):
        return None                       # ambiguous by construction — drop it
    if rest.startswith('Ia'):
        return 'SN_Ia'
    if rest.startswith('Ib'):
        return 'SN_Ib'
    if rest.startswith('Ic'):
        return 'SN_Ic'
    if rest.startswith('II'):
        return 'SN_II'
    return None


def extract_ztf_name(internal_names):
    if not isinstance(internal_names, str):
        return None
    for tok in internal_names.split(','):
        tok = tok.strip()
        if tok.startswith('ZTF'):
            return tok
    return None


tns_full['sn_subtype'] = tns_full['type'].apply(classify_sn_subtype)
tns_full['ztf_name']   = tns_full['internal_names'].apply(extract_ztf_name)

KEEP = ['name', 'ra', 'declination', 'redshift', 'type', 'discoverydate',
        'internal_names', 'ztf_name', 'sn_subtype']

print('TNS rows per SN subtype:')
print(tns_full['sn_subtype'].value_counts().to_string())
print('\nof which already carry a ZTF internal name (no cross-match needed):')
print(tns_full[tns_full['ztf_name'].notna()]['sn_subtype'].value_counts().to_string())

### TDE: now cross-checkable against ALeRCE  *(brief Section 3.1)*

The TDE pool below is v4's TNS + BTS union, unchanged. What is new is that ALeRCE's 2025
classifier emits a TDE class, so every TNS TDE can be checked against it — done in
Step 10, as the direct analogue of Superphot+ Table 3.

In [ ]:
# ---------------------------------------------------------------------------
# ZTF Bright Transient Survey Sample Explorer.
#
# Why this is here: v2's minority subtypes were pool-limited, not luck-limited. BTS is
# the ZTF-native, spectroscopically classified, magnitude-limited SN sample. Every row
# already carries the ZTF object ID, so `resolve_oid`'s ALeRCE cone search — which was
# a silent failure mode in v2 — is skipped entirely. It also ships peak absolute
# magnitude, rise, fade, duration and host-galaxy colours per object, which we keep as
# extra features and as a cross-check on our own fits.
#
# The catalogue is one HTTP GET, no credentials.
# ---------------------------------------------------------------------------
BTS_URL = 'https://sites.astro.caltech.edu/ztf/bts/explorer.php'

def fetch_bts(classstring='', quality=True, timeout=180):
    params = {'f': 's', 'format': 'csv', 'classstring': classstring}
    if quality:
        params['quality'] = 'y'
    r = requests.get(BTS_URL, params=params, timeout=timeout)
    r.raise_for_status()
    df = pd.read_csv(io.StringIO(r.text))
    df.columns = [c.strip() for c in df.columns]
    return df

# 'quality=y' is BTS's own cut: coverage around peak, clean reference frames,
# A_V < 1 mag, visible for a month post-peak. It is stricter than ours and is exactly
# the population that survives a parametric fit, so we take it.
bts_raw = fetch_bts(quality=True)
print('BTS rows (quality cut):', len(bts_raw))
print(bts_raw.columns.tolist())

bts = bts_raw.rename(columns={'ZTFID': 'ztf_name', 'IAUID': 'name', 'RA': 'ra',
                              'Dec': 'declination', 'type': 'type',
                              'redshift': 'redshift', 'peakabs': 'bts_peakabs',
                              'peakmag': 'bts_peakmag', 'rise': 'bts_rise',
                              'fade': 'bts_fade', 'duration': 'bts_duration',
                              'A_V': 'bts_av', 'b': 'bts_b'}).copy()
bts['sn_subtype'] = bts['type'].apply(classify_sn_subtype)

# BTS writes '>21.5' when the light curve is truncated — coerce to NaN, do not parse
# the inequality as a number.
for c in ['bts_rise', 'bts_fade', 'bts_duration', 'bts_peakabs', 'bts_peakmag',
          'redshift', 'bts_av']:
    if c in bts:
        bts[c] = pd.to_numeric(bts[c].astype(str).str.replace('>', '', regex=False),
                               errors='coerce')

bts.to_csv(os.path.join(BASE_DIR, 'bts', 'bts_quality.csv'), index=False)
print('\nBTS rows per SN subtype:')
print(bts['sn_subtype'].value_counts().to_string())

In [ ]:
# ---------------------------------------------------------------------------
# Union pool per SN subtype: BTS first (ZTF id + absolute magnitude already in hand),
# then TNS rows that BTS does not have. De-duplicated on the ZTF object id.
#
# This is the fix for "the minority classes cannot reach 30 survivors": the pool for
# SN_Ib and SN_Ic roughly doubles, and every BTS row skips the cross-match step that
# was the most common failure in v2.
# ---------------------------------------------------------------------------
def build_pool(subtype, target, oversample=POOL_OVERSAMPLE, random_state=RANDOM_STATE):
    b = bts[bts['sn_subtype'] == subtype].copy()
    b['source'] = 'BTS'
    b = b[['ztf_name', 'name', 'ra', 'declination', 'redshift', 'type', 'source',
           'bts_peakabs', 'bts_rise', 'bts_fade', 'bts_duration', 'bts_av']]

    t = tns_full[tns_full['sn_subtype'] == subtype].copy()
    t = t[~t['ztf_name'].isin(set(b['ztf_name'].dropna()))]
    t['source'] = 'TNS'
    for c in ['bts_peakabs', 'bts_rise', 'bts_fade', 'bts_duration', 'bts_av']:
        t[c] = np.nan
    t = t[['ztf_name', 'name', 'ra', 'declination', 'redshift', 'type', 'source',
           'bts_peakabs', 'bts_rise', 'bts_fade', 'bts_duration', 'bts_av']]

    pool = pd.concat([b, t], ignore_index=True)
    # A ZTF id is what the light-curve query needs; rows without one go last, since
    # they cost an extra cone search that often fails.
    pool['has_ztf'] = pool['ztf_name'].notna().astype(int)
    if QG['require_redshift']:
        pool = pool[pool['redshift'].notna()]
    pool = pool.sample(frac=1.0, random_state=random_state)
    pool = pool.sort_values('has_ztf', ascending=False).reset_index(drop=True)
    cap = min(len(pool), max(target * oversample, target * 4))
    return pool.head(cap).reset_index(drop=True)


sn_pools = {}
print(f'{"subtype":10s} {"BTS":>6s} {"TNS extra":>10s} {"pool":>6s} {"target":>7s}  headroom')
for st in SN_SUBTYPES:
    p = build_pool(st, N_PER_SN_SUBTYPE)
    sn_pools[st] = p
    nb_ = int((p['source'] == 'BTS').sum()); nt_ = int((p['source'] == 'TNS').sum())
    print(f'{st:10s} {nb_:6d} {nt_:10d} {len(p):6d} {N_PER_SN_SUBTYPE:7d}  '
          f'{len(p)/max(N_PER_SN_SUBTYPE,1):.0f}x')

if min(len(p) for p in sn_pools.values()) < N_PER_SN_SUBTYPE * 4:
    print('\nWARNING: a pool is under 4x its target. Under a strict quality gate that '
          'may not be enough headroom — loosen QG or lower the target for that class.')

In [ ]:
# ---------------------------------------------------------------------------
# AGN and TDE pools.
#
# AGN reached 150/150 in v3 from TNS + Milliquas, so that path is unchanged.
#
# TDE is the real scarcity problem. v3 drew from TNS alone (153 rows), lost 80 to
# resolution and quality failures, and stopped at 73. v4 adds the ZTF BTS TDE list --
# 78 spectroscopically confirmed TDEs that already carry their ZTF object id, so they
# skip the cone search that was v3's most common failure -- and unions it with TNS.
# ---------------------------------------------------------------------------
agn_mask = tns_full['type'].astype(str).str.contains('AGN|QSO|Quasar', case=False, na=False)
tde_mask = tns_full['type'].astype(str).str.contains('TDE', case=False, na=False)

tns_agn = tns_full[agn_mask][KEEP].copy()
tns_tde = tns_full[tde_mask][KEEP].copy()
print(f'TNS AGN {len(tns_agn)}   TNS TDE {len(tns_tde)}')

try:
    from astroquery.vizier import Vizier
    v = Vizier(columns=['RAJ2000', 'DEJ2000', 'Name', 'z', 'Type'], row_limit=20000)
    v.ROW_LIMIT = 20000
    mq = v.query_constraints(catalog='VII/294/catalog', Type='Q*', DEJ2000='>-30')[0]
    mq = mq.to_pandas().rename(columns={'RAJ2000': 'ra', 'DEJ2000': 'declination',
                                        'Name': 'name', 'z': 'redshift'})
    mq['ztf_name'] = np.nan
    mq['type'] = 'AGN'
    mq = mq.sample(min(len(mq), N_PER_CLASS * 30), random_state=RANDOM_STATE)
    print('Milliquas AGN candidates pulled:', len(mq))
except Exception as e:
    mq = pd.DataFrame(columns=['ra', 'declination', 'name', 'redshift', 'ztf_name', 'type'])
    print('Milliquas lookup unavailable (', e, ') - falling back to TNS AGN only.')

agn_pool = pd.concat([tns_agn.assign(source='TNS'),
                      mq.assign(source='Milliquas')], ignore_index=True)
agn_pool = agn_pool.sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)
print(f'AGN pool {len(agn_pool)}  ({len(agn_pool)/N_PER_CLASS:.0f}x target)')

In [ ]:
# The ZTF BTS TDE list. Fetched live when the site is reachable; otherwise the inline
# snapshot below (scraped 2026-09-10) is used, so this cell never blocks a run.
BTS_TDE_INLINE = """ztf_name,name,type,redshift,bts_peakabs,bts_rise,bts_fade,bts_duration
ZTF17aaazdba,AT2019azh,TDE,0.022,-19.76,24.318,42.102,66.42
ZTF18aabdajx,AT2022dbl,TDE,0.0284,-18.57,9.741,,
ZTF18aabkvpi,AT2018meh,TDE,0.011,-16.44,,19.386,
ZTF18aabtxvd,AT2018zr,TDE,0.071,-19.20,,,
ZTF18aahqkbt,AT2018bsi,TDE,0.051,-18.94,,6.013,
ZTF18abxftqm,AT2018hco,TDE,0.09,-19.82,35.972,89.537,125.509
ZTF18abxnvoz,AT2024pvu,TDE,0.048,-19.82,21.234,20.381,41.615
ZTF18acaqdaa,AT2018iih,TDE,0.212,-21.71,,141.381,
ZTF18acpdvos,AT2018hyz,TDE,0.04573,-18.42,9.896,37.963,47.859
ZTF19aabbnzo,AT2018lna,TDE,0.091,-19.50,17.59,34.324,51.914
ZTF19aakswrb,AT2019bhf,TDE,0.1206,-20.06,9.92,32.361,42.281
ZTF19aapreis,AT2019dsg,TDE,0.0512,-19.23,21.823,57.977,79.8
ZTF19aarioci,AT2019ehz,TDE,0.074,-19.46,14.984,41.421,56.405
ZTF19abidbya,AT2019lwu,TDE,0.117,-19.82,10.152,30.537,40.689
ZTF19abzrhgq,AT2019qiz,TDE,0.0151,-18.64,7.76,22.191,29.951
ZTF19acspeuw,AT2019vcb,TDE,0.089,-19.66,27.499,21.672,49.171
ZTF20aabqihu,AT2020pj,TDE,0.068,-18.75,10.69,19.885,30.575
ZTF20aahmtso,AT2022gri,TDE,0.028,-18.16,0.703,5.088,5.791
ZTF20abfcszi,AT2020mot,TDE,0.07,-19.46,37.679,66.377,104.056
ZTF20abgwfek,AT2020neh,TDE,0.062,-19.51,7.896,18.016,25.912
ZTF20abisysx,AT2020nov,TDE,0.084,-20.34,84.891,,
ZTF20abwtifz,AT2020afhd,TDE,0.027,-18.92,25.074,,
ZTF20acaazkt,AT2020vdq,TDE,0.044,-19.58,6.211,8.071,14.282
ZTF20achpcvt,AT2020vwl,TDE,0.0325,-18.29,,,
ZTF20acitpfz,AT2020wey,TDE,0.02738,-17.58,7.816,11.253,19.069
ZTF20acnznms,AT2020yue,TDE,0.204,-21.33,43.867,57.934,101.801
ZTF20acqoiyt,AT2020zso,TDE,0.061,-19.69,13.478,14.726,28.204
ZTF20acwytxn,AT2020acka,TDE,0.338,-22.94,,48.449,
ZTF21aaaokyp,AT2021axu,TDE,0.192,-21.13,28.678,25.912,54.59
ZTF21aanxhjv,AT2021ehb,TDE,0.018,-17.01,99.925,6.997,106.922
ZTF21aauuybx,AT2021jjm,TDE,0.153,-20.51,5.636,22.013,27.649
ZTF21abaxaqq,AT2021mhg,TDE,0.073,-18.96,,,
ZTF21abcgnqn,AT2021nwa,TDE,0.047,-18.17,23.017,75.598,98.615
ZTF21abhrchb,AT2021qth,TDE,0.0805,-19.24,19.53,37.834,57.364
ZTF21abjrysr,AT2021sdu,TDE,0.059,-19.16,7.453,15.029,22.482
ZTF21abqtckk,AT2021utq,TDE,0.127,-20.21,9.259,,
ZTF21abxngcz,AT2021yzv,TDE,0.286,-21.94,,,
ZTF22aaabovl,AT2022aee,TDE,0.124,-20.44,,,
ZTF22aaaedas,AT2022rz,TDE,0.107,-19.55,,,
ZTF22aaaekqh,AT2022adm,TDE,0.061,-19.25,,,
ZTF22aaahtqz,AT2022bdw,TDE,0.03782,-18.80,,,
ZTF22aaaihgr,AT2022arb,TDE,0.062,-18.33,,38.011,
ZTF22aabimec,AT2022csn,TDE-H-He,0.148,-20.78,,60.483,
ZTF22aacgcwv,AT2022dyt,TDE,0.072,-18.58,,,
ZTF22aadgefj,AT2022exr,TDE,0.096,-20.15,,,
ZTF22aagvrlq,AT2022ibq,TDE,0.395,-22.74,,36.179,
ZTF22aagyuao,AT2022hvp,TDE,0.12,-22.21,9.264,,
ZTF22aaigqsr,AT2024tvd,TDE,0.04493,-19.60,15.497,,
ZTF22aavvqyh,AT2022pna,TDE,0.095,-20.05,,17.212,
ZTF22abajudi,AT2022lri,TDE,0.03275,-17.36,,48.906,
ZTF22abegjtx,AT2022upj,TDE,0.054,-18.84,35.433,,
ZTF22abkfhua,AT2022wtn,TDE,0.049,-18.82,34.054,38.802,72.856
ZTF23aadcbay,AT2023cvb,TDE,0.071,-20.66,,,
ZTF23aapyidj,AT2023mfm,TDE,0.087,-19.77,20.934,28.954,49.888
ZTF23aaqdjhi,AT2023mhs,TDE,0.0482,-20.85,3.637,13.703,17.34
ZTF23abkixdb,AT2023wdb,TDE,0.06,-18.71,14.813,,
ZTF24aaahxwr,AT2024as,TDE,0.13,-20.73,,46.278,
ZTF24aaecooj,AT2024bgz,TDE,0.0585,-19.58,,33.965,
ZTF24aajvvhj,AT2024gre,TDE,0.12,-20.74,23.449,,
ZTF24aakaiha,AT2024gxr,TDE-He,0.051,-18.33,,40.378,
ZTF24aatxshz,AT2024qab,TDE,0.1398,-20.17,22.726,65.652,88.378
ZTF24abfaake,AT2024ule,TDE-H-He,0.111,-20.06,,,
ZTF24abjfuew,AT2024wsd,TDE,0.063,-18.89,,30.09,
ZTF24abmaujk,AT2024ymu,TDE,0.12,-20.44,24.939,,
ZTF24abmybnp,AT2024yqo,TDE,0.06168,-19.45,12.637,35.917,48.554
ZTF24abyhjvc,AT2024aepd,TDE,0.0835,-19.92,,31.852,
ZTF25aacnkte,TDE2025axp,TDE-He,0.1045,-19.57,,,
ZTF25aafofcs,TDE2025bri,TDE-He,0.061,-19.91,29.63,26.308,55.938
ZTF25aafywpr,TDE2025ccg,TDE-He,0.2,-21.17,4.99,23.863,28.853
ZTF25aagevje,TDE2025chm,TDE,0.101,-20.26,25.876,33.148,59.024
ZTF25aajfjhm,TDE2025eve,TDE,0.089,-20.14,18.972,,
ZTF25aaymnha,TDE2025qhf,TDE,0.093,-19.40,23.191,35.051,58.242
ZTF25ablxhsq,TDE2025vjw,TDE-H-He,0.21,-22.25,2.032,,
ZTF25abuovxv,TDE2025zmb,TDE-featureless,0.345,-23.63,79.585,59.869,139.454
ZTF25abwvuln,TDE2025abqg,TDE,0.0772,-18.79,9.632,32.524,42.156
ZTF25abxvhmk,TDE2025abcr,TDE-H-He,0.05,-17.75,36.826,23.502,60.328
ZTF25acfyeke,TDE2025afvr,TDE-featureless,0.403,-23.40,41.785,,
ZTF26aaoebni,SN2026hiw,TDE,0.035,-17.69,30.596,41.613,72.209"""

try:
    bts_tde = fetch_bts(classstring='TDE', quality=False)
    bts_tde = bts_tde.rename(columns={'ZTFID': 'ztf_name', 'IAUID': 'name', 'RA': 'ra',
                                      'Dec': 'declination', 'peakabs': 'bts_peakabs',
                                      'rise': 'bts_rise', 'fade': 'bts_fade',
                                      'duration': 'bts_duration', 'A_V': 'bts_av'})
    print('BTS TDE rows fetched live:', len(bts_tde))
except Exception as e:
    print('BTS TDE fetch failed (', e, ') - using the inline snapshot.')
    bts_tde = pd.read_csv(io.StringIO(BTS_TDE_INLINE))

for c in ['bts_peakabs', 'bts_rise', 'bts_fade', 'bts_duration', 'redshift']:
    if c in bts_tde:
        bts_tde[c] = pd.to_numeric(bts_tde[c].astype(str).str.replace('>', '', regex=False),
                                   errors='coerce')
for c in ['ra', 'declination', 'bts_av']:
    if c not in bts_tde:
        bts_tde[c] = np.nan
bts_tde['source'] = 'BTS'

tns_tde_extra = tns_tde[~tns_tde['ztf_name'].isin(set(bts_tde['ztf_name'].dropna()))].copy()
tns_tde_extra['source'] = 'TNS'
for c in ['bts_peakabs', 'bts_rise', 'bts_fade', 'bts_duration', 'bts_av']:
    tns_tde_extra[c] = np.nan

_cols = ['ztf_name', 'name', 'ra', 'declination', 'redshift', 'type', 'source',
         'bts_peakabs', 'bts_rise', 'bts_fade', 'bts_duration', 'bts_av']
tde_pool = pd.concat([bts_tde.reindex(columns=_cols),
                      tns_tde_extra.reindex(columns=_cols)], ignore_index=True)
tde_pool['has_ztf'] = tde_pool['ztf_name'].notna().astype(int)
tde_pool = (tde_pool.sample(frac=1.0, random_state=RANDOM_STATE)
                    .sort_values('has_ztf', ascending=False).reset_index(drop=True))

n_bts = int((tde_pool['source'] == 'BTS').sum())
n_tns = int((tde_pool['source'] == 'TNS').sum())
print(f'TDE pool {len(tde_pool)}  (BTS {n_bts}, TNS extra {n_tns}) for a target of {N_TDE_TARGET}')
print(f'  {int(tde_pool["has_ztf"].sum())} already carry a ZTF object id; '
      f'v3 had to cone-search every candidate')

In [ ]:
FLARE_STAR_NAMES = [
    # UV Ceti-type flare stars — unchanged from v2, plus more names so the retry loop
    # has somewhere to go if sectors fail the quality gate.
    "UV Ceti", "AD Leo", "EV Lac", "YZ CMi", "AU Mic", "Proxima Centauri",
    "GJ 1243", "Ross 154", "Ross 128", "Wolf 359", "Lalande 21185", "Kruger 60",
    "TZ Ari", "GJ 1111", "V1216 Sgr", "EQ Peg", "DO Cep", "V1005 Ori",
    "V371 Ori", "WX UMa", "Luyten 726-8", "GJ 896A", "GJ 1156", "GJ 1245A",
    "GJ 3236", "GJ 3338", "GJ 3737", "GJ 3685A", "GJ 424", "GJ 1002",
    "TRAPPIST-1", "LHS 1140", "Teegarden's Star", "GJ 1061", "YZ Cet",
    "Luyten's Star", "Lacaille 8760", "Lacaille 9352", "Gliese 1", "Gliese 876",
    "Gliese 682", "Gliese 832", "Gliese 667 C", "Kepler-411",
    "GJ 4247", "GJ 4053", "V1054 Oph", "BY Dra", "CR Dra", "DG CVn", "GJ 65 A",
    "AT Mic", "AZ Cnc", "BL Cet", "CN Leo", "FL Vir", "GT Peg", "HU Del",
]
print(len(FLARE_STAR_NAMES), 'flare-star candidates')

# STEP 3 — Preprocessing, the Villar fit, and the quality gate  *(brief Phase 3)*

The four GP shape numbers are gone. In their place is the **Villar et al. (2019)
parametric supernova model**, the same functional form Superphot+ fits:

$$
f(t)=\frac{A}{1+e^{-(t-t_0)/\tau_{\rm rise}}}\times
\begin{cases}
1-\beta\,(t-t_0), & t<\gamma+t_0\\[4pt]
(1-\beta\gamma)\,e^{-(t-\gamma-t_0)/\tau_{\rm fall}}, & t\ge\gamma+t_0
\end{cases}
$$

Six parameters per band, each of which means something physical: $A$ amplitude,
$t_0$ time of peak, $\tau_{\rm rise}$ rise timescale, $\gamma$ plateau length,
$\beta$ plateau slope, $\tau_{\rm fall}$ decline timescale. A Type IIP's plateau lives
in $\gamma$ and $\beta$; a SLSN's slow evolution lives in $\tau_{\rm rise}$ and
$\tau_{\rm fall}$; a Ia's characteristic decline lives in $\tau_{\rm fall}$.

Everything is measured **relative to $t_0$**, so nothing depends on where the
observing window happened to start — the defect that made v2's `rise_time` report
105 days for a Type Ia.

Three further additions:

* **`peak_abs_mag`** — peak apparent magnitude corrected for distance modulus and
  Galactic extinction. This is what makes SLSNe separable; in the benchmark run it was
  worth 15 accuracy points on its own.
* **g/r ratios** of every fit parameter, which encode colour *evolution* rather than a
  single colour at one epoch.
* **`chisq_dof`** kept as a feature, not just a gate. A model that fits an SN badly is
  itself informative.

The **quality gate** is applied inside the retry loop. An object that fails is
replaced by the next candidate, so the final counts stay exact at 30 per subtype —
this is the "keep pulling until N survive" contract from v2, now with a bar that is
actually high enough to be worth clearing.

In [ ]:
# v4 photometry, Villar fit, quality gate, features (verbatim)
# ---------------------------------------------------------------
# Inlined verbatim from btp5/photometry.py, which the repository's test suite
# (tests5/) exercises. Edit it there and rebuild this notebook with
# tools/build_notebook_v5.py rather than editing it here.
# ---------------------------------------------------------------

"""
Photometry, the Villar (2019) parametric fit, the quality gate, and feature extraction.

Every definition below is VERBATIM from the v4 notebook
(`notebooks/source/BTP_TNS_Hierarchical_v4_original.ipynb`), lifted by AST span by
`tools/extract_v5_photometry.py`. The v5 brief (Section 1) says to preserve v4's
pipeline except where explicitly changed, and none of these were.

They live in a module so the v5 additions can be tested against them offline and so
Phase 2/3 can run headless. `tools/build_notebook_v5.py` inlines this module back into
the notebook, and `tests5/test_photometry_parity.py` asserts the two stay identical.
"""

import numpy as np
import pandas as pd
from astropy.cosmology import FlatLambdaCDM
from scipy.optimize import least_squares



# ---------------------------------------------------------------------------
# Photometry helpers. ALeRCE returns PSF magnitudes; the parametric model is defined
# in flux, so convert once, up front, and carry the uncertainty through properly.
# ---------------------------------------------------------------------------
ZP = 27.5   # arbitrary but fixed zeropoint; only differences of magnitudes matter


def mag_to_flux(mag, sigma_mag=None):
    f = 10.0 ** (-0.4 * (np.asarray(mag, float) - ZP))
    if sigma_mag is None:
        return f
    fe = f * 0.4 * np.log(10.0) * np.asarray(sigma_mag, float)
    return f, fe


def clean_band(t, f, fe, min_points=5, sigma_clip=6.0):
    """Drop NaNs and gross outliers. Looser than v2's 5-sigma clip on magnitudes,
    because a real supernova peak IS a large excursion and v2's clip was sometimes
    eating it."""
    ok = np.isfinite(t) & np.isfinite(f) & np.isfinite(fe) & (fe > 0)
    t, f, fe = np.asarray(t)[ok], np.asarray(f)[ok], np.asarray(fe)[ok]
    if len(t) < min_points:
        return None
    # clip only on the residual to a running median, so the transient itself survives
    med = np.median(f)
    mad = np.median(np.abs(f - med)) * 1.4826 + 1e-12
    keep = (f - med) > -sigma_clip * mad          # one-sided: keep bright excursions
    t, f, fe = t[keep], f[keep], fe[keep]
    if len(t) < min_points:
        return None
    o = np.argsort(t)
    return t[o], f[o], fe[o]


def villar(t, A, beta, gamma, t0, tau_rise, tau_fall):
    tau_rise = max(tau_rise, 1e-3); tau_fall = max(tau_fall, 1e-3)
    sig = 1.0 / (1.0 + np.exp(-(t - t0) / tau_rise))
    out = np.empty_like(t, dtype=float)
    plateau = t < (gamma + t0)
    out[plateau]  = A * (1.0 - beta * (t[plateau] - t0))
    out[~plateau] = A * (1.0 - beta * gamma) * np.exp(-(t[~plateau] - (gamma + t0)) / tau_fall)
    return sig * out


def fit_villar(t, f, fe):
    """Returns (params dict, chi2/dof) or None. Weighted least squares with bounds."""
    t = np.asarray(t, float); f = np.asarray(f, float); fe = np.asarray(fe, float)
    fe = np.where(fe > 0, fe, np.nanmedian(fe[fe > 0]) if (fe > 0).any() else 1.0)
    i = int(np.argmax(f)); A0 = max(f[i], 1e-6); t_peak0 = t[i]
    lo = [1e-8, -0.5,   0.5, t.min() - 100,  0.05,   0.5]
    hi = [A0 * 20 + 1e-6, 0.5, 400.0, t.max() + 100, 100.0, 400.0]
    best = None
    for g0 in (10.0, 40.0, 120.0):
        p0 = np.clip([A0, 1e-3, g0, t_peak0, 5.0, 20.0], lo, hi)
        try:
            r = least_squares(lambda p: (villar(t, *p) - f) / fe, p0,
                              bounds=(lo, hi), method='trf', max_nfev=400)
        except Exception:
            continue
        if best is None or r.cost < best.cost:
            best = r
    if best is None:
        return None
    dof = max(len(t) - 6, 1)
    keys = ['A', 'beta', 'gamma', 't0', 'tau_rise', 'tau_fall']
    return dict(zip(keys, best.x)), float(2 * best.cost / dof)


def _log10(x, floor=1e-8):
    return float(np.log10(max(float(x), floor)))


# ---------------------------------------------------------------------------
# The quality gate. Returns (True, '') or (False, reason).
# Called from inside the retry loop, so a rejection costs one extra candidate and
# nothing else. Every rejection reason is counted so a stalled class explains itself.
#
# v4 additions, each one measured: min_obs_span, min_days_postpeak raised 5 -> 30,
# and require_two_bands. See the ablation table at the top of the notebook.
# ---------------------------------------------------------------------------
def quality_gate(bands, redshift, qg=QG):
    """`bands` maps 'g'/'r' -> (t, f, fe, params, chisq)."""
    if not bands:
        return False, 'no band survived cleaning'
    if qg.get('require_two_bands') and len(bands) < 2:
        return False, 'only one band'
    n_total = sum(len(v[0]) for v in bands.values())
    if n_total < qg['min_det_total']:
        return False, f'only {n_total} detections'
    best = max(bands, key=lambda b: len(bands[b][0]))
    if len(bands[best][0]) < qg['min_det_best_band']:
        return False, f'best band has only {len(bands[best][0])} detections'
    t, f, fe, p, chi = bands[best]
    span = float(t.max() - t.min())
    if span < qg.get('min_obs_span', 0.0):
        return False, f'baseline only {span:.0f} d'
    if float(np.max(f / fe)) < qg['min_peak_snr']:
        return False, 'peak SNR too low'
    if not np.isfinite(chi) or chi > qg['max_chisq_dof']:
        return False, f'fit chisq/dof {chi:.1f}'
    if qg['require_redshift'] and not (np.isfinite(redshift) and redshift > 0):
        return False, 'no redshift'
    if qg['require_bracket']:
        t0 = p['t0']
        if (t0 - t.min()) < qg['min_days_prepeak']:
            return False, 'no pre-peak coverage'
        if (t.max() - t0) < qg['min_days_postpeak']:
            return False, 'not enough post-peak coverage'
    return True, ''


COSMO = FlatLambdaCDM(H0=70, Om0=0.3)


def distance_modulus(z):
    if not (np.isfinite(z) and z > 0):
        return np.nan
    return float(COSMO.distmod(max(z, 1e-4)).value)


def process_ztf_object(path, oid, label, redshift=np.nan, extra=None):
    """CSV of ALeRCE detections -> one feature row, or None if the quality gate fails."""
    df = pd.read_csv(path)
    if not {'mjd', 'fid', 'magpsf'}.issubset(df.columns):
        return None
    if 'sigmapsf' not in df.columns:
        df['sigmapsf'] = 0.1

    bands = {}
    for fid, bname in [(1, 'g'), (2, 'r')]:
        s = df[df['fid'] == fid]
        if len(s) < 5:
            continue
        f, fe = mag_to_flux(s['magpsf'].values, s['sigmapsf'].values)
        c = clean_band(s['mjd'].values, f, fe)
        if c is None:
            continue
        out = fit_villar(*c)
        if out is None:
            continue
        p, chi = out
        bands[bname] = (c[0], c[1], c[2], p, chi)

    ok, reason = quality_gate(bands, redshift)
    if not ok:
        return None

    ref = 'r' if 'r' in bands else 'g'
    t, f, fe, p, chi = bands[ref]

    row = {'id': oid, 'label': label, 'survey': 'ZTF', 'ref_band': ref,
           'log_A':        _log10(p['A']),
           'beta':         float(p['beta']),
           'log_gamma':    _log10(p['gamma']),
           'log_tau_rise': _log10(p['tau_rise']),
           'log_tau_fall': _log10(p['tau_fall']),
           'log_chisq':    _log10(chi)}

    # colour evolution: ratios of every fit parameter between the two bands
    if 'g' in bands and 'r' in bands:
        pg = bands['g'][3]; pr = bands['r'][3]
        row.update({'gr_log_A':        _log10(pg['A']) - _log10(pr['A']),
                    'gr_dt0':          float(pg['t0'] - pr['t0']),
                    'gr_log_tau_rise': _log10(pg['tau_rise']) - _log10(pr['tau_rise']),
                    'gr_log_tau_fall': _log10(pg['tau_fall']) - _log10(pr['tau_fall']),
                    'gr_dbeta':        float(pg['beta'] - pr['beta']),
                    'gr_log_gamma':    _log10(pg['gamma']) - _log10(pr['gamma']),
                    'has_color': 1})
    else:
        for k in ['gr_log_A', 'gr_dt0', 'gr_log_tau_rise', 'gr_log_tau_fall',
                  'gr_dbeta', 'gr_log_gamma']:
            row[k] = np.nan
        row['has_color'] = 0

    # physical scale — the feature v2 threw away
    extra = extra or {}
    av = float(extra.get('bts_av') or 0.0)
    peak_mag_app = ZP - 2.5 * row['log_A']
    dm = distance_modulus(redshift)
    row['redshift']     = float(redshift) if np.isfinite(redshift) else np.nan
    row['peak_abs_mag'] = peak_mag_app - dm - av if np.isfinite(dm) else np.nan

    # model-derived, phase-anchored timescales (NOT window-anchored)
    tt = np.linspace(p['t0'] - 200, p['t0'] + 500, 2000)
    m = villar(tt, **p); pk = float(m.max())
    if pk > 0:
        above = tt[m >= pk / 2.0]
        row['duration_half'] = float(above.max() - above.min()) if len(above) else np.nan
        row['rise_half']     = float(p['t0'] - above.min()) if len(above) else np.nan
        row['fade_half']     = float(above.max() - p['t0']) if len(above) else np.nan
    else:
        row['duration_half'] = row['rise_half'] = row['fade_half'] = np.nan

    row['n_det']    = int(sum(len(v[0]) for v in bands.values()))
    row['max_snr']  = float(np.max(f / fe))
    row['obs_span'] = float(t.max() - t.min())
    return row


# ---------------------------------------------------------------------------
# Stellar flares from TESS. Same feature vocabulary so the columns line up, but the
# Villar model is a supernova model and does not describe a flare — so for flares the
# fit is deliberately run on the *relative* flux and the resulting parameters are still
# informative (a flare is an extremely fast rise and fall, which is exactly what
# tau_rise and tau_fall report). There is no redshift and no absolute magnitude, and
# `has_color` is 0, which is itself a real observational signal, not an artefact.
# ---------------------------------------------------------------------------
def process_tess_object(path, star_name, label):
    df = pd.read_csv(path)
    if not {'bjd', 'flux'}.issubset(df.columns):
        return None
    fe = df['flux_err'].values if 'flux_err' in df else np.full(len(df), np.nan)
    if not np.isfinite(fe).any():
        fe = np.full(len(df), float(np.nanstd(df['flux'].values)) or 1.0)
    c = clean_band(df['bjd'].values, df['flux'].values, fe, min_points=20)
    if c is None:
        return None
    t, f, ferr = c
    med = np.median(f)
    if med == 0:
        return None
    f = f / med; ferr = ferr / abs(med)
    out = fit_villar(t, f, ferr)
    if out is None:
        return None
    p, chi = out
    if len(t) < QG['min_det_best_band']:
        return None

    row = {'id': star_name, 'label': label, 'survey': 'TESS', 'ref_band': 'TESS',
           'log_A': _log10(p['A']), 'beta': float(p['beta']),
           'log_gamma': _log10(p['gamma']), 'log_tau_rise': _log10(p['tau_rise']),
           'log_tau_fall': _log10(p['tau_fall']), 'log_chisq': _log10(chi)}
    for k in ['gr_log_A', 'gr_dt0', 'gr_log_tau_rise', 'gr_log_tau_fall',
              'gr_dbeta', 'gr_log_gamma']:
        row[k] = np.nan
    row['has_color'] = 0
    row['redshift'] = 0.0
    row['peak_abs_mag'] = np.nan     # imputed later; flares are Galactic
    tt = np.linspace(p['t0'] - 20, p['t0'] + 60, 2000)
    m = villar(tt, **p); pk = float(m.max())
    above = tt[m >= pk / 2.0] if pk > 0 else np.array([])
    row['duration_half'] = float(above.max() - above.min()) if len(above) else np.nan
    row['rise_half']     = float(p['t0'] - above.min()) if len(above) else np.nan
    row['fade_half']     = float(above.max() - p['t0']) if len(above) else np.nan
    row['n_det']   = int(len(t))
    row['max_snr'] = float(np.nanmax(f / ferr))
    row['obs_span'] = float(t.max() - t.min())
    return row

### Finding the light curves again

v4's feature rows and acquisition manifests carry **no sky coordinates**. Host offset
(Step 7) needs them and augmentation (Step 8) needs to re-read each light curve. v4's
detection CSVs do keep per-detection `ra`/`dec`, so both are recovered from those. The
first v5 build missed this: its tests supplied coordinates that real v4 rows do not
have, so in a live run every object would have come back `has_host_match = 0`.

In [ ]:
# light-curve index and coordinate recovery
# ---------------------------------------------------------------
# Inlined verbatim from btp5/lightcurves.py, which the repository's test suite
# (tests5/) exercises. Edit it there and rebuild this notebook with
# tools/build_notebook_v5.py rather than editing it here.
# ---------------------------------------------------------------

"""
Locate each object's saved light curve, and recover its sky position from it.

Why this exists: v4's feature rows (process_ztf_object) and acquisition manifests
(build_class_to_target) carry no coordinates at all. The host-offset feature needs
them, and so does augmentation (which re-reads the parent light curve). Without this,
a live run would give every object has_host_match=0 and the host feature would be
silently empty — a failure that no offline test with a coordinate-bearing fixture
could see.

The detection CSVs v4 saves DO keep per-detection ra/dec, and the median over an
object's detections is a robust position (ZTF alert astrometry scatters by
~0.1 arcsec; the median discards the occasional bad centroid). This works identically
for TNS-sourced and ALeRCE-sourced objects, since both are saved by the same loop.
"""

import os

import numpy as np
import pandas as pd


def index_lightcurves(base_dir, exclude_dirs=('tns_catalog',)):
    """{object id -> csv path} for every saved light curve under base_dir.

    Searching by filename rather than rebuilding v4's subdir naming keeps this
    correct whichever class folder an object was saved under.
    """
    idx = {}
    if not base_dir or not os.path.isdir(base_dir):
        return idx
    for root, dirs, files in os.walk(base_dir):
        dirs[:] = [d for d in dirs if d not in exclude_dirs]
        for fn in files:
            if fn.endswith('.csv') and not fn.endswith(('_manifest_v3.csv', '_failed_v3.csv')):
                idx.setdefault(fn[:-4], os.path.join(root, fn))
    return idx


def position_from_csv(path):
    """Median (ra, dec) over an object's detections, or (nan, nan)."""
    try:
        df = pd.read_csv(path, usecols=lambda c: c in ('ra', 'dec'))
    except Exception:
        return np.nan, np.nan
    if not {'ra', 'dec'}.issubset(df.columns) or df.empty:
        return np.nan, np.nan
    ra = df['ra'].to_numpy(float)
    dec = df['dec'].to_numpy(float)
    ok = np.isfinite(ra) & np.isfinite(dec)
    if not ok.any():
        return np.nan, np.nan
    ra, dec = ra[ok], dec[ok]
    # RA wraps at 360: an object at ra~0.0001 and ~359.9999 must not average to 180.
    if ra.max() - ra.min() > 180:
        ra = np.where(ra > 180, ra - 360, ra)
    r = float(np.median(ra)) % 360.0
    return r, float(np.median(dec))


def attach_coordinates(df, lc_index, verbose=True):
    """
    Add `ra`, `dec` and `lc_path` to a feature table.

    Rows that already have finite coordinates keep them. TESS rows get NaN, which is
    correct: a Galactic flare star has no host galaxy to measure an offset from.
    """
    out = df.copy()
    for c in ('ra', 'dec'):
        if c not in out.columns:
            out[c] = np.nan
    out['lc_path'] = out['id'].astype(str).map(lc_index)

    need = out['ra'].isna() | out['dec'].isna()
    if 'survey' in out.columns:
        need &= out['survey'].eq('ZTF')
    for i in out.index[need]:
        p = out.at[i, 'lc_path']
        if isinstance(p, str):
            out.at[i, 'ra'], out.at[i, 'dec'] = position_from_csv(p)

    if verbose:
        ztf = out['survey'].eq('ZTF') if 'survey' in out.columns else np.ones(len(out), bool)
        n_pos = int((ztf & out['ra'].notna()).sum())
        n_lc = int((ztf & out['lc_path'].notna()).sum())
        print(f'light curves found for {n_lc}/{int(ztf.sum())} ZTF objects; '
              f'positions recovered for {n_pos}')
        if n_lc < int(ztf.sum()):
            print('  (an object with no saved CSV was acquired in an earlier session '
                  'under a different BASE_DIR; its host offset will be NaN)')
    return out

# STEP 4 — Balanced acquisition  *(brief Phase 4)*

v4's retry-until-target loop, unchanged.

In [ ]:
def _load_checkpoint(manifest_path, features_path, failed_path):
    manifest, features, failed = [], [], set()
    if os.path.exists(manifest_path) and os.path.exists(features_path):
        manifest = pd.read_csv(manifest_path).to_dict('records')
        features = pd.read_csv(features_path).to_dict('records')
    if os.path.exists(failed_path):
        failed = set(pd.read_csv(failed_path)['candidate'].astype(str))
    return manifest, features, failed


def _save_checkpoint(manifest, features, failed, manifest_path, features_path, failed_path):
    pd.DataFrame(manifest).to_csv(manifest_path, index=False)
    pd.DataFrame(features).to_csv(features_path, index=False)
    pd.DataFrame({'candidate': sorted(failed)}).to_csv(failed_path, index=False)


def resolve_oid(row, radius_arcsec=2.0):
    """BTS rows already carry the ZTF id, so this only runs for TNS-only candidates."""
    z = row.get('ztf_name')
    if isinstance(z, str) and z.startswith('ZTF'):
        return z
    try:
        res = alerce.query_objects(ra=row['ra'], dec=row['declination'],
                                   radius=radius_arcsec, format='pandas')
        if len(res):
            return res.iloc[0]['oid']
    except Exception:
        pass
    return None


def query_detections(oid):
    return alerce.query_detections(oid, format='pandas')

In [ ]:
def build_class_to_target(label, pool, target, *, base_dir=BASE_DIR, feature_dir=FEATURE_DIR,
                          out_subdir=None, min_detections=None, sleep=0.12,
                          checkpoint_every=10, progress=tqdm, verbose=True):
    """
    Pull candidates from `pool` until exactly `target` of them survive resolution,
    download, cleaning, the Villar fit AND the quality gate.
    """
    min_detections = min_detections or QG['min_det_total']
    out_subdir = out_subdir or label.lower()
    out_dir = os.path.join(base_dir, out_subdir)
    os.makedirs(out_dir, exist_ok=True); os.makedirs(feature_dir, exist_ok=True)

    manifest_path = os.path.join(base_dir, f'{out_subdir}_manifest_v3.csv')
    features_path = os.path.join(feature_dir, f'{out_subdir}_features_v3.csv')
    failed_path   = os.path.join(base_dir,    f'{out_subdir}_failed_v3.csv')

    manifest, features, failed = _load_checkpoint(manifest_path, features_path, failed_path)
    done = {str(m['tns_name']) for m in manifest}
    if features and verbose:
        print(f'Resuming {label}: {len(features)}/{target} survived, {len(failed)} known failures')
    if len(features) >= target:
        return manifest[:target], features[:target]

    reasons = {}
    bar = progress(total=target, initial=len(features), desc=f'{label} (survivors)') if progress else None
    attempted = 0
    try:
        for count, (_, row) in enumerate(pool.iterrows()):
            if len(features) >= target:
                break
            name = str(row.get('name') or row.get('ztf_name'))
            if name in done or name in failed:
                continue
            attempted += 1

            oid = resolve_oid(row)
            if oid is None:
                failed.add(name); reasons['no ZTF id'] = reasons.get('no ZTF id', 0) + 1; continue
            try:
                det = query_detections(oid)
            except Exception:
                failed.add(name); reasons['ALeRCE error'] = reasons.get('ALeRCE error', 0) + 1; continue
            if det is None or len(det) < min_detections:
                failed.add(name); reasons['too few detections'] = reasons.get('too few detections', 0) + 1; continue

            keep = [c for c in ['mjd', 'fid', 'magpsf', 'sigmapsf', 'ra', 'dec'] if c in det.columns]
            csv_path = os.path.join(out_dir, f'{oid}.csv')
            det[keep].to_csv(csv_path, index=False)

            feat = process_ztf_object(csv_path, oid, label,
                                      redshift=row.get('redshift', np.nan),
                                      extra={'bts_av': row.get('bts_av', 0.0)})
            if feat is None:
                failed.add(name); reasons['quality gate / fit'] = reasons.get('quality gate / fit', 0) + 1; continue

            # carry BTS's own measurements through as cross-check columns
            for c in ['bts_peakabs', 'bts_rise', 'bts_fade', 'bts_duration']:
                feat[c] = row.get(c, np.nan)
            manifest.append({'tns_name': name, 'oid': oid, 'label': label,
                             'n_points': len(det), 'redshift': row.get('redshift'),
                             'source': row.get('source')})
            features.append(feat); done.add(name)
            if bar is not None: bar.update(1)
            if sleep: time.sleep(sleep)
            if count % checkpoint_every == 0:
                _save_checkpoint(manifest, features, failed, manifest_path, features_path, failed_path)
                gc.collect()
    finally:
        if bar is not None: bar.close()
        _save_checkpoint(manifest, features, failed, manifest_path, features_path, failed_path)

    if verbose:
        if len(features) < target:
            print(f'  SHORTFALL {label}: {len(features)}/{target} from a pool of {len(pool)} '
                  f'({attempted} attempted). Rejection reasons: {reasons}')
        else:
            print(f'  {label}: {target}/{target} reached ({attempted} attempted). '
                  f'Rejections: {reasons}')
    return manifest[:target], features[:target]

In [ ]:
def build_flares_to_target(star_names, target, *, base_dir=BASE_DIR, feature_dir=FEATURE_DIR,
                           out_subdir='stellar_flares', max_sectors_per_star=14,
                           checkpoint_every=5, progress=tqdm, verbose=True):
    """Unchanged in contract from v2; the survival test is now process_tess_object."""
    import lightkurve as lk
    out_dir = os.path.join(base_dir, out_subdir); os.makedirs(out_dir, exist_ok=True)
    manifest_path = os.path.join(base_dir, f'{out_subdir}_manifest_v3.csv')
    features_path = os.path.join(feature_dir, f'{out_subdir}_features_v3.csv')
    failed_path   = os.path.join(base_dir,    f'{out_subdir}_failed_v3.csv')

    manifest, features, failed = _load_checkpoint(manifest_path, features_path, failed_path)
    done_files = {str(m['file']) for m in manifest}
    if len(features) >= target:
        return manifest[:target], features[:target]

    bar = progress(total=target, initial=len(features), desc='stellar_flare (survivors)') if progress else None
    stars_used = 0
    try:
        for name in star_names:
            if len(features) >= target: break
            stars_used += 1
            try:
                search = lk.search_lightcurve(name, mission='TESS')
            except Exception:
                continue
            if search is None or len(search) == 0:
                continue
            safe = str(name).replace(' ', '_')
            for i, entry in enumerate(list(search)[:max_sectors_per_star]):
                if len(features) >= target: break
                fname = f'{safe}_sector{i}.csv'
                if fname in done_files or fname in failed: continue
                try:
                    lc = entry.download()
                    d = lc.to_pandas().reset_index()
                    d = d.rename(columns={d.columns[0]: 'bjd'})
                    d = d[['bjd', 'flux'] + (['flux_err'] if 'flux_err' in d else [])].dropna()
                except Exception:
                    failed.add(fname); continue
                if d is None or len(d) == 0:
                    failed.add(fname); continue
                csv_path = os.path.join(out_dir, fname); d.to_csv(csv_path, index=False)
                feat = process_tess_object(csv_path, name, 'stellar_flare')
                if feat is None:
                    failed.add(fname); continue
                feat = dict(feat); feat['id'] = f'{safe}_sector{i}'
                manifest.append({'star_name': name, 'file': fname,
                                 'label': 'stellar_flare', 'n_points': len(d)})
                features.append(feat); done_files.add(fname)
                if bar is not None: bar.update(1)
            if stars_used % checkpoint_every == 0:
                _save_checkpoint(manifest, features, failed, manifest_path, features_path, failed_path)
                gc.collect()
    finally:
        if bar is not None: bar.close()
        _save_checkpoint(manifest, features, failed, manifest_path, features_path, failed_path)
    if verbose:
        status = 'reached target' if len(features) >= target else 'SHORTFALL'
        print(f'  stellar_flare: {status} {len(features)}/{target} (used {stars_used} stars)')
    return manifest[:target], features[:target]

In [ ]:
# ---- run acquisition -------------------------------------------------------
all_manifests, all_features, counts, targets = {}, {}, {}, {}

for st in SN_SUBTYPES:
    m, f = build_class_to_target(st, sn_pools[st], N_PER_SN_SUBTYPE, out_subdir=st.lower())
    all_manifests[st], all_features[st] = m, f
    counts[st] = len(f); targets[st] = N_PER_SN_SUBTYPE

for lab, pool, sub, tgt in [('AGN', agn_pool, 'agn', N_PER_CLASS),
                            ('TDE', tde_pool, 'tde', N_TDE_TARGET)]:
    m, f = build_class_to_target(lab, pool, tgt, out_subdir=sub)
    all_manifests[lab], all_features[lab] = m, f
    counts[lab] = len(f); targets[lab] = tgt

m, f = build_flares_to_target(FLARE_STAR_NAMES, N_PER_CLASS)
all_manifests['stellar_flare'], all_features['stellar_flare'] = m, f
counts['stellar_flare'] = len(f); targets['stellar_flare'] = N_PER_CLASS

print()
for k, v in counts.items():
    print(f'  {"OK " if v >= targets[k] else "!! "}{k:<16s} {v}/{targets[k]}')

In [ ]:
def assemble_feature_df(feature_lists, verbose=True):
    rows = []
    for lst in feature_lists:
        rows.extend(lst)
    df = pd.DataFrame(rows)

    # Make sure every catalogue column exists even if no BTS-sourced object survived.
    for c in ['bts_peakabs', 'bts_rise', 'bts_fade', 'bts_duration']:
        if c not in df:
            df[c] = np.nan

    # `from_bts` says whether a catalogue measurement existed at all. Keep it BEFORE
    # imputing, because after imputation the missingness is unrecoverable, and
    # "this object is in the BTS quality sample" is itself real information.
    df['from_bts'] = df['bts_peakabs'].notna().astype(int)

    # Imputation, stated explicitly rather than hidden:
    #  * gr_* are NaN when only one band survived. Under v4's require_two_bands that
    #    should now be empty for ZTF objects; TESS flares still have none.
    #  * peak_abs_mag is NaN for Galactic flares -> a sentinel far from any SN value,
    #    so a tree can split cleanly on "not extragalactic".
    #  * every fill uses a class-INDEPENDENT statistic. Filling with a class median
    #    leaks the label into the feature, which is a real and easy mistake to make.
    for c in [c for c in df.columns if c.startswith('gr_')]:
        df[c] = df[c].fillna(df[c].median())
    df['peak_abs_mag'] = df['peak_abs_mag'].fillna(0.0)
    for c in FEATURE_COLS:
        if c in df:
            df[c] = df[c].fillna(df[c].median())

    if verbose:
        print(f'Combined feature table: {df.shape[0]} rows x {df.shape[1]} cols')
        print(f'  catalogue measurements present on {int(df["from_bts"].sum())}/{len(df)} rows')
        missing = [c for c in FEATURE_COLS if c not in df.columns]
        if missing:
            print('  WARNING: FEATURE_COLS names columns that do not exist:', missing)
        unused = [c for c in df.columns
                  if c not in FEATURE_COLS + ['id', 'label', 'survey', 'ref_band', 'coarse_label']]
        if unused:
            print('  columns present but NOT used as features:', unused)
    return df.reset_index(drop=True)


feature_df = assemble_feature_df(list(all_features.values()))
feature_df.to_csv(os.path.join(FEATURE_DIR, 'features_v4.csv'), index=False)
print(feature_df['label'].value_counts().to_string())
feature_df.head()

In [ ]:
def verify_counts(feature_df, n_per_class, n_per_subtype, sn_subtype_labels, verbose=True):
    coarse = feature_df['label'].apply(lambda l: 'SNe' if l in sn_subtype_labels else l)
    rows, ok = [], True
    for cls in sorted(coarse.unique()):
        n = int((coarse == cls).sum())
        tgt = (n_per_subtype * len(sn_subtype_labels) if cls == 'SNe'
               else (N_TDE_TARGET if cls == 'TDE' else n_per_class))
        rows.append({'level': 'coarse', 'class': cls, 'count': n, 'target': tgt,
                     'shortfall': max(0, tgt - n)})
        ok = ok and n >= tgt
    for st in sn_subtype_labels:
        n = int((feature_df['label'] == st).sum())
        rows.append({'level': 'subtype', 'class': st, 'count': n,
                     'target': n_per_subtype, 'shortfall': max(0, n_per_subtype - n)})
        ok = ok and n >= n_per_subtype
    report = pd.DataFrame(rows)
    if verbose:
        print(report.to_string(index=False))
        print('All targets met.' if ok else "NOT all targets met — see 'shortfall'.")
    return ok, report

targets_ok, count_report = verify_counts(feature_df, N_PER_CLASS, N_PER_SN_SUBTYPE, SN_SUBTYPES)

In [ ]:
# Columns v4 created later, in its modelling cells, which v5 does not include. The first
# v5 build omitted `coarse_label` here and would have failed at its first training call.
feature_df['coarse_label']   = feature_df['label'].apply(lambda l: 'SNe' if l in SN_SUBTYPES else l)
feature_df['is_synthetic']   = False
feature_df['passes_v4_gate'] = True          # every row here cleared the full v4 gate
feature_df['source']         = np.where(feature_df['survey'] == 'TESS', 'TESS (curated list)', 'TNS/BTS')
# "TNS-confirmed" means a TNS spectroscopic label. TESS flares come from a curated star
# list, not TNS: they train the TNS track (as in v4) but are not TNS-confirmed objects.
feature_df['has_tns_label']  = feature_df['label'].notna() & feature_df['survey'].eq('ZTF')
print(feature_df['source'].value_counts().to_string())

# STEP 5 — ALeRCE-labelled ground truth  *(brief Phase 2B, Section 3.2)*

**The crosswalk is built against the class list fetched live, and printed, before any
ALeRCE label is used.** Mapping decisions:

* **AGN** ← `AGN`, `QSO`, `Blazar` — all accretion onto a supermassive black hole.
* **Not AGN** — ALeRCE's stochastic branch also holds `YSO` (young stellar objects) and
  `CV/Nova` (accreting white-dwarf binaries). Both are Galactic. They map to
  `stellar_flare` as the nearest analogue, which is an approximation: they are not the
  UV-Ceti M-dwarf population of our TESS sample.
* **SNe** ← `SNIa`, `SESN`, `SNII`, `SNIIn`, `SLSN`; for Stage 2, `SESN` → `SN_Ibc`
  (ALeRCE never separates Ib from Ic) and `SNIIn` → `SN_II`.
* **TDE** ← `TDE`.
* **Excluded, never reassigned:** every periodic class and `Microlensing`.

In [ ]:
taxonomy = fetch_taxonomy(alerce)

In [ ]:
live_classes = fetch_classes(alerce, ALERCE_PRIMARY_CLASSIFIER, ALERCE_PRIMARY_VERSION)
print()
crosswalk_table, unmapped_classes = build_crosswalk(live_classes)

### Pool 2 sourced from ALeRCE itself

Section 3.3 defines the ALeRCE-confident pool as objects ALeRCE classifies confidently,
**independent of whether they also have a TNS label**. So it has to be sourced from
ALeRCE, not only by labelling objects TNS already supplied — otherwise it is a subset of
the TNS pool and the spectroscopic-vs-photometric comparison of Section 6.2 is impossible
(the first v5 build had exactly this problem).

* The sample is **uniform over every object above the cut**, drawn by random rank across
  ALeRCE's pages, because ALeRCE returns the least confident objects first.
* Objects already acquired from TNS are excluded here (they are already in the overlap).
* Each is cross-matched to TNS on its ZTF id and labelled with v4's own rules; one with a
  TNS type counts as TNS-confirmed, one without is photometric-only.
* **Only v4's `require_redshift` is relaxed**, and only inside this acquisition. An object
  nobody took a spectrum of has no spectroscopic redshift, so under v4's gate the
  photometric-only subset would be empty by construction. Every other threshold stands.
  These objects have NaN redshift and peak absolute magnitude — like de Soto et al.'s
  photometric-only subset — and that is one reason to expect lower agreement there.

In [ ]:
# Section 3.3 pool 2 — ALeRCE-native acquisition
# ---------------------------------------------------------------
# Inlined verbatim from btp5/alerce_native.py, which the repository's test suite
# (tests5/) exercises. Edit it there and rebuild this notebook with
# tools/build_notebook_v5.py rather than editing it here.
# ---------------------------------------------------------------

"""
Section 3.3, pool 2, built the way the brief defines it: objects ALeRCE itself
classifies with confidence, sourced FROM ALeRCE — "independent of whether they also
have a TNS label".

The first v5 build did not do this. It attached ALeRCE labels only to objects already
acquired from TNS, which made the ALeRCE-confident pool a strict subset of the
TNS-confirmed pool: the three-pool structure collapsed to two, and Section 6.2's
spectroscopic vs photometric-only comparison could not be computed at all.

Three details below are deliberate, each because the live API does something a
reasonable person would not assume:

1. **Sampling within the confidence cut.** `query_objects` returns matches in
   ASCENDING probability by default: the first objects back for SNIa at p>=0.5 are
   0.500, 0.500, 0.501... Taking "the first N" would build the pool from exactly the
   objects whose ALeRCE labels are least reliable. The full set above the cut is
   fetched and sampled uniformly at random instead — an unbiased draw from
   top-1 >= 0.5, which is the selection Superphot+ used.

2. **Redshift.** v4's quality gate requires one, for peak absolute magnitude. An
   object that ALeRCE classified but TNS never observed spectroscopically has no
   spectroscopic redshift, so under v4's gate every one of them would be rejected and
   the photometric-only subset would be empty by construction. For these objects the
   redshift requirement alone is relaxed (every other gate threshold stands), and
   their redshift / peak_abs_mag are NaN, imputed downstream like any other missing
   feature. That mirrors de Soto et al.'s photometric-only subset, which likewise has
   no spectroscopic redshifts — and it is one reason to EXPECT lower agreement there.

3. **TNS cross-match.** A native object may still have a TNS spectroscopic type. It is
   matched on ZTF object id against v4's `tns_full['ztf_name']`, labelled with v4's
   own rules (classify_sn_subtype, and the same AGN|QSO|Quasar / TDE masks), and
   marked has_tns_label accordingly. TNS types outside our scheme (CV, Varstar, ...)
   are recorded but do not count as a label in our scheme.
"""

from contextlib import contextmanager

import numpy as np
import pandas as pd


SOURCE_NATIVE = 'ALeRCE-native'


def native_query_groups(live_classes):
    """
    {our group -> [ALeRCE classes to query]}, derived from the crosswalk against the
    LIVE class list. SN classes group by our fine label (so Stage 2 gets every SN
    type); other classes group by our coarse label.
    """
    groups = {}
    for c in sorted(live_classes):
        coarse = COARSE_CROSSWALK.get(c)
        if coarse is None:
            continue
        key = FINE_CROSSWALK.get(c) if coarse == 'SNe' else coarse
        if key:
            groups.setdefault(key, []).append(c)
    return groups


def class_total(alerce, classifier, cls, threshold):
    """How many objects ALeRCE classifies as `cls` at top-1 >= threshold (count=True)."""
    try:
        r = alerce.query_objects(classifier=classifier, class_name=cls, probability=threshold,
                                 page_size=1, count=True, format='json', **_survey_kwargs())
        return int(r.get('total') or 0) if isinstance(r, dict) else None
    except Exception:
        return None


def uniform_sample_class(alerce, classifier, cls, threshold, n, rng, page_size=1000):
    """
    n objects drawn uniformly at random from EVERY object of this class above the cut.

    The API returns matches in ascending probability, so one big page is not a uniform
    sample once a class outgrows it: it is the least confident `page_size` objects.
    (Measured: AGN has 10,744 objects at p>=0.5, so a 5,000-row page is its bottom
    half.) Instead, draw uniformly random ranks over the whole ordered set and fetch
    only the pages that contain them.
    """
    kw = _survey_kwargs()
    total = class_total(alerce, classifier, cls, threshold)
    if total is None:
        # count unavailable: fall back to one page and say so if it was truncated
        r = alerce.query_objects(classifier=classifier, class_name=cls, probability=threshold,
                                 page_size=page_size, format='pandas', **kw)
        if r is None or len(r) == 0:
            return pd.DataFrame(), 0
        if len(r) >= page_size:
            print(f'  WARNING: {cls}: total count unavailable and the page is full — this '
                  'sample is biased toward borderline-confidence objects')
        take = min(n, len(r))
        return r.iloc[np.sort(rng.choice(len(r), size=take, replace=False))], len(r)
    if total == 0:
        return pd.DataFrame(), 0

    take = min(n, total)
    ranks = np.sort(rng.choice(total, size=take, replace=False))
    parts = []
    for page in np.unique(ranks // page_size):
        r = alerce.query_objects(classifier=classifier, class_name=cls, probability=threshold,
                                 page_size=page_size, page=int(page) + 1,
                                 format='pandas', **kw)
        if r is None or len(r) == 0:
            continue
        offs = ranks[(ranks // page_size) == page] % page_size
        parts.append(r.iloc[[o for o in offs if o < len(r)]])
    got = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
    return got, total


def query_native_candidates(alerce, classifier, live_classes, per_group, threshold, rng,
                            oversample=8, page_size=1000, exclude_oids=(), verbose=True):
    """
    Candidate pools of ALeRCE-classified objects, one per group, each a uniform random
    sample of everything ALeRCE classifies into that group at top-1 >= threshold.
    Within a multi-class group (AGN = AGN + QSO + Blazar) the draw is allocated in
    proportion to each class's total, so it is uniform over the union.
    """
    exclude = set(map(str, exclude_oids))
    pools, rows = {}, []
    for group, classes in native_query_groups(live_classes).items():
        want = int(per_group * oversample)
        totals = {c: class_total(alerce, classifier, c, threshold) for c in classes}
        known = {c: t for c, t in totals.items() if t}
        if known:
            T = sum(known.values())
            alloc = {c: int(np.floor(want * t / T)) for c, t in known.items()}
            for c in sorted(known, key=lambda c: -(want * known[c] / T - alloc[c])):
                if sum(alloc.values()) >= want:
                    break
                alloc[c] += 1
        else:
            alloc = {c: want for c in classes}
        frames, n_avail = [], 0
        for c in classes:
            if alloc.get(c, 0) <= 0:
                continue
            # Already-acquired objects are removed after sampling; the 8x oversample
            # leaves ample headroom for them.
            got, tot = uniform_sample_class(alerce, classifier, c, threshold,
                                            alloc[c], rng, page_size)
            n_avail += tot or 0
            if len(got):
                got = got.copy()
                got['alerce_query_class'] = c
                frames.append(got)
        if not frames:
            rows.append({'group': group, 'ALeRCE classes': ', '.join(classes),
                         f'available (p>={threshold})': n_avail, 'pooled': 0})
            continue
        pool = pd.concat(frames, ignore_index=True)
        pool['oid'] = pool['oid'].astype(str)
        pool = pool.drop_duplicates('oid')
        pool = pool[~pool['oid'].isin(exclude)]
        pool = pool.sample(frac=1.0, random_state=int(rng.integers(1e9))).reset_index(drop=True)
        pool['native_group'] = group
        pools[group] = pool
        rows.append({'group': group, 'ALeRCE classes': ', '.join(classes),
                     f'available (p>={threshold})': n_avail, 'pooled': len(pool),
                     'median p (pooled)': round(float(pool['probability'].median()), 3)})
    summary = pd.DataFrame(rows)
    if verbose:
        print(summary.to_string(index=False))
        print('\nEach pool is a uniform random sample of ALL objects above the cut — ALeRCE '
              'returns the least confident first, so "the first N" would be the worst-'
              'labelled objects.')
    return pools, summary


def tns_label_from_type(t, sn_subtype_fn):
    """v4's labelling rules, applied to one TNS type string."""
    if not isinstance(t, str) or not t or t == 'nan':
        return None
    st = sn_subtype_fn(t)
    if st:
        return st
    tl = t.lower()
    if any(k in tl for k in ('agn', 'qso', 'quasar')):
        return 'AGN'
    if 'tde' in tl:
        return 'TDE'
    return None


def crossmatch_tns(cands, tns_full, sn_subtype_fn):
    """Attach TNS name / type / redshift / our-scheme label by ZTF object id."""
    out = cands.copy()
    if tns_full is None or 'ztf_name' not in tns_full.columns:
        out['tns_name'] = out['tns_type'] = None
        out['tns_redshift'] = np.nan
        out['tns_label'] = None
        out['has_tns_label'] = False
        return out
    ref = (tns_full.dropna(subset=['ztf_name']).drop_duplicates('ztf_name')
           .set_index('ztf_name'))
    oid = out['oid'].astype(str)
    out['tns_name'] = oid.map(ref['name']) if 'name' in ref else None
    out['tns_type'] = oid.map(ref['type']) if 'type' in ref else None
    out['tns_redshift'] = (pd.to_numeric(oid.map(ref['redshift']), errors='coerce')
                           if 'redshift' in ref else np.nan)
    out['tns_label'] = out['tns_type'].apply(lambda t: tns_label_from_type(t, sn_subtype_fn))
    out['has_tns_label'] = out['tns_label'].notna()
    return out


def to_acquisition_pool(cands):
    """
    Reshape into the row format v4's build_class_to_target consumes. `ztf_name` is the
    ALeRCE oid, so v4's resolve_oid returns it directly and never cone-searches.
    """
    z = cands['tns_redshift'] if 'tns_redshift' in cands else np.nan
    return pd.DataFrame({
        'name': cands['oid'].astype(str).values,
        'ztf_name': cands['oid'].astype(str).values,
        'internal_names': cands['oid'].astype(str).values,
        'ra': cands.get('meanra', np.nan), 'declination': cands.get('meandec', np.nan),
        'redshift': z, 'type': cands.get('tns_type'), 'source': SOURCE_NATIVE,
    }).reset_index(drop=True)


@contextmanager
def redshift_optional(qg):
    """
    Relax ONLY v4's require_redshift, for the duration of the native acquisition, and
    restore it afterwards even if acquisition raises. Mutating the dict in place is
    what reaches v4's quality_gate, whose default argument is this same object.
    """
    saved = qg.get('require_redshift')
    qg['require_redshift'] = False
    try:
        yield qg
    finally:
        qg['require_redshift'] = saved


def finalize_native_rows(features, cands, sn_subtypes):
    """
    Feature rows from the native acquisition, labelled honestly: `label` is the TNS
    label if one exists and NaN otherwise — never ALeRCE's opinion, which lives in the
    alerce_* columns. `passes_v4_gate` is True only when a real redshift exists, i.e.
    when the full, unrelaxed v4 gate would also have accepted the object.
    """
    if not features:
        return pd.DataFrame()
    df = pd.DataFrame(features)
    ref = cands.drop_duplicates('oid').set_index('oid')
    oid = df['id'].astype(str)
    df['label'] = oid.map(ref['tns_label']) if 'tns_label' in ref else np.nan
    df['has_tns_label'] = df['label'].notna()
    df['coarse_label'] = df['label'].apply(
        lambda l: ('SNe' if l in sn_subtypes else l) if isinstance(l, str) else np.nan)
    df['source'] = SOURCE_NATIVE
    df['native_group'] = oid.map(ref['native_group']) if 'native_group' in ref else np.nan
    df['is_synthetic'] = False
    df['from_bts'] = 0
    z = pd.to_numeric(df.get('redshift', np.nan), errors='coerce')
    df['passes_v4_gate'] = np.isfinite(z) & (z > 0)
    return df

In [ ]:
native_rng = np.random.default_rng(RANDOM_STATE)
native_pools, native_summary = query_native_candidates(
    alerce, ALERCE_PRIMARY_CLASSIFIER, live_classes, N_ALERCE_NATIVE_PER_GROUP,
    ALERCE_CONF_THRESHOLD, native_rng, exclude_oids=feature_df['id'].astype(str))

In [ ]:
# Acquire through v4's own loop, with ONLY the redshift requirement relaxed.
native_features, native_cands = [], []
with redshift_optional(QG):
    for group, cands in native_pools.items():
        cands = crossmatch_tns(cands, tns_full, classify_sn_subtype)
        native_cands.append(cands)
        _, feats = build_class_to_target(f'ALERCE_{group}', to_acquisition_pool(cands),
                                         N_ALERCE_NATIVE_PER_GROUP,
                                         out_subdir=f'alerce_{group.lower()}')
        native_features.extend(feats)
print('require_redshift restored to', QG['require_redshift'])

native_cands = pd.concat(native_cands, ignore_index=True) if native_cands else pd.DataFrame()
native_df = finalize_native_rows(native_features, native_cands, SN_SUBTYPES)
if len(native_df):
    print(f"{len(native_df)} ALeRCE-native objects acquired: "
          f"{int(native_df['has_tns_label'].sum())} also TNS-confirmed, "
          f"{int((~native_df['has_tns_label']).sum())} photometric-only")
    print(native_df.groupby('native_group').size().to_string())
feature_df = pd.concat([feature_df, native_df], ignore_index=True)

### ALeRCE labels for every ZTF object

Both the thresholded label (for the ALeRCE-confident pool) and ALeRCE's raw top-1 class
(for the Section 3.1 TDE cross-check, which must see what ALeRCE thinks of every TNS TDE,
not only the confident ones). TESS rows are skipped: they have no ZTF alerts.

In [ ]:
alerce_label_rows = []
ztf_ids = feature_df.loc[feature_df['survey'] == 'ZTF', 'id'].astype(str).unique()
print(f'querying ALeRCE for {len(ztf_ids)} ZTF objects '
      f'({int((feature_df["survey"] == "TESS").sum())} TESS rows skipped by construction)')
for oid in tqdm(ztf_ids, desc='ALeRCE probabilities'):
    try:
        probs = alerce.query_probabilities(oid, format='pandas', **_survey_kwargs())
    except Exception:
        continue
    d = describe_probabilities(probs, ALERCE_PRIMARY_CLASSIFIER, ALERCE_PRIMARY_VERSION,
                               threshold=ALERCE_CONF_THRESHOLD)
    if d is not None:
        d['id'] = oid
        alerce_label_rows.append(d)
n_conf = sum(r['alerce_coarse'] is not None for r in alerce_label_rows)
print(f'{len(alerce_label_rows)} objects have an ALeRCE classification; '
      f'{n_conf} pass the {ALERCE_CONF_THRESHOLD} cut and map into our scheme')

# STEP 6 — Pool construction  *(brief Phase 2C, Section 3.3)*

Three pools, tagged on every row and never collapsed. Every result below names its pool.

In [ ]:
# Section 3.3 — the three pools
# ---------------------------------------------------------------
# Inlined verbatim from btp5/pools.py, which the repository's test suite
# (tests5/) exercises. Edit it there and rebuild this notebook with
# tools/build_notebook_v5.py rather than editing it here.
# ---------------------------------------------------------------

"""
Section 3.3 — the three pools, kept separate throughout.

  1. TNS-confirmed    — objects with a TNS spectroscopic label (v4 behaviour).
  2. ALeRCE-confident — objects ALeRCE itself classifies above the confidence cut,
                        independent of whether they also carry a TNS label.
  3. Overlap          — in both. Much smaller than either parent, and where the
                        sharpest comparison (Section 6.3) happens.

Every table and plot downstream must state which pool it was computed on; the helpers
here carry a `pool` column so that cannot be forgotten.

-------------------------------------------------------------------------------
A structural asymmetry the brief did not anticipate, found while building this
-------------------------------------------------------------------------------
Our stellar-flare class comes from **TESS**, via lightkurve. ALeRCE classifies **ZTF**
alerts. A TESS M-dwarf has no ZTF object id and therefore no ALeRCE probability
vector at all — not a low-confidence one, none.

Consequences, which are reported rather than hidden:
  * The ALeRCE-confident pool cannot contain our TESS flares.
  * The overlap pool therefore has **no stellar_flare class at all**, so the
    ALeRCE-labelled Stage 1 is a 3-way problem (AGN / SNe / TDE) on the overlap,
    against 4-way for the TNS-labelled track.
  * ALeRCE's nearest flare-like classes, CV/Nova and YSO, are ZTF-detected Galactic
    variables — cataclysmic binaries and young stellar objects. They are not the same
    population as UV-Ceti-type M-dwarf flares, so mapping them onto our stellar_flare
    class is an approximation, flagged here and in the summary, not a clean match.

`stage1_class_set` below derives the class list a track can actually support instead
of assuming all four, so the comparison never silently scores a class one side could
not have predicted.
"""

import numpy as np
import pandas as pd

POOL_TNS = 'TNS-confirmed'
POOL_ALERCE = 'ALeRCE-confident'
POOL_OVERLAP = 'overlap'

# ALeRCE classes that stand in for our TESS flare sample, with the caveat above.
FLARE_ANALOGUE_CLASSES = ['CV/Nova', 'YSO']


def attach_alerce_labels(feature_df, alerce_label_rows, verbose=True):
    """
    Join per-object ALeRCE labels onto the feature table.

    `alerce_label_rows` is a list of dicts from
    `alerce_labels.label_from_probabilities`, each carrying an 'id'.
    Objects with no ALeRCE entry get NaN — which for TESS rows is the expected,
    structural outcome described above, not a failure.
    """
    df = feature_df.copy()
    cols = ['alerce_class', 'alerce_prob', 'alerce_coarse', 'alerce_fine',
            'alerce_top1_class', 'alerce_top1_prob']
    lab = pd.DataFrame(alerce_label_rows) if alerce_label_rows else pd.DataFrame(
        columns=['id'] + cols)
    for c in cols:
        if c not in lab.columns:
            lab[c] = np.nan
    df = df.drop(columns=[c for c in cols if c in df.columns])
    df = df.merge(lab[['id'] + cols], on='id', how='left')
    if verbose:
        n = int(df['alerce_coarse'].notna().sum())
        print(f'ALeRCE labels attached to {n}/{len(df)} objects')
        if 'survey' in df.columns:
            by = df.groupby('survey')['alerce_coarse'].apply(lambda s: s.notna().sum())
            for sv, k in by.items():
                tot = int((df['survey'] == sv).sum())
                note = ('  <- structural: ALeRCE classifies ZTF, not TESS'
                        if sv == 'TESS' and k == 0 else '')
                print(f'   {sv}: {k}/{tot}{note}')
    return df


def _as_plain_str(series):
    """numpy string scalars render as np.str_('AGN') in printed class lists and in
    any table built from them. Coerce to builtin str so output stays readable."""
    return series.map(lambda v: str(v) if pd.notna(v) else v)


def build_pools(df, has_tns_col='has_tns_label', verbose=True):
    """
    Tag every row with which pools it belongs to. Returns the frame plus a counts
    table. Nothing is dropped here — membership is marked, so a later step can never
    silently collapse the three pools into one.
    """
    out = df.copy()
    for c in ('alerce_coarse', 'alerce_fine', 'alerce_class'):
        if c in out.columns:
            out[c] = _as_plain_str(out[c])
    if has_tns_col not in out.columns:
        # v4's acquisition is TNS-driven, so every acquired row has a TNS label
        # unless something explicitly says otherwise.
        out[has_tns_col] = out['label'].notna()
    out['in_tns_pool'] = out[has_tns_col].astype(bool)
    out['in_alerce_pool'] = out['alerce_coarse'].notna()
    out['in_overlap_pool'] = out['in_tns_pool'] & out['in_alerce_pool']

    counts = pd.DataFrame([
        {'pool': POOL_TNS, 'n': int(out['in_tns_pool'].sum())},
        {'pool': POOL_ALERCE, 'n': int(out['in_alerce_pool'].sum())},
        {'pool': POOL_OVERLAP, 'n': int(out['in_overlap_pool'].sum())},
    ])
    if verbose:
        print(counts.to_string(index=False))
        print()
        print('Class composition per pool (coarse):')
        for name, mask in [(POOL_TNS, out['in_tns_pool']),
                           (POOL_ALERCE, out['in_alerce_pool']),
                           (POOL_OVERLAP, out['in_overlap_pool'])]:
            sub = out[mask]
            col = 'coarse_label' if name == POOL_TNS else 'alerce_coarse'
            if col not in sub.columns or sub.empty:
                print(f'  {name}: empty')
                continue
            comp = sub[col].value_counts().to_dict()
            print(f'  {name} (by {col}): {comp}')
    return out, counts


def get_pool(df, pool):
    """One pool as its own frame, tagged so downstream output can name it."""
    key = {POOL_TNS: 'in_tns_pool', POOL_ALERCE: 'in_alerce_pool',
           POOL_OVERLAP: 'in_overlap_pool'}[pool]
    sub = df[df[key]].copy()
    sub['pool'] = pool
    return sub


def stage1_class_set(df, label_col, verbose=True):
    """
    Which Stage 1 classes this track can actually support, derived from the data
    rather than assumed.

    Guards the asymmetry above: scoring a 4-class confusion matrix on a pool whose
    ALeRCE side can never contain stellar_flare would report a row of structural
    zeros as if it were a model failure.
    """
    present = sorted(pd.unique(df[label_col].dropna()))
    if verbose:
        print(f'{label_col}: {len(present)} classes present -> {present}')
        missing = set(['AGN', 'SNe', 'TDE', 'stellar_flare']) - set(present)
        if missing:
            print(f'  absent from this pool: {sorted(missing)}')
            if 'stellar_flare' in missing:
                print('  (stellar_flare absent from an ALeRCE-labelled pool is '
                      'expected: TESS objects carry no ZTF alerts to classify)')
    return present

In [ ]:
feature_df = attach_alerce_labels(feature_df, alerce_label_rows)
feature_df, pool_counts = build_pools(feature_df)

# STEP 7 — Host offset  *(brief Phase 3B, Section 4.1)*

ALeRCE's 2025 TDE update added exactly this feature type, because nuclear offset targets
the TDE / AGN / nuclear-transient boundary — Stage 1's weakest link in v4. A TDE is
nuclear by definition; a core-collapse SN sits in its host's disc.

Missing data follows v4's existing convention: NaN plus a presence flag, as with
colour / `has_color`. TESS flare stars are Galactic and correctly have no host galaxy.

In [ ]:
# Section 4.1 — host offset via astro-ghost
# ---------------------------------------------------------------
# Inlined verbatim from btp5/hostoffset.py, which the repository's test suite
# (tests5/) exercises. Edit it there and rebuild this notebook with
# tools/build_notebook_v5.py rather than editing it here.
# ---------------------------------------------------------------

"""
Section 4.1 — host-galaxy offset via astro-ghost (Gagliano et al. 2021).

Why this feature and not the other two candidates: ALeRCE's own 2025 TDE update added
exactly this feature type, because nuclear offset directly targets the
TDE-vs-AGN-vs-nuclear-transient boundary — which v4's diagnostics identified as Stage
1's weakest link. A TDE is nuclear by definition (offset ~0); a core-collapse SN is
displaced into its host's disc; an AGN sits at the nucleus but varies stochastically.

Missing data follows v4's existing convention exactly — NaN plus a presence flag,
the same pattern as color_g_r/has_color. No new missing-data convention is invented.
"""

import numpy as np
import pandas as pd



def ghost_available():
    """astro-ghost is an optional dependency; the pipeline degrades to NaN without it."""
    try:
        import astro_ghost  # noqa: F401
        return True
    except Exception:
        return False


def _empty_host_row():
    return {'host_offset_arcsec': np.nan, 'host_offset_norm': np.nan,
            'has_host_match': 0}


def host_offset_for(ra, dec, snName=None, resolver=None, verbose=False):
    """
    Offset between a transient and its associated host.

    `resolver(ra, dec, name) -> dict|None` is injectable so the control flow can be
    tested without network access; the default uses astro-ghost.

    Returns the three HOST_COLS. A failed association is NaN + has_host_match=0 —
    expected to happen regularly (hostless transients, crowded fields, catalogue
    gaps), not an error.
    """
    if resolver is None:
        resolver = _ghost_resolver
    try:
        res = resolver(ra, dec, snName)
    except Exception as e:
        if verbose:
            print(f'  host lookup failed for ({ra}, {dec}): {type(e).__name__}')
        return _empty_host_row()
    if not res:
        return _empty_host_row()

    offset = res.get('offset_arcsec')
    radius = res.get('host_radius_arcsec')
    if offset is None or not np.isfinite(offset):
        return _empty_host_row()

    row = {'host_offset_arcsec': float(offset),
           # Normalised by host size: 0.1 host-radii is nuclear whatever the distance,
           # while 2 arcsec means very different things for a nearby vs distant galaxy.
           'host_offset_norm': (float(offset) / float(radius)
                                if radius and np.isfinite(radius) and radius > 0
                                else np.nan),
           'has_host_match': 1}
    return row


def _ghost_resolver(ra, dec, snName=None):  # pragma: no cover - needs network
    """Default resolver: astro-ghost's transient-host association."""
    from astro_ghost.ghostHelperFunctions import getTransientHosts
    from astropy.coordinates import SkyCoord
    import astropy.units as u

    coord = SkyCoord(ra=float(ra) * u.deg, dec=float(dec) * u.deg)
    hosts = getTransientHosts(snCoord=[coord], snName=[snName or 'transient'],
                              verbose=False, starcut='gentle', ascentMatch=False)
    if hosts is None or len(hosts) == 0:
        return None
    h = hosts.iloc[0]
    host_ra = h.get('raMean', np.nan)
    host_dec = h.get('decMean', np.nan)
    if not (np.isfinite(host_ra) and np.isfinite(host_dec)):
        return None
    hc = SkyCoord(ra=float(host_ra) * u.deg, dec=float(host_dec) * u.deg)
    offset = float(coord.separation(hc).arcsec)
    # Kron radius in the r band is astro-ghost's most reliably populated size column.
    radius = h.get('rKronRad', np.nan)
    return {'offset_arcsec': offset,
            'host_radius_arcsec': float(radius) if np.isfinite(radius) else np.nan}


def add_host_features(df, ra_col='ra', dec_col='dec', name_col='id',
                      resolver=None, progress=None, verbose=True):
    """
    Add the three HOST_COLS to a feature table.

    Objects without coordinates (our TESS flares) get NaN + has_host_match=0, which
    is correct rather than missing: a Galactic flare star has no host galaxy at all.
    """
    out = df.copy()
    if resolver is None and not ghost_available():
        if verbose:
            print('astro-ghost not installed — host columns filled with NaN and '
                  'has_host_match=0. Install it (pip install astro-ghost) to enable '
                  'this feature; the pipeline runs either way.')
        for c in HOST_COLS:
            out[c] = np.nan if c != 'has_host_match' else 0
        return out

    rows = []
    it = out.iterrows()
    if progress is not None:
        it = progress(it, total=len(out), desc='host offsets')
    for _, r in it:
        ra, dec = r.get(ra_col, np.nan), r.get(dec_col, np.nan)
        if not (np.isfinite(ra) and np.isfinite(dec)):
            rows.append(_empty_host_row())
            continue
        rows.append(host_offset_for(ra, dec, r.get(name_col), resolver=resolver))
    host = pd.DataFrame(rows, index=out.index)
    for c in HOST_COLS:
        out[c] = host[c]

    if verbose:
        n = int(out['has_host_match'].sum())
        print(f'host association succeeded for {n}/{len(out)} objects')
        if 'survey' in out.columns:
            tess = int(((out['survey'] == 'TESS') & (out['has_host_match'] == 0)).sum())
            if tess:
                print(f'  ({tess} of the misses are TESS flare stars, which are '
                      'Galactic and correctly have no host galaxy)')
    return out

In [ ]:
lc_index = index_lightcurves(BASE_DIR)
feature_df = attach_coordinates(feature_df, lc_index)
feature_df = add_host_features(feature_df, progress=tqdm)

# STEP 8 — Noise-model augmentation  *(brief Phase 3C, Section 5)*

Townsend et al. (2026, NoiZTF, arXiv:2602.13036), implemented as their Sects. 3.1–3.3
describe it — checked against the paper, and different in five places from the first v5
build, which had followed a paraphrase:

| | Townsend et al. | first v5 build |
|---|---|---|
| flux units | Jy scaled to AB → **µJy, ZP 23.9** | ZP 27.5 (noise ~25x too small) |
| observation dates | **unchanged** | stretched by (1+z) |
| z_sim | **[z_true, z_true + 0.1]**, p ∝ z² | z_true × 1.2–2.5 |
| error | original σ **rescaled by their Eq. 4** | fresh draw |
| flux | D·f + N(0, σ_z²) | noise on a fresh σ |

The unit is an **inference**: the paper says "janskys scaled to be consistent with the AB
magnitude system" without printing a zero point, but its flux bins (f < 400,
800–1200, > 1600) only make sense in µJy.

* **K-correction:** sncosmo templates (SALT2 at Townsend's [x1, c] = [1.0, 0.2] for Ia;
  Nugent templates for Ib/c and II); the Hogg et al. (2002) form for SLSN, as Townsend
  do. TDEs are outside Townsend's SN-only scope, so the Hogg form applied to them is an
  **extrapolation**. Townsend's own template list (their Table 7) was not verified.
* **Not done: timestamp jitter** — a directly evidenced negative result in the same paper
  (0.1 d jitter degraded SN Ia precision by corrupting colour evolution).
* **Departure:** copies keep only detections at SNR ≥ 5, because this pipeline trains on
  the alert stream, which only ever contains ≥ 5σ detections. Each copy must then pass
  v4's own fit and quality gate, like a real object.
* **Leakage:** Townsend split before augmenting so copies never straddle train/test. Here
  every copy carries `parent_id` and the split drops any copy whose parent is held out.
* **Deferred: full SNANA simulation** (Aleo et al. 2023 / YSE-DR1, ~60,000 per class).
  Mature templates exist for Ia/II/Ibc but not obviously for SLSN or TDE — exactly the
  classes that need help — so the cost is badly matched to the bottleneck.

In [ ]:
# Section 5 — Townsend et al. (2026) noise-model augmentation
# ---------------------------------------------------------------
# Inlined verbatim from btp5/augment.py, which the repository's test suite
# (tests5/) exercises. Edit it there and rebuild this notebook with
# tools/build_notebook_v5.py rather than editing it here.
# ---------------------------------------------------------------

"""
Section 5 — noise-model augmentation for the small classes (SLSN, Ib, Ic, TDE).

Implements the procedure of Townsend et al. 2026 (NoiZTF, arXiv:2602.13036, Sects.
3.1-3.3) as the paper describes it — checked against the paper text, not paraphrased
from memory. Each point below is what the paper does, and where this pipeline departs
from it the departure is stated.

1. **Units.** The error model's constants (e_b=18.0, m=0.04, c=4.7, delta=-0.006) are
   fitted to fluxes "expressed in janskys and scaled to be consistent with the AB
   magnitude system" (their footnote 5), with flux bins at f<400, 800-1200, >1600.
   Those bins only make sense in microjanskys, i.e. AB zero point 23.9 (mag 18 ->
   ~230 uJy). The paper does not print the zero point, so this is an INFERENCE, but
   the alternatives are off by orders of magnitude: at v4's internal ZP of 27.5 the
   bias floor e_b would sit at mag ~24.4, far below anything ZTF detects. v4's ZP is
   untouched for its own photometry; conversion happens only inside this module.

2. **Redshift.** z_sim is drawn from [z_true, z_true + 0.1] with p(z) ~ z^z_scale
   (z_scale in {0, 2}; 2 follows a constant volumetric rate, dV ~ z^2 dz).

3. **Flux.** f_z = D*f_t + N(0, sigma_z^2), with D = D_L(z_true)^2 / D_L(z_sim)^2
   (their Eq. 3), and sigma_z the ORIGINAL error scaled by their Eq. 4:
       sigma_z/sigma_t = [sqrt(D f + e_b^2) + D f (eps+delta)]
                       / [sqrt(f + e_b^2)   + f (eps+delta)],   eps ~ Exp(m f + c).

4. **K-correction** as an additive magnitude offset: sncosmo templates where one
   exists, else the Hogg et al. (2002) form K = -2.5 log10[(1+z_sim)/(1+z_true)],
   which Townsend use for SLSNe. TDEs are outside Townsend's (SN-only) scope; the
   Hogg form is applied to them too and that is an extrapolation, flagged here.

5. **Observation dates are left exactly as observed.** Townsend do not apply time
   dilation ("without altering the cadence"; <~10% for dz<=0.1), and the brief
   forbids timestamp perturbation because 0.1-day jitter measurably degraded SN Ia
   precision. An earlier version of this module stretched the time axis by
   (1+z_sim)/(1+z_true) and anchored each band separately, which shifted the g-band
   and r-band epochs relative to one another — exactly the colour-evolution
   corruption the brief warns about. That is gone.

6. **Quality cuts.** Townsend require a peak detection with SNR>5 and >=5 detections
   with SNR>5. Departure: this pipeline trains on the ZTF *alert stream*, which only
   ever contains >=5-sigma detections, so points that fall below SNR 5 at the new
   redshift are dropped rather than kept as low-SNR forced photometry. The copy then
   also has to pass v4's own quality gate via process_ztf_object, so a copy pushed
   too faint to be a useful training object is rejected, as a real one would be.

7. **Split before augmentation.** Townsend split train/test *before* augmenting "to
   ensure that no copies of the same object appear in both sets". Here every
   synthetic row carries parent_id, and modeling.prepare_split drops any synthetic
   row whose parent landed in a test partition — the same guarantee.

**Deferred: full SNANA simulation** (Aleo et al. 2023 / YSE-DR1, ~60,000 objects per
class). Mature templates exist for Ia/II/Ibc but not obviously for SLSN or TDE —
exactly the classes that need help — so the cost is badly matched to the bottleneck.
"""

import os

import numpy as np
import pandas as pd


UJY_ZP = 23.9          # AB zero point for flux density in microjansky
LN10 = np.log(10.0)

# sncosmo source per class. Townsend's Table 7 (their exact template list) was not
# verified here; these are the standard sncosmo templates for each type, with SALT2
# fixed at their stated [x1, c] = [1.0, 0.2].
SNCOSMO_TEMPLATES = {
    'SN_Ia': ('salt2', {'x1': 1.0, 'c': 0.2}),
    'SN_Ib': ('nugent-sn1bc', {}),
    'SN_Ic': ('nugent-sn1bc', {}),
    'SN_II': ('nugent-sn2p', {}),
    # SLSN and TDE: no clean template -> Hogg fallback (Townsend do this for SLSN).
}
ZTF_BANDS = {1: 'ztfg', 2: 'ztfr'}


# --------------------------------------------------------------------------
# Units
# --------------------------------------------------------------------------

def mag_to_ujy(mag, sigma_mag):
    f = 10.0 ** (-0.4 * (np.asarray(mag, float) - UJY_ZP))
    return f, f * 0.4 * LN10 * np.asarray(sigma_mag, float)


def ujy_to_mag(f, fe):
    f = np.asarray(f, float)
    with np.errstate(divide='ignore', invalid='ignore'):
        mag = UJY_ZP - 2.5 * np.log10(f)
        smag = (2.5 / LN10) * np.asarray(fe, float) / f
    return mag, smag


# --------------------------------------------------------------------------
# The Townsend et al. model
# --------------------------------------------------------------------------

def sample_z_sim(z_true, rng, dz_max=0.1, z_scale=2):
    """Draw z_sim in [z_true, z_true + dz_max] with p(z) ~ z^z_scale (inverse CDF)."""
    a, b = float(z_true), float(z_true) + float(dz_max)
    k = float(z_scale) + 1.0
    u = rng.uniform()
    return float((a ** k + u * (b ** k - a ** k)) ** (1.0 / k))


def distance_factor(z_true, z_sim, cosmo):
    """Their Eq. 3: D = D_L(z_true)^2 / D_L(z_sim)^2."""
    dt = float(cosmo.luminosity_distance(max(z_true, 1e-4)).value)
    ds = float(cosmo.luminosity_distance(max(z_sim, 1e-4)).value)
    return (dt / ds) ** 2


def error_scale_factor(f_t, D, eps, e_b=None, delta=None):
    """
    Their Eq. 4. For very bright points (f >~ 2.8e4 uJy, mag <~ 12.8) the negative
    delta can drive the denominator through zero — far outside the flux range the
    model was fitted on (their highest bin is f>1600). There the pure Poisson+bias
    scaling of their Eq. 1 is used instead, so the factor never goes negative.
    """
    p = NOISE_MODEL
    e_b = p['e_b'] if e_b is None else e_b
    delta = p['delta'] if delta is None else delta
    f = np.clip(np.asarray(f_t, float), 0.0, None)
    num = np.sqrt(D * f + e_b ** 2) + D * f * (eps + delta)
    den = np.sqrt(f + e_b ** 2) + f * (eps + delta)
    poisson = np.sqrt(D * f + e_b ** 2) / np.sqrt(f + e_b ** 2)
    ok = (num > 0) & (den > 0)
    return np.where(ok, num / np.where(ok, den, 1.0), poisson)


def draw_epsilon(f_t, rng, m=None, c=None):
    """eps ~ Exp(lambda_f), lambda_f = m f + c is a RATE; numpy takes scale = 1/rate."""
    p = NOISE_MODEL
    m = p['m'] if m is None else m
    c = p['c'] if c is None else c
    lam = m * np.clip(np.asarray(f_t, float), 0.0, None) + c
    return rng.exponential(1.0 / lam)


# --------------------------------------------------------------------------
# K-correction
# --------------------------------------------------------------------------

_SNCOSMO_CACHE = {}


def sncosmo_available():
    try:
        import sncosmo  # noqa: F401
        return True
    except Exception:
        return False


def hogg_k(z_true, z_sim):
    """Hogg et al. (2002) constant form, used by Townsend for SLSNe."""
    return float(-2.5 * np.log10((1.0 + z_sim) / (1.0 + z_true)))


def k_correction(label, fid, z_true, z_sim, use_sncosmo=True):
    """
    Additive magnitude offset between z_true and z_sim, at peak. Returns (K, method).
    Template K is the difference of the model's observed peak magnitude at the two
    redshifts with its amplitude held fixed, so distance dimming — already in D — is
    not counted twice.
    """
    tmpl = SNCOSMO_TEMPLATES.get(label)
    band = ZTF_BANDS.get(int(fid))
    if use_sncosmo and tmpl and band and sncosmo_available():
        key = (label, band, round(z_true, 4), round(z_sim, 4))
        if key in _SNCOSMO_CACHE:
            return _SNCOSMO_CACHE[key], 'sncosmo:' + tmpl[0]
        try:
            import sncosmo
            model = sncosmo.Model(source=tmpl[0])
            if tmpl[1]:
                model.set(**tmpl[1])
            phase = model.source.peakphase('bessellb')
            mags = []
            for z in (z_true, z_sim):
                model.set(z=z, t0=0.0)
                mags.append(float(model.bandmag(band, 'ab', phase * (1.0 + z))))
            k = mags[1] - mags[0]
            if np.isfinite(k):
                _SNCOSMO_CACHE[key] = k
                return k, 'sncosmo:' + tmpl[0]
        except Exception:
            pass
    return hogg_k(z_true, z_sim), 'hogg2002'


# --------------------------------------------------------------------------
# One synthetic light curve
# --------------------------------------------------------------------------

def augment_detections(det, label, z_true, z_sim, rng, cosmo, snr_min=5.0,
                       subsampling_rate=1.0, use_sncosmo=True):
    """
    Move one object's alert detections from z_true to z_sim.

    `det` has v4's CSV columns (mjd, fid, magpsf, sigmapsf[, ra, dec]). Returns
    (new_det, meta) or (None, reason). mjd is returned untouched, row for row.
    """
    D = distance_factor(z_true, z_sim, cosmo)
    parts, ks = [], {}
    for fid, g in det.groupby('fid'):
        f, fe = mag_to_ujy(g['magpsf'].values, g['sigmapsf'].values)
        k, method = k_correction(label, fid, z_true, z_sim, use_sncosmo=use_sncosmo)
        ks[int(fid)] = (k, method)
        eps = draw_epsilon(f, rng)
        fe_z = fe * error_scale_factor(f, D, eps)
        f_z = D * f * 10 ** (-0.4 * k) + rng.normal(0.0, fe_z)
        out = g.copy()
        mag, smag = ujy_to_mag(f_z, fe_z)
        out['magpsf'], out['sigmapsf'] = mag, smag
        out['_snr'] = f_z / fe_z
        parts.append(out)
    new = pd.concat(parts).sort_values('mjd')

    # alert stream: only >=5-sigma detections exist
    new = new[np.isfinite(new['_snr']) & (new['_snr'] >= snr_min)]
    if subsampling_rate < 1.0 and new.groupby('fid').size().max() > 10:
        new = new[rng.uniform(size=len(new)) < subsampling_rate]
    if len(new) < 5:
        return None, 'fewer than 5 detections above SNR 5 at z_sim'
    new = new.drop(columns='_snr')
    return new, {'D': D, 'k': ks}


# --------------------------------------------------------------------------
# Many synthetic rows
# --------------------------------------------------------------------------

def generate_synthetic_rows(df, classes, copies_per_parent, process_fn, out_dir, rng,
                            cosmo=None, dz_max=0.1, z_scale=2, max_attempts=3,
                            subsampling_rate=1.0, use_sncosmo=True, verbose=True):
    """
    Synthetic training rows for `classes`, `copies_per_parent` per real parent.

    Each copy is written as its own CSV and pushed through `process_fn` — v4's
    process_ztf_object — so it faces exactly the fit and quality gate a real object
    does. Copies that fail are retried with a fresh z_sim up to `max_attempts`, then
    abandoned; the tally says why.
    """
    if cosmo is None:
        from astropy.cosmology import FlatLambdaCDM
        cosmo = FlatLambdaCDM(H0=70, Om0=0.3)   # v4's cosmology, so peak_abs_mag agrees
    os.makedirs(out_dir, exist_ok=True)

    real = df[~df.get('is_synthetic', pd.Series(False, index=df.index)).astype(bool)]
    parents = real[real['label'].isin(classes) & real.get('survey', 'ZTF').eq('ZTF')
                   & real['lc_path'].apply(lambda p: isinstance(p, str) and os.path.exists(p))
                   & np.isfinite(real['redshift'].astype(float)) & (real['redshift'] > 0)]

    carry = ['coarse_label', 'has_tns_label', 'ra', 'dec', 'has_host_match',
             'host_offset_norm']
    rows, tally = [], {}
    for _, p in parents.iterrows():
        det = pd.read_csv(p['lc_path'])
        for k in range(copies_per_parent):
            made = False
            for attempt in range(max_attempts):
                z_sim = sample_z_sim(float(p['redshift']), rng, dz_max, z_scale)
                new, meta = augment_detections(det, p['label'], float(p['redshift']), z_sim,
                                               rng, cosmo, subsampling_rate=subsampling_rate,
                                               use_sncosmo=use_sncosmo)
                if new is None:
                    tally[meta] = tally.get(meta, 0) + 1
                    continue
                sid = f"{p['id']}__aug{k}"
                path = os.path.join(out_dir, f'{sid}.csv')
                new.to_csv(path, index=False)
                row = process_fn(path, sid, p['label'], redshift=z_sim)
                if row is None:
                    tally['v4 fit / quality gate'] = tally.get('v4 fit / quality gate', 0) + 1
                    continue
                row = dict(row)
                for c in carry:
                    if c in p.index:
                        row[c] = p[c]
                # Offset in host-radii is distance-free; the arcsec offset shrinks with
                # angular-diameter distance as the same galaxy moves further away.
                if 'host_offset_arcsec' in p.index and np.isfinite(p['host_offset_arcsec']):
                    da_t = float(cosmo.angular_diameter_distance(p['redshift']).value)
                    da_s = float(cosmo.angular_diameter_distance(z_sim).value)
                    row['host_offset_arcsec'] = float(p['host_offset_arcsec']) * da_t / da_s
                else:
                    row['host_offset_arcsec'] = np.nan
                for c in ['bts_peakabs', 'bts_rise', 'bts_fade', 'bts_duration']:
                    row[c] = np.nan      # BTS measured the PARENT; nothing for the copy
                row.update({'from_bts': 0, 'is_synthetic': True, 'parent_id': p['id'],
                            'z_true': float(p['redshift']), 'z_sim': z_sim,
                            'k_method': sorted({m for _, m in meta['k'].values()})[0],
                            'lc_path': path})
                rows.append(row)
                made = True
                break
            if not made:
                tally['abandoned after retries'] = tally.get('abandoned after retries', 0) + 1

    out = pd.DataFrame(rows)
    if verbose:
        print(f'{len(parents)} eligible parents in {list(classes)}; '
              f'{len(out)} synthetic rows kept')
        if len(out):
            print(out.groupby('label').size().rename('synthetic rows').to_string())
            print('K-correction method used:', out['k_method'].value_counts().to_dict())
        if tally:
            print('rejections:', tally)
    return out


def plan_augmentation(counts, copies_per_parent, classes_to_augment, verbose=True):
    """What augmentation will attempt, per class, before anything is generated."""
    rows = [{'class': c, 'real': int(counts.get(c, 0)),
             'copies_per_parent': copies_per_parent,
             'max_synthetic': int(counts.get(c, 0)) * copies_per_parent}
            for c in classes_to_augment]
    plan = pd.DataFrame(rows)
    if verbose:
        print(plan.to_string(index=False))
        starved = plan[plan['real'] == 0]
        if len(starved):
            print(f"WARNING: no real objects to augment for {list(starved['class'])}")
        if copies_per_parent > 3:
            print('NOTE: >3 copies per parent. Copies of one parent are correlated; past '
                  '~3x this inflates apparent training size without independent information.')
    return plan


def augmented_rows_are_flagged(df, flag_col='is_synthetic'):
    """Synthetic rows must be identifiable and traceable to a parent."""
    if flag_col not in df.columns:
        return False, f'no {flag_col} column — synthetic rows are untraceable'
    n = int(df[flag_col].astype(bool).sum())
    if n and ('parent_id' not in df.columns
              or df.loc[df[flag_col].astype(bool), 'parent_id'].isna().any()):
        return False, 'synthetic rows without a parent_id cannot be kept out of test sets'
    return True, f'{n} synthetic / {len(df)} total rows, flagged by {flag_col} with parent_id'


def augmentation_consistency(syn, parents, verbose=True):
    """
    Does a synthetic copy's fitted absolute magnitude agree with its parent's?

    After redshift rescaling it should differ only by the K-correction. Reported
    rather than asserted, because two effects are real and worth seeing:

      * Truncation bias. Copies pushed near ZTF's alert depth keep only the points
        that scattered bright past SNR 5, so their fitted peak is biased bright.
        Measured on synthetic parents: median residual -0.12 mag at dz<=0.03,
        -0.30 mag at dz<=0.1. Faint REAL objects suffer the same bias, so this is
        realism, not error, but it means copies are not unbiased clones.
      * v4's peak_abs_mag comes from the Villar A parameter, which is degenerate
        with t0 and tau_rise. Under an identical noise realization its scatter was
        0.255 mag against 0.035 mag for the fitted curve's actual peak — about 7x
        noisier. That is a v4 property, preserved here, and flagged as the single
        cheapest improvement to v4's feature set.
    """
    if syn is None or len(syn) == 0:
        return None
    pm = parents.set_index('id')['peak_abs_mag']
    k = [hogg_k(zt, zs) for zt, zs in zip(syn['z_true'], syn['z_sim'])]
    resid = syn['peak_abs_mag'].values - syn['parent_id'].map(pm).values - np.array(k)
    out = {'n': int(len(syn)), 'median_residual_mag': float(np.nanmedian(resid)),
           'residual_spread_mag': float(np.nanstd(resid)),
           'median_dz': float(np.median(syn['z_sim'] - syn['z_true']))}
    if verbose:
        print(f"{out['n']} copies: fitted peak_abs_mag minus (parent + K) has median "
              f"{out['median_residual_mag']:+.3f} mag, spread {out['residual_spread_mag']:.3f} "
              f"(median dz {out['median_dz']:.3f}).")
        print('  Negative = copies fit brighter than their parent predicts: the '
              'truncation bias of near-limit detection. Spread is dominated by v4 '
              'deriving peak_abs_mag from the Villar A parameter.')
    return out

In [ ]:
print('sncosmo available:', sncosmo_available())
aug_plan = plan_augmentation(feature_df[feature_df['is_synthetic'] == False]['label']
                             .value_counts().to_dict(),
                             AUGMENT_COPIES_PER_PARENT, AUGMENT_CLASSES)
synthetic_df = generate_synthetic_rows(
    feature_df, AUGMENT_CLASSES, AUGMENT_COPIES_PER_PARENT, process_ztf_object,
    os.path.join(BASE_DIR, 'synthetic'), np.random.default_rng(RANDOM_STATE),
    dz_max=AUGMENT_DZ_MAX, z_scale=AUGMENT_Z_SCALE,
    subsampling_rate=AUGMENT_SUBSAMPLING_RATE, use_sncosmo=True)

In [ ]:
# Copies are training material only: they belong to no pool, and ALeRCE never saw them.
if len(synthetic_df):
    synthetic_df['source'] = 'synthetic'
    synthetic_df['passes_v4_gate'] = True
    for c in ('in_tns_pool', 'in_alerce_pool', 'in_overlap_pool'):
        synthetic_df[c] = False
    augmentation_check = augmentation_consistency(synthetic_df, feature_df)
    feature_df = pd.concat([feature_df, synthetic_df], ignore_index=True)
else:
    augmentation_check = None
ok, msg = augmented_rows_are_flagged(feature_df)
print(msg)
assert ok, msg

# STEP 9 — Model training, both tracks  *(brief Phase 4, Section 7)*

One parameterised code path, called once per label source. Three headline models;
HistGradientBoosting and Bagging (SVM) remain available via `build_diagnostic_models()`
only, because v4 measured all five converging to within ~1 point at Stage 1.

* **TNS track:** every row with a TNS label (plus the curated TESS flares, as in v4) that
  clears v4's *full* gate, with synthetic copies in training only.
* **ALeRCE track:** the ALeRCE-confident pool, with `label` / `coarse_label` replaced by
  ALeRCE's crosswalked output. Its Stage 2 is 4-way (Ia / Ibc / II / SLSN) against the
  TNS track's 5-way — by design; Ib and Ic are not merged on the TNS side to match.

Stage 2's split is inherited from Stage 1's partitions, so an SN held out of Stage 1 can
never train Stage 2.

In [ ]:
# Phase 4 — dual-track training
# ---------------------------------------------------------------
# Inlined verbatim from btp5/modeling.py, which the repository's test suite
# (tests5/) exercises. Edit it there and rebuild this notebook with
# tools/build_notebook_v5.py rather than editing it here.
# ---------------------------------------------------------------

"""
Phase 4 — model training, run once per label source.

The brief is explicit that the two tracks must share one parameterized code path
rather than two copy-pasted blocks, so `run_track` takes the label source as an
argument and everything else is identical by construction. That also means a fix
applied to one track cannot fail to reach the other.

Models, evaluation protocol and the one-global-split discipline are carried over from
v4 unchanged: three headline models, class_weight='balanced', 5-fold CV with an
adaptive fold count, and a bootstrap 95% CI on test accuracy.
"""

import numpy as np
import pandas as pd
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier



def build_models(random_state=RANDOM_STATE, n_jobs=-1):
    """The three headline models, keyed by the display names used in every table."""
    return {
        'Logistic Regression': LogisticRegression(
            max_iter=2000, class_weight='balanced', random_state=random_state),
        'Random Forest': RandomForestClassifier(
            n_estimators=300, class_weight='balanced',
            random_state=random_state, n_jobs=n_jobs),
        'Bagging Classifier': BaggingClassifier(
            estimator=DecisionTreeClassifier(class_weight='balanced',
                                             random_state=random_state),
            n_estimators=300, max_samples=0.8, max_features=0.7,
            random_state=random_state, n_jobs=n_jobs),
    }


def safe_cv_folds(y, desired=5, verbose=True):
    """v4's adaptive fold count: k-fold needs at least k members of the rarest class."""
    counts = pd.Series(y).value_counts()
    folds = int(min(desired, counts.min()))
    if folds < desired and verbose:
        print(f'  NOTE: CV folds {desired} -> {max(folds, 2)} '
              f'(rarest class has {int(counts.min())} training examples)')
    return max(folds, 2)


def bootstrap_ci(y_true, y_pred, n_boot=2000, random_state=RANDOM_STATE):
    """Percentile bootstrap 95% CI on accuracy — v4's protocol, unchanged."""
    rng = np.random.default_rng(random_state)
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    n = len(y_true)
    if n == 0:
        return (float('nan'), float('nan'))
    accs = [float((y_true[i] == y_pred[i]).mean())
            for i in (rng.integers(0, n, n) for _ in range(n_boot))]
    return (float(np.percentile(accs, 2.5)), float(np.percentile(accs, 97.5)))


def impute(X_train, X_test):
    """Median imputation fitted on train only. LogReg and SVMs cannot take NaN, and
    our feature table has structural NaNs (no colour, no host, no redshift, and the
    bts_* columns whenever a candidate came from TNS rather than BTS).

    A column that is entirely NaN in training has no median at all; numpy warns and
    returns NaN, so those fall back to 0.0 after standardisation-safe handling. That
    is expected for whole-column gaps, not a data problem, so the warning is
    suppressed here rather than printed once per fit."""
    with np.errstate(invalid='ignore'):
        import warnings as _w
        with _w.catch_warnings():
            _w.simplefilter('ignore', RuntimeWarning)
            med = np.nanmedian(X_train, axis=0)
    med = np.where(np.isfinite(med), med, 0.0)
    return (np.where(np.isfinite(X_train), X_train, med),
            np.where(np.isfinite(X_test), X_test, med))


def prepare_split(df, feature_cols, label_col, test_size=0.2,
                  random_state=RANDOM_STATE, exclude_synthetic_from_test=True,
                  verbose=True):
    """
    One stratified split for a track.

    Synthetic (augmented) rows are forced into the training partition: a synthetic
    copy shares its parent's light curve, so letting one into the test set is leakage
    dressed up as extra data.
    """
    sub = df[df[label_col].notna()].reset_index(drop=True)
    y_raw = sub[label_col].astype(str).values

    # Classes with a single member cannot be stratified or scored; drop with a notice.
    vc = pd.Series(y_raw).value_counts()
    too_small = vc[vc < 2].index.tolist()
    if too_small:
        if verbose:
            print(f'  dropping {too_small} — fewer than 2 objects, cannot split')
        keep = ~pd.Series(y_raw).isin(too_small).values
        sub, y_raw = sub[keep].reset_index(drop=True), y_raw[keep]

    le = LabelEncoder()
    y = le.fit_transform(y_raw)
    X = sub[feature_cols].astype(float).values
    idx = np.arange(len(sub))

    synth = (sub['is_synthetic'].astype(bool).values
             if exclude_synthetic_from_test and 'is_synthetic' in sub.columns
             else np.zeros(len(sub), bool))

    n_dropped_children = 0
    if synth.any():
        real_idx = idx[~synth]
        tr_r, te_r = train_test_split(real_idx, test_size=test_size,
                                      stratify=y[real_idx], random_state=random_state)
        # A synthetic copy is a noisier view of its parent's own light curve. If the
        # parent is being held out, training on its copy lets the model rehearse the
        # test object — so such copies are dropped. This is the guarantee Townsend et
        # al. (2026) get by splitting before augmenting.
        synth_idx = idx[synth]
        if 'parent_id' in sub.columns:
            test_ids = set(sub['id'].iloc[te_r].astype(str))
            parent = sub['parent_id'].astype(str).values
            keep_synth = np.array([parent[i] not in test_ids for i in synth_idx], bool)
            n_dropped_children = int((~keep_synth).sum())
            synth_idx = synth_idx[keep_synth]
        idx_train = np.sort(np.concatenate([tr_r, synth_idx]))
        idx_test = np.sort(te_r)
    else:
        idx_train, idx_test = train_test_split(idx, test_size=test_size,
                                               stratify=y, random_state=random_state)
        idx_train, idx_test = np.sort(idx_train), np.sort(idx_test)

    Xtr, Xte = impute(X[idx_train], X[idx_test])
    scaler = StandardScaler()
    Xtr_s = scaler.fit_transform(Xtr)
    Xte_s = scaler.transform(Xte)

    if verbose:
        print(f'  {len(idx_train)} train / {len(idx_test)} test, '
              f'{len(le.classes_)} classes'
              + (f', {int(synth.sum()) - n_dropped_children} synthetic rows in train '
                 f'({n_dropped_children} dropped: their parent is in test)'
                 if synth.any() else ''))
    return {'df': sub, 'X': X, 'le': le, 'y': y, 'scaler': scaler,
            'idx_train': idx_train, 'idx_test': idx_test,
            'X_train': Xtr_s, 'X_test': Xte_s,
            'y_train': y[idx_train], 'y_test': y[idx_test],
            'feature_cols': list(feature_cols)}


def prepare_nested_split(parent, label_col, row_mask, verbose=True):
    """
    Stage 2's split, INHERITED from Stage 1 rather than recomputed.

    `row_mask` selects the Stage 2 rows (the SN-labelled ones) as positions into the
    parent split's frame. Training rows are the masked rows inside the parent's
    training partition; evaluation rows are the masked rows inside its test partition.
    Calling prepare_split() on the SN subset instead would draw a fresh split, and an
    SN object held out of Stage 1 could land in Stage 2's training set — leakage that
    inflates the cascade without being visible in any single number.
    """
    pdf = parent['df']
    mask = np.asarray(row_mask, bool)
    tr = np.array([i for i in parent['idx_train'] if mask[i]], dtype=int)
    te = np.array([i for i in parent['idx_test'] if mask[i]], dtype=int)

    labels = pdf[label_col].astype(str).values
    keep_tr = np.array([i for i in tr if labels[i] not in ('nan', 'None')], dtype=int)
    keep_te = np.array([i for i in te if labels[i] not in ('nan', 'None')], dtype=int)
    if len(keep_tr) < 5 or len(np.unique(labels[keep_tr])) < 2:
        return None

    le = LabelEncoder()
    le.fit(labels[np.concatenate([keep_tr, keep_te])] if len(keep_te)
           else labels[keep_tr])
    # An unseen class in test would break inverse_transform; drop those rows loudly.
    known = set(le.classes_)
    keep_te = np.array([i for i in keep_te if labels[i] in known], dtype=int)

    X = parent['X']
    Xtr, Xte = impute(X[keep_tr], X[keep_te] if len(keep_te) else X[keep_tr][:0])
    scaler = StandardScaler()
    Xtr_s = scaler.fit_transform(Xtr)
    Xte_s = scaler.transform(Xte) if len(keep_te) else Xte

    if verbose:
        print(f'  {len(keep_tr)} train / {len(keep_te)} test SN rows, '
              f'inherited from Stage 1 partitions ({len(le.classes_)} classes)')
    return {'df': pdf, 'X': X, 'le': le, 'y': None, 'scaler': scaler,
            'idx_train': keep_tr, 'idx_test': keep_te,
            'X_train': Xtr_s, 'X_test': Xte_s,
            'y_train': le.transform(labels[keep_tr]),
            'y_test': le.transform(labels[keep_te]) if len(keep_te) else np.array([], int),
            'feature_cols': list(parent['feature_cols'])}


def evaluate(split, stage_name, random_state=RANDOM_STATE, verbose=True):
    """Fit the three models on a prepared split and score them."""
    models = build_models(random_state=random_state)
    folds = safe_cv_folds(split['y_train'], verbose=verbose)
    labels_idx = np.arange(len(split['le'].classes_))

    fitted, rows, confusions = {}, [], {}
    for name, model in models.items():
        cv = cross_val_score(model, split['X_train'], split['y_train'],
                             cv=folds, scoring='accuracy')
        model.fit(split['X_train'], split['y_train'])
        pred = model.predict(split['X_test'])
        acc = accuracy_score(split['y_test'], pred)
        f1 = f1_score(split['y_test'], pred, average='macro', zero_division=0)
        lo, hi = bootstrap_ci(split['y_test'], pred, random_state=random_state)
        fitted[name] = model
        confusions[name] = confusion_matrix(split['y_test'], pred, labels=labels_idx)
        rows.append({'Model': name, 'CV folds': folds,
                     'CV accuracy': cv.mean(), 'CV std': cv.std(),
                     'Test accuracy': acc, 'Macro-F1': f1,
                     'CI low': lo, 'CI high': hi, 'n_test': len(split['y_test'])})
        if verbose:
            print(f'  [{stage_name}] {name:<22} acc {acc:.3f} '
                  f'[{lo:.3f}-{hi:.3f}]  macro-F1 {f1:.3f}')
    return fitted, pd.DataFrame(rows), confusions


def run_track(df, feature_cols, coarse_label_col, fine_label_col, track_name,
              sn_coarse_value='SNe', random_state=RANDOM_STATE, verbose=True):
    """
    One full two-stage run for one label source.

    Stage 2 trains on the SN-labelled rows inside Stage 1's TRAINING partition and is
    evaluated on the SN-labelled rows inside Stage 1's TEST partition, so Stage 2
    cannot see a Stage 1 test object. Same nesting discipline as v4.
    """
    if verbose:
        print(f'\n=== {track_name}: Stage 1 (coarse) ===')
    s1 = prepare_split(df, feature_cols, coarse_label_col,
                       random_state=random_state, verbose=verbose)
    s1_models, s1_results, s1_cm = evaluate(s1, f'{track_name}/S1',
                                            random_state=random_state, verbose=verbose)

    if verbose:
        print(f'\n=== {track_name}: Stage 2 (SN subtypes) ===')
    sn_mask = ((s1['df'][coarse_label_col] == sn_coarse_value)
               & s1['df'][fine_label_col].notna()).values
    s2 = s2_models = s2_results = s2_cm = None
    if sn_mask.sum() >= 10:
        s2 = prepare_nested_split(s1, fine_label_col, sn_mask, verbose=verbose)
    if s2 is not None and len(s2['y_test']) > 0:
        s2_models, s2_results, s2_cm = evaluate(s2, f'{track_name}/S2',
                                                random_state=random_state,
                                                verbose=verbose)
    else:
        s2 = None
        if verbose:
            print(f'  Stage 2 skipped: {int(sn_mask.sum())} SN-labelled rows is too '
                  'few to inherit a usable train/test split from Stage 1.')

    return {'track': track_name,
            'stage1': {'split': s1, 'models': s1_models,
                       'results': s1_results, 'confusions': s1_cm},
            'stage2': ({'split': s2, 'models': s2_models,
                        'results': s2_results, 'confusions': s2_cm}
                       if s2 is not None else None)}


def track_summary(tracks):
    """Section 6.3's first two bullets: accuracy and macro-F1 per model, per track,
    per stage, with the pool each was computed on named."""
    rows = []
    for t in tracks:
        for stage in ('stage1', 'stage2'):
            st = t.get(stage)
            if st is None or st['results'] is None:
                continue
            for _, r in st['results'].iterrows():
                rows.append({'Track': t['track'], 'Stage': stage,
                             'Model': r['Model'], 'Accuracy': r['Test accuracy'],
                             'Macro-F1': r['Macro-F1'],
                             '95% CI': f"{r['CI low']:.3f}-{r['CI high']:.3f}",
                             'n_test': int(r['n_test'])})
    return pd.DataFrame(rows)


def build_diagnostic_models(random_state=RANDOM_STATE, n_jobs=-1):
    """
    The five-model set, for the diagnostics section only.

    Section 7 of the brief: the headline comparison is deliberately three models,
    because v4 measured all five converging to within ~1 point at Stage 1, so the
    extra two add noise to the headline table without adding information. They stay
    available here for cross-checking, under a separate name so they can never
    silently replace the three-model headline set.
    """
    from sklearn.ensemble import HistGradientBoostingClassifier
    from sklearn.svm import SVC

    models = build_models(random_state=random_state, n_jobs=n_jobs)
    models['Bagging (SVM)'] = BaggingClassifier(
        estimator=SVC(kernel='rbf', class_weight='balanced', random_state=random_state),
        n_estimators=50, max_samples=0.8, max_features=0.7,
        random_state=random_state, n_jobs=n_jobs)
    models['HistGradientBoosting'] = HistGradientBoostingClassifier(
        random_state=random_state)
    return models


def oof_predictions(df, feature_cols, label_col, model_name, n_splits=5,
                    random_state=RANDOM_STATE, verbose=True):
    """
    Out-of-fold predictions for every REAL labelled row: each object is predicted by a
    model that never saw it.

    Why this, for the agreement matrices, rather than the held-out test split: the test
    split is ~20% of the sample, which at Stage 2 means a handful of objects per class
    — too few for a class-by-class agreement matrix to mean anything. Out-of-fold
    predictions cover every object once, with the same no-peeking guarantee. de Soto
    et al. likewise build their purity/completeness matrices from validation folds.

    Synthetic rows are only ever trained on, and only in folds where their parent is
    also on the training side.
    """
    from sklearn.model_selection import StratifiedKFold

    sub = df[df[label_col].notna()].reset_index(drop=True)
    synth = (sub['is_synthetic'].astype(bool).values if 'is_synthetic' in sub.columns
             else np.zeros(len(sub), bool))
    y_raw = sub[label_col].astype(str).values
    real_idx = np.flatnonzero(~synth)
    vc = pd.Series(y_raw[real_idx]).value_counts()
    keep_classes = set(vc[vc >= 2].index)
    real_idx = np.array([i for i in real_idx if y_raw[i] in keep_classes])
    if len(real_idx) < 10 or len(keep_classes) < 2:
        return pd.DataFrame(columns=['id', 'true', 'pred', 'confidence'])

    le = LabelEncoder().fit(sorted(keep_classes))
    X = sub[feature_cols].astype(float).values
    k = int(min(n_splits, pd.Series(y_raw[real_idx]).value_counts().min()))
    k = max(k, 2)
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=random_state)

    ids = sub['id'].astype(str).values
    parent = (sub['parent_id'].astype(str).values if 'parent_id' in sub.columns
              else np.array([''] * len(sub)))
    synth_idx = np.flatnonzero(synth & np.isin(y_raw, list(keep_classes)))

    preds, confs, rows = {}, {}, []
    for tr_pos, te_pos in skf.split(real_idx, y_raw[real_idx]):
        tr, te = real_idx[tr_pos], real_idx[te_pos]
        tr_ids = set(ids[tr])
        kids = [i for i in synth_idx if parent[i] in tr_ids]
        tr_all = np.concatenate([tr, np.array(kids, int)])
        Xtr, Xte = impute(X[tr_all], X[te])
        sc = StandardScaler().fit(Xtr)
        m = build_models(random_state=random_state)[model_name]
        m.fit(sc.transform(Xtr), le.transform(y_raw[tr_all]))
        Xte_s = sc.transform(Xte)
        p = m.predict(Xte_s)
        c = (m.predict_proba(Xte_s).max(axis=1) if hasattr(m, 'predict_proba')
             else np.full(len(te), np.nan))
        for i, pi, ci in zip(te, p, c):
            rows.append({'id': ids[i], 'true': y_raw[i],
                         'pred': le.inverse_transform([pi])[0], 'confidence': float(ci)})
    out = pd.DataFrame(rows)
    if verbose:
        acc = float((out['true'] == out['pred']).mean()) if len(out) else float('nan')
        print(f'  {model_name}: {len(out)} out-of-fold predictions ({k} folds), '
              f'OOF accuracy {acc:.3f}')
    return out


def predict_unseen(train_df, feature_cols, label_col, model_name, target_df,
                   random_state=RANDOM_STATE):
    """
    Predictions for objects that have NO label on this track (the photometric-only
    ALeRCE objects), from a model trained on every labelled row. Those objects were
    never in training, so this carries the same no-peeking guarantee as OOF.
    """
    tr = train_df[train_df[label_col].notna()]
    if len(tr) == 0 or len(target_df) == 0:
        return pd.DataFrame(columns=['id', 'pred', 'confidence'])
    le = LabelEncoder().fit(tr[label_col].astype(str))
    Xtr, Xte = impute(tr[feature_cols].astype(float).values,
                      target_df[feature_cols].astype(float).values)
    sc = StandardScaler().fit(Xtr)
    m = build_models(random_state=random_state)[model_name]
    m.fit(sc.transform(Xtr), le.transform(tr[label_col].astype(str)))
    Xs = sc.transform(Xte)
    conf = m.predict_proba(Xs).max(axis=1) if hasattr(m, 'predict_proba') else np.nan
    return pd.DataFrame({'id': target_df['id'].astype(str).values,
                         'pred': le.inverse_transform(m.predict(Xs)),
                         'confidence': conf})

In [ ]:
# track frames, v4 adapters and the two-track diagnostics runner
# ---------------------------------------------------------------
# Inlined verbatim from btp5/diagnostics.py, which the repository's test suite
# (tests5/) exercises. Edit it there and rebuild this notebook with
# tools/build_notebook_v5.py rather than editing it here.
# ---------------------------------------------------------------

"""
Running v4's diagnostics — unchanged — on both label tracks.

v4's diagnostic cells are preserved verbatim. What changes is only what they are
pointed at:

  * **Real rows only.** v4's diagnostics draw their own random train/test splits
    (ablation, redshift check, learning curve, ...). Handed a table containing
    synthetic copies, those splits would put a copy in training and its parent in
    test, and every number would be inflated. They get real objects only.

  * **Both tracks, by rebinding, not copy-pasting.** The TNS pass runs the cells as
    ordinary notebook cells. The ALeRCE pass re-executes the SAME source (a test
    asserts it is byte-identical to the visible cells) with `feature_df_full`,
    `SN_SUBTYPES` and v4's modelling objects rebound to the ALeRCE-labelled track,
    and restored afterwards.

  * **Two cells are TNS-scheme-only.** v4's class-scheme comparison and class-imbalance
    test hardcode the 5-way vocabulary (SN_Ib, SN_Ic). ALeRCE never separates Ib from
    Ic, and its SN scheme already IS the "4-way Ia / Ib-c / II / SLSN" row of the
    class-scheme comparison — so those two run on the TNS track only, for that reason.

  * **v4 modelling objects via adapters.** The cascade and permutation-importance
    cells expect v4's `split` / `stage1` / `stage2a` dictionaries. `v4_adapters`
    presents v5's track output in that shape, rather than editing the cells.
"""

import numpy as np
import pandas as pd


ALERCE_SN_CLASSES = ['SN_Ia', 'SN_Ibc', 'SN_II', 'SLSN']

# v4 notebook cell index -> why it cannot run on the ALeRCE track
TNS_SCHEME_ONLY = {
    35: ('class-scheme comparison: its merge maps are written for the 5-way TNS '
         'vocabulary, and ALeRCE\'s SN scheme is already its "4-way Ia / Ib-c" row'),
    37: ('class-imbalance test: its target proportions name SN_Ib and SN_Ic '
         'separately, which ALeRCE never emits'),
}

_MISSING = object()


def is_real(df):
    s = df['is_synthetic'] if 'is_synthetic' in df.columns else pd.Series(False, index=df.index)
    return ~s.fillna(False).astype(bool)


def tns_training_frame(df):
    """
    What the TNS track trains on: every row with a TNS label that clears v4's full,
    unrelaxed quality gate — real objects plus their synthetic copies (which the split
    keeps parent-aware). ALeRCE-native objects without a redshift are excluded: they
    only cleared a gate with the redshift requirement switched off.
    """
    ok = df['label'].notna()
    if 'passes_v4_gate' in df.columns:
        ok &= df['passes_v4_gate'].fillna(True).astype(bool)
    return df[ok].copy()


def tns_diagnostic_frame(df):
    """TNS track, real objects only — for v4's own random-split diagnostics."""
    t = tns_training_frame(df)
    return t[is_real(t)].copy()


def alerce_track_frame(df, real_only=False):
    """
    The ALeRCE-labelled view: ALeRCE-confident objects with `label` and `coarse_label`
    REPLACED by ALeRCE's crosswalked output, so v4's code (which reads `label`) sees
    ALeRCE's labels. The TNS label is kept alongside as `tns_label`.
    """
    col = 'in_alerce_pool' if 'in_alerce_pool' in df.columns else None
    f = df[df[col]] if col else df[df['alerce_coarse'].notna()]
    f = f.copy()
    f['tns_label'] = f['label']
    f['coarse_label'] = f['alerce_coarse']
    f['label'] = np.where(f['alerce_coarse'] == 'SNe', f['alerce_fine'], f['alerce_coarse'])
    f = f[f['label'].notna()]
    if real_only:
        f = f[is_real(f)]
    return f


def v4_adapters(track):
    """
    v5 track output -> v4's (split, stage1, stage2a) dictionaries.

    v4 fed RAW features into its scalers, which worked because v4 imputed the whole
    table up front. v5's table carries structural NaNs (host offset, native redshift),
    so `split['X']` here is imputed with Stage 1's TRAINING medians — the same values
    Stage 1 itself used — keeping v4's cascade code valid without editing it.
    """
    s1 = track['stage1']
    sp1 = s1['split']
    X_imp = impute(sp1['X'][sp1['idx_train']], sp1['X'])[1]
    split = {'feature_df': sp1['df'], 'X': X_imp,
             'idx_train': sp1['idx_train'], 'idx_test': sp1['idx_test']}

    def stage(st):
        if st is None or st.get('models') is None:
            return None
        sp = st['split']
        return {'models': st['models'], 'scaler': sp['scaler'], 'le': sp['le'],
                'X_test': sp['X_test'], 'y_test': sp['y_test'],
                'results': st['results'].rename(columns={'Macro-F1': 'Macro F1'})}

    return split, stage(s1), stage(track.get('stage2'))


def enough_per_class(frame, classes, min_per_class):
    counts = frame['label'].value_counts()
    short = {c: int(counts.get(c, 0)) for c in classes if counts.get(c, 0) < min_per_class}
    return (not short), short


def run_v4_cells(sources, namespace, bindings, label, min_per_class=8, verbose=True):
    """
    Execute v4 diagnostic cell sources with `bindings` temporarily in scope, then put
    every rebound name back exactly as it was (including removing names that did not
    exist before). Skips — loudly — when any SN class is too small to split.
    """
    frame = bindings.get('feature_df_full')
    classes = bindings.get('SN_SUBTYPES', [])
    if frame is not None and classes:
        ok, short = enough_per_class(frame, classes, min_per_class)
        if not ok:
            if verbose:
                print(f'[{label}] skipped: too few objects to split in {short} '
                      f'(need >= {min_per_class} per class)')
            return False
    saved = {k: namespace.get(k, _MISSING) for k in bindings}
    try:
        namespace.update(bindings)
        for i, src in sources:
            if verbose:
                print(f'\n----- [{label}] v4 diagnostic cell {i} -----')
            exec(compile(src, f'<v4 diagnostic {i}, {label}>', 'exec'), namespace)
    finally:
        for k, v in saved.items():
            if v is _MISSING:
                namespace.pop(k, None)
            else:
                namespace[k] = v
    return True


def v4_impute(frame, feature_cols):
    """
    Impute a frame the way v4's assemble_feature_df did, so v4's diagnostics see the
    fully imputed table they were written for.

    v4 filled every feature gap up front: gr_* and all other feature columns with a
    CLASS-INDEPENDENT median, peak_abs_mag with a 0.0 sentinel (far from any SN value,
    so a tree can split cleanly on "not extragalactic"). v5 adds columns after that
    step — host offset, redshift-less ALeRCE-native rows, synthetic copies — so a
    column can be entirely NaN (e.g. every host column, when astro-ghost cannot
    resolve hosts), and HistGradientBoosting's binner raises on an all-NaN column.
    A column with no values at all gets 0.0; it is then constant and carries nothing.

    Used only for v4's diagnostics. Training (run_track) imputes with TRAINING-fold
    medians inside each split instead, which is the leak-free choice.
    """
    out = frame.copy()
    for c in feature_cols:
        if c not in out.columns:
            continue
        col = pd.to_numeric(out[c], errors='coerce')
        if c == 'peak_abs_mag':
            out[c] = col.fillna(0.0)
            continue
        med = col.median()
        out[c] = col.fillna(med if np.isfinite(med) else 0.0)
    return out

In [ ]:
TRACK_FEATURES = [c for c in FEATURE_COLS if c in feature_df.columns]
tns_frame = tns_training_frame(feature_df)
tns_track = run_track(tns_frame, TRACK_FEATURES, 'coarse_label', 'label', 'TNS')

alerce_frame = alerce_track_frame(feature_df)
alerce_track = run_track(alerce_frame, TRACK_FEATURES, 'coarse_label', 'label', 'ALeRCE')

In [ ]:
summary_table = track_summary([tns_track, alerce_track])
summary_table.insert(1, 'Pool', summary_table['Track'].map(
    {'TNS': 'TNS-confirmed (+ TESS flares)', 'ALeRCE': 'ALeRCE-confident'}))
display(summary_table.round(3))
print('Scored against TNS truth on the TNS track and against ALeRCE labels on the ALeRCE '
      'track. These two columns are NOT comparable as bare numbers — different samples, '
      'label reliability and Stage 2 class definitions. Step 10 is the comparison.')

### Did augmentation help?

The same TNS track with and without the synthetic copies. The real test partitions are
identical in both runs — asserted, not assumed — so this is a paired comparison.

In [ ]:
tns_track_noaug = run_track(tns_frame[is_real(tns_frame)], TRACK_FEATURES,
                            'coarse_label', 'label', 'TNS (no augmentation)', verbose=False)
aug_ablation = None
if tns_track['stage2'] and tns_track_noaug['stage2']:
    a, b = tns_track['stage2']['split'], tns_track_noaug['stage2']['split']
    same = (set(a['df'].iloc[a['idx_test']]['id']) == set(b['df'].iloc[b['idx_test']]['id']))
    assert same, 'augmentation changed the test set — the comparison would be meaningless'
    aug_ablation = (tns_track['stage2']['results'].set_index('Model')[['Test accuracy', 'Macro-F1']]
                    .join(tns_track_noaug['stage2']['results'].set_index('Model')
                          [['Test accuracy', 'Macro-F1']], rsuffix=' (no aug)'))
    aug_ablation['delta acc'] = aug_ablation['Test accuracy'] - aug_ablation['Test accuracy (no aug)']
    display(aug_ablation.round(3))
    print(f"Stage 2 test set: {len(a['idx_test'])} objects. A difference smaller than the "
          'bootstrap CI width is not evidence either way.')

# STEP 10 — The comparison  *(brief Phase 5, Section 6)*

After de Soto et al. (2024): $A_{\rm expected} = P_{\rm ours}^{\rm T}\,C_{\rm ALeRCE}$ —
our purity matrix times ALeRCE's completeness matrix — is what agreement would be if the
two pipelines' errors were independent. Actual agreement **above** it means shared signal
(both key off the same light-curve physics), not independent verification.

* **Our predictions are out-of-fold.** Every object is predicted by a model that never saw
  it; objects with no TNS label are predicted by a model trained on all TNS objects. The
  held-out test split alone is too small at Stage 2 for a per-class matrix.
* **P_ours** comes from the full TNS validation set, as de Soto et al. build theirs.
  **C_ALeRCE** can only come from objects with both a TNS truth and an ALeRCE label.
* **How Section 6.2's split is read.** It asks for the overlap pool split into
  TNS-confirmed and non-confirmed subsets — but Section 3.3 defines the overlap pool as
  objects that *are* TNS-confirmed, so the second subset would be empty. de Soto et al.
  compare every object **both** pipelines classify; here that is the ALeRCE-confident pool,
  split into its **TNS-confirmed part (which is exactly the overlap pool)** and its
  **photometric-only part**.

In [ ]:
# Section 6 — agreement framework
# ---------------------------------------------------------------
# Inlined verbatim from btp5/agreement.py, which the repository's test suite
# (tests5/) exercises. Edit it there and rebuild this notebook with
# tools/build_notebook_v5.py rather than editing it here.
# ---------------------------------------------------------------

"""
Section 6 — the comparison methodology, after de Soto et al. 2024 (Superphot+,
arXiv:2403.07975).

The naive thing — putting the TNS-track accuracy next to the ALeRCE-track accuracy —
is confounded by three things at once: different sample sizes, different label
reliability, and different class definitions at Stage 2. So instead:

  A_expected = P_ours^T · C_ALeRCE

where P_ours is our model's purity (precision) matrix and C_ALeRCE is ALeRCE's
completeness (recall) matrix. This is what agreement would look like if our errors and
ALeRCE's errors were statistically independent. Comparing the ACTUAL agreement matrix
against it is the informative move: actual > expected means the two pipelines share
underlying signal (both keying off similar physics), not that either has been
independently verified.
"""

import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix


def purity_matrix(y_true, y_pred, classes):
    """
    Column-normalised confusion matrix: P[i, j] = P(true = i | predicted = j).
    'Of everything we called j, what fraction really was i' — precision, per class.
    """
    cm = confusion_matrix(y_true, y_pred, labels=classes).astype(float)
    col = cm.sum(axis=0, keepdims=True)
    with np.errstate(invalid='ignore', divide='ignore'):
        P = np.divide(cm, col, out=np.zeros_like(cm), where=col > 0)
    return pd.DataFrame(P, index=classes, columns=classes)


def completeness_matrix(y_true, y_pred, classes):
    """
    Row-normalised confusion matrix: C[i, j] = P(predicted = j | true = i).
    'Of everything that really was i, what fraction did we call j' — recall, per class.
    """
    cm = confusion_matrix(y_true, y_pred, labels=classes).astype(float)
    row = cm.sum(axis=1, keepdims=True)
    with np.errstate(invalid='ignore', divide='ignore'):
        C = np.divide(cm, row, out=np.zeros_like(cm), where=row > 0)
    return pd.DataFrame(C, index=classes, columns=classes)


def expected_agreement(purity_ours, completeness_alerce):
    """
    Section 6.1: A_expected = P_ours^T · C_ALeRCE.

    Both matrices must be over the same class list, in the same order — mismatched
    ordering silently produces a plausible-looking but meaningless matrix, so it is
    checked rather than assumed.
    """
    assert list(purity_ours.index) == list(completeness_alerce.index), \
        'purity and completeness matrices are over different classes/orders'
    A = purity_ours.values.T @ completeness_alerce.values
    return pd.DataFrame(A, index=purity_ours.index, columns=completeness_alerce.columns)


def actual_agreement(pred_ours, pred_alerce, classes, normalize='all'):
    """
    Section 6.2: how often our prediction and ALeRCE's prediction actually coincide,
    over the overlap pool. `normalize='all'` gives fractions of the whole pool;
    'index' gives, per our-predicted class, how ALeRCE distributed them.
    """
    cm = confusion_matrix(pred_ours, pred_alerce, labels=classes).astype(float)
    if normalize == 'all' and cm.sum() > 0:
        cm = cm / cm.sum()
    elif normalize == 'index':
        row = cm.sum(axis=1, keepdims=True)
        with np.errstate(invalid='ignore', divide='ignore'):
            cm = np.divide(cm, row, out=np.zeros_like(cm), where=row > 0)
    return pd.DataFrame(cm, index=classes, columns=classes)


def agreement_rate(pred_ours, pred_alerce):
    """Scalar: fraction of overlap objects where the two pipelines agree exactly."""
    a = np.asarray(pred_ours)
    b = np.asarray(pred_alerce)
    return float((a == b).mean()) if len(a) else float('nan')


def compare_expected_actual(pred_ours, pred_alerce, purity_ours, completeness_alerce,
                            classes):
    """
    Section 6.3: the headline comparison for one model, on one pool.

    Returns a dict with both matrices, their diagonals (per-class agreement), and the
    scalar actual vs expected rates. The interpretation of the sign of
    (actual - expected) is written out by `interpret_agreement`.
    """
    A_exp = expected_agreement(purity_ours, completeness_alerce)
    A_act = actual_agreement(pred_ours, pred_alerce, classes, normalize='all')

    expected_rate = float(np.trace(A_exp.values) / max(len(classes), 1))
    # A_exp is a product of two stochastic matrices, so its diagonal is a per-class
    # expected agreement, not a population fraction. Weight it by how often each class
    # actually occurs so it is comparable to the observed rate.
    counts = pd.Series(pred_ours).value_counts(normalize=True)
    w = np.array([counts.get(c, 0.0) for c in classes])
    expected_weighted = float((np.diag(A_exp.values) * w).sum())

    return {
        'classes': list(classes),
        'expected_matrix': A_exp,
        'actual_matrix': A_act,
        'expected_rate_unweighted': expected_rate,
        'expected_rate_weighted': expected_weighted,
        'actual_rate': agreement_rate(pred_ours, pred_alerce),
        'per_class_expected': pd.Series(np.diag(A_exp.values), index=classes),
        'per_class_actual': pd.Series(np.diag(A_act.values), index=classes),
        'n': int(len(pred_ours)),
    }


def interpret_agreement(result, model_name, pool_name, external_note=True):
    """
    Section 6.3 demands a written interpretation, not just numbers. This produces it
    from the computed values, so the prose cannot drift from the table.
    """
    act, exp = result['actual_rate'], result['expected_rate_weighted']
    n = result['n']
    L = []
    L.append(f"**{model_name} on the {pool_name} pool (n={n}).** "
             f"Our pipeline and ALeRCE agree on {100 * act:.1f}% of objects; "
             f"independent-error agreement would be {100 * exp:.1f}%.")
    if not np.isfinite(act) or not np.isfinite(exp):
        L.append("Too few objects in this pool to compute a meaningful rate.")
        return ' '.join(L)

    if act > exp + 0.02:
        L.append(
            f"Actual agreement exceeds expected by {100 * (act - exp):.1f} points. "
            "That is *not* independent verification: if two pipelines erred "
            "independently, agreement would sit at the expected value. Exceeding it "
            "means both are keying off substantially the same underlying signal — "
            "unsurprising, since both are trained on ZTF light-curve shape and colour. "
            "Agreement here measures shared method, not shared correctness.")
    elif act < exp - 0.02:
        L.append(
            f"Actual agreement falls {100 * (exp - act):.1f} points BELOW the "
            "independent-error baseline. The two pipelines are disagreeing more than "
            "chance would predict, which points at a systematic difference in class "
            "definition or in the feature basis rather than at random error.")
    else:
        L.append(
            "Actual and expected agreement are within ~2 points, which is what "
            "genuinely independent errors would look like.")

    pc_a, pc_e = result['per_class_actual'], result['per_class_expected']
    gap = (pc_a - pc_e).sort_values()
    if len(gap):
        worst = gap.index[0]
        L.append(f"Weakest class: **{worst}** "
                 f"(actual {pc_a[worst]:.2f} vs expected {pc_e[worst]:.2f}).")
        if external_note and worst in ('TDE', 'SLSN'):
            L.append(
                "TDE/SLSN disagreement is a documented, cross-pipeline pattern rather "
                "than evidence that either side is broken — see the external "
                "corroboration table (Section 6.4): Superphot+ Table 3 found 54.9% of "
                "true TDEs called SLSN-I when forced through a 5-way SN classifier, and "
                "Townsend et al. 2026 confirmed the same confusion direction "
                "independently on real ZTF data.")
    return ' '.join(L)


# --------------------------------------------------------------------------
# Section 6.4 — external corroboration, quoted with sources
# --------------------------------------------------------------------------

EXTERNAL_RESULTS = pd.DataFrame([
    {'source': 'Superphot+ (de Soto et al. 2024, arXiv:2403.07975)',
     'task': '5-way SN, no redshift', 'result': '83% acc / 0.61 macro-F1', 'N': '6061'},
    {'source': 'SuperRAENN (Villar et al. 2020)',
     'task': '5-way SN', 'result': '87% acc / 0.66 purity', 'N': '~2885'},
    {'source': 'YSE-DR1 (Aleo et al. 2023)',
     'task': '3-way SN (Ia/II/Ibc)', 'result': '82% acc, real spectroscopic sample',
     'N': '472'},
    {'source': 'Superphot+ vs ALeRCE agreement (de Soto et al. 2024)',
     'task': '4-way SN',
     'result': '82% actual vs 69% expected (spec. subset); 72% actual (photometric-only)',
     'N': '—'},
    {'source': 'Superphot+ Table 3 (true TDEs through a 5-way SN classifier)',
     'task': 'TDE mis-assignment',
     'result': '54.9% -> SLSN-I, 21.6% -> SN IIn, 13.7% -> SN Ia', 'N': '—'},
    {'source': 'Townsend et al. 2026 (NoiZTF, arXiv:2602.13036)',
     'task': 'TDE vs SLSN/IIn',
     'result': 'same confusion direction, independently confirmed on real ZTF data',
     'N': '—'},
])

TDE_SLSN_STATEMENT = (
    "TDE<->SLSN/IIn confusion has now been documented across three independent "
    "pipelines — ALeRCE pre-2025, Superphot+ (de Soto et al. 2024) and NoiZTF "
    "(Townsend et al. 2026) — on top of whatever this notebook finds. Any such "
    "confusion in our own results should be read as expected, physically motivated "
    "and consistent with the literature, not as a defect specific to this pipeline.")

TNS_LABEL_NOISE_CAVEAT = (
    "TNS labels are the stronger of the two ground-truth sources, but they are not an "
    "oracle. Aleo et al. (2023) independently re-ran spectral classification (SNID) on "
    "public TNS spectra and found objects where their reclassification disagreed with "
    "the TNS-listed type, and some TNS spectra listed as 'classified' were on "
    "reanalysis noise-dominated with no safe classification available. Results here are "
    "therefore described as agreement with TNS's *recorded* classification, not as "
    "agreement with ground truth.")


# --------------------------------------------------------------------------
# The full Section 6 report, both subsets, one stage at a time
# --------------------------------------------------------------------------

SUBSET_SPEC = 'TNS-confirmed (= overlap pool)'
SUBSET_PHOT = 'photometric-only (ALeRCE-confident, no TNS label)'


def agreement_report(frame, our_col, alerce_col, truth_col, classes, model_name, stage,
                     tns_col='has_tns_label', min_n=5, purity_ours=None):
    """
    Expected vs actual agreement for one model at one stage.

    How Section 6.2's split is read, stated because the brief's two definitions
    collide: 6.2 asks for the overlap pool split into TNS-confirmed and non-confirmed
    subsets, but Section 3.3 defines the overlap pool as objects that ARE
    TNS-confirmed, so that second subset is empty by construction. de Soto et al.'s
    actual design compares every object BOTH pipelines classify. Here that is the
    ALeRCE-confident pool, split into its TNS-confirmed part — which is exactly the
    overlap pool — and its photometric-only part.

    P_ours and C_ALeRCE come from the TNS-confirmed objects (the only ones with a truth
    to score against), and are then used as the independent-error baseline for BOTH
    subsets. That is also de Soto et al.'s construction.
    """
    f = frame[frame[our_col].isin(classes) & frame[alerce_col].isin(classes)].copy()
    truthed = f[f[truth_col].isin(classes)]
    out = {'model': model_name, 'stage': stage, 'classes': list(classes), 'subsets': {}}
    if len(truthed) < min_n:
        out['note'] = (f'only {len(truthed)} objects with a truth label in these classes — '
                       'purity/completeness matrices cannot be estimated')
        return out
    # P_ours: pass the purity matrix from the FULL validation set when available (de
    # Soto et al. build it from their whole validation sample, not from the handful
    # of objects that happen to sit in the comparison pool). C_ALeRCE can only come
    # from objects that have both a TNS truth and an ALeRCE label, i.e. the overlap.
    P = (purity_ours.reindex(index=classes, columns=classes).fillna(0.0)
         if purity_ours is not None
         else purity_matrix(truthed[truth_col], truthed[our_col], classes))
    C = completeness_matrix(truthed[truth_col], truthed[alerce_col], classes)
    out['purity_ours'], out['completeness_alerce'] = P, C
    out['expected_matrix'] = expected_agreement(P, C)

    for name, mask in [(SUBSET_SPEC, f[tns_col].astype(bool)),
                       (SUBSET_PHOT, ~f[tns_col].astype(bool))]:
        sub = f[mask]
        if len(sub) < min_n:
            out['subsets'][name] = {'n': int(len(sub)), 'note': 'too few objects'}
            continue
        res = compare_expected_actual(sub[our_col].values, sub[alerce_col].values,
                                      P, C, classes)
        res['text'] = interpret_agreement(res, f'{model_name} ({stage})', name)
        out['subsets'][name] = res
    return out


def contrast_subsets(report):
    """
    The de Soto comparison: agreement on the spectroscopic vs photometric subset.
    They found 82% vs 72%. Reported either way, not assumed.
    """
    s, p = report['subsets'].get(SUBSET_SPEC, {}), report['subsets'].get(SUBSET_PHOT, {})
    if 'actual_rate' not in s or 'actual_rate' not in p:
        return (f"{report['model']} ({report['stage']}): the spectroscopic/photometric "
                f"contrast needs both subsets populated (n={s.get('n', 0)} and "
                f"{p.get('n', 0)}).")
    a, b = s['actual_rate'], p['actual_rate']
    msg = (f"{report['model']} ({report['stage']}): agreement {100 * a:.1f}% on the "
           f"TNS-confirmed subset (n={s['n']}) vs {100 * b:.1f}% on the photometric-only "
           f"subset (n={p['n']}). ")
    if b < a - 0.02:
        msg += ("Lower on the photometric-only subset, the same direction as de Soto et al. "
                "(82% vs 72%). Expected: those objects are fainter or less well sampled on "
                "average (nobody took a spectrum), and here they also lack a redshift, so "
                "peak absolute magnitude is imputed for every one of them.")
    elif b > a + 0.02:
        msg += ("HIGHER on the photometric-only subset — the opposite of de Soto et al.'s "
                "finding. Worth checking class composition: the photometric-only pool is "
                "drawn per ALeRCE class and may be richer in the easy classes.")
    else:
        msg += 'The two subsets agree to within ~2 points, unlike de Soto et al.'
    return msg


def tde_crosscheck(frame, tns_label_col='label', top1_col='alerce_top1_class',
                   of_interest=('TDE', 'SLSN'), min_n=3):
    """
    Section 3.1: what does ALeRCE's own classifier make of objects TNS calls a TDE (and
    SLSN)? Uses ALeRCE's top-1 class for every object queried, regardless of confidence,
    so it is the direct analogue of Superphot+ Table 3 (true TDEs forced through a SN
    classifier: 54.9% -> SLSN-I, 21.6% -> SN IIn, 13.7% -> SN Ia). The difference is that
    ALeRCE's 2025 classifier HAS a TDE class, so this measures how often it uses it.
    """
    f = frame[frame[tns_label_col].isin(of_interest) & frame[top1_col].notna()]
    if len(f) < min_n:
        return None, (f'only {len(f)} TNS {"/".join(of_interest)} objects have an ALeRCE '
                      'classification — cross-check not meaningful')
    tab = pd.crosstab(f[tns_label_col], f[top1_col], normalize='index')
    n = f[tns_label_col].value_counts()
    tab.insert(0, 'n', n.reindex(tab.index))
    lines = []
    if 'TDE' in tab.index:
        row = tab.loc['TDE'].drop('n')
        hit = float(row.get('TDE', 0.0))
        rest = row.drop('TDE', errors='ignore').sort_values(ascending=False)
        lines.append(f"Of {int(tab.loc['TDE', 'n'])} TNS-classified TDEs, ALeRCE's top-1 "
                     f"class is TDE for {100 * hit:.0f}%.")
        if len(rest) and rest.iloc[0] > 0:
            lines.append(f"The most common alternative is {rest.index[0]} "
                         f"({100 * rest.iloc[0]:.0f}%).")
        if 'SLSN' in rest.index and rest['SLSN'] > 0:
            lines.append("TDE->SLSN is the confusion Superphot+ Table 3 found for 54.9% of "
                         "true TDEs; seeing it here as well is consistent with a shared, "
                         "physical ambiguity (both are slow, blue, luminous nuclear-ish "
                         "transients), not a defect of either pipeline.")
    return tab, ' '.join(lines)

In [ ]:
# Section 3.1 — what ALeRCE's 2025 classifier makes of TNS TDEs (and SLSNe).
tde_table, tde_text = tde_crosscheck(feature_df[is_real(feature_df)])
if tde_table is not None:
    display(tde_table.round(2))
print(tde_text)

In [ ]:
# ---- Stage 1 agreement, per model ------------------------------------------------
alerce_real = alerce_track_frame(feature_df, real_only=True)
tns_truth = feature_df.drop_duplicates('id').set_index('id')
tns_coarse = tns_truth['coarse_label'].where(tns_truth['has_tns_label'].astype(bool))
S1_CLASSES = [c for c in ['AGN', 'SNe', 'TDE'] if c in set(alerce_real['coarse_label'])]
print('Stage 1 comparison classes:', S1_CLASSES,
      '(stellar_flare cannot appear: no TNS-confirmed object in the comparison is a flare)')

stage1_ours, stage1_reports = {}, {}
for name in HEADLINE_MODELS:
    oof = oof_predictions(tns_frame, TRACK_FEATURES, 'coarse_label', name)
    v = oof[oof['true'].isin(S1_CLASSES) & oof['pred'].isin(S1_CLASSES)]
    P = purity_matrix(v['true'], v['pred'], S1_CLASSES)
    ours = dict(zip(oof['id'], oof['pred']))
    unseen = alerce_real[~alerce_real['id'].astype(str).isin(ours)]
    up = predict_unseen(tns_frame, TRACK_FEATURES, 'coarse_label', name, unseen)
    ours.update(zip(up['id'], up['pred']))
    stage1_ours[name] = ours
    comp = pd.DataFrame({'id': alerce_real['id'].astype(str).values,
                         'ours': alerce_real['id'].astype(str).map(ours).values,
                         'alerce': alerce_real['coarse_label'].values,
                         'truth': alerce_real['id'].map(tns_coarse).values,
                         'has_tns_label': alerce_real['has_tns_label'].astype(bool).values})
    rep = agreement_report(comp, 'ours', 'alerce', 'truth', S1_CLASSES, name, 'Stage 1',
                           purity_ours=P)
    stage1_reports[name] = rep
    print(f'\n=== {name}, Stage 1 ===')
    if 'expected_matrix' in rep:
        print('expected (independent errors):'); display(rep['expected_matrix'].round(3))
    for sub, r in rep['subsets'].items():
        if 'actual_matrix' in r:
            print(f'actual — {sub}:'); display(r['actual_matrix'].round(3))
            print(r['text'])
        else:
            print(f'{sub}: n={r.get("n", 0)} — {r.get("note", "")}')
    print(contrast_subsets(rep))

In [ ]:
# ---- Stage 2 agreement, per model ------------------------------------------------
# Population: objects ALeRCE calls SN-like AND our own Stage 1 routes to SNe — the
# cascade-realistic comparison. Our 5-way Ib/Ic are collapsed to Ibc ONLY here, to meet
# ALeRCE's vocabulary; the TNS track's own Stage 2 stays 5-way.
sn_tns = tns_frame[tns_frame['label'].isin(SN_SUBTYPES)]
tns_fine_c = tns_truth['label'].where(tns_truth['has_tns_label'].astype(bool)
                                      & tns_truth['label'].isin(SN_SUBTYPES)).map(collapse_tns_fine)
stage2_reports = {}
for name in HEADLINE_MODELS:
    oof2 = oof_predictions(sn_tns, TRACK_FEATURES, 'label', name)
    oof2['pred_c'], oof2['true_c'] = oof2['pred'].map(collapse_tns_fine), oof2['true'].map(collapse_tns_fine)
    P2 = purity_matrix(oof2['true_c'], oof2['pred_c'], ALERCE_SN_CLASSES)
    pop = alerce_real[alerce_real['label'].isin(ALERCE_SN_CLASSES)]
    pop = pop[pop['id'].astype(str).map(stage1_ours[name]) == 'SNe']
    ours2 = dict(zip(oof2['id'], oof2['pred_c']))
    unseen = pop[~pop['id'].astype(str).isin(ours2)]
    up = predict_unseen(sn_tns, TRACK_FEATURES, 'label', name, unseen)
    ours2.update(zip(up['id'], up['pred'].map(collapse_tns_fine)))
    comp2 = pd.DataFrame({'ours': pop['id'].astype(str).map(ours2).values,
                          'alerce': pop['label'].values,
                          'truth': pop['id'].map(tns_fine_c).values,
                          'has_tns_label': pop['has_tns_label'].astype(bool).values})
    rep2 = agreement_report(comp2, 'ours', 'alerce', 'truth', ALERCE_SN_CLASSES, name,
                            'Stage 2', purity_ours=P2)
    stage2_reports[name] = rep2
    print(f'\n=== {name}, Stage 2 (n={len(comp2)}) ===')
    if 'note' in rep2:
        print(rep2['note']); continue
    for sub, r in rep2['subsets'].items():
        print(r['text'] if 'text' in r else f'{sub}: n={r.get("n", 0)} — {r.get("note", "")}')
    print(contrast_subsets(rep2))

In [ ]:
print('Section 6.4 — external corroboration:')
display(EXTERNAL_RESULTS)
print(TDE_SLSN_STATEMENT)
print()
print(TNS_LABEL_NOISE_CAVEAT)

# STEP 11 — v4 diagnostics, on both tracks  *(brief Phase 6)*

v4's diagnostics, preserved. Two things differ from v4, both about what they are pointed at:

* **Real objects only.** They draw their own random splits; given synthetic copies, a copy
  could train while its parent is tested, inflating every number.
* **Imputed the way v4 imputed.** v4 filled every feature gap before its diagnostics ran
  (class-independent medians; a 0.0 sentinel for peak absolute magnitude). v5 adds
  columns after that point — host offset, redshift-less native objects — which can be
  entirely empty, and v4's HistGradientBoosting fits crash on an all-empty column. The
  diagnostic view is imputed with v4's own rules; training is not (it imputes inside
  each split).
* **Both tracks.** The TNS pass runs as ordinary cells below. The ALeRCE pass re-executes
  the **same source** (a repository test asserts it is identical) with the track rebound.
  Cells [35, 37] — the class-scheme comparison and the class-imbalance test — run
  on the TNS track only: they are written for the 5-way vocabulary, and ALeRCE's SN scheme
  is already the class-scheme comparison's own "4-way Ia / Ib-c / II / SLSN" row.

One deliberate change to a preserved cell: the ablation (below) gains a guard for a
feature group that is entirely missing, which otherwise crashed it when no BTS-sourced
object survived acquisition.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# TNS pass: point v4's names at the TNS track, real objects only.
_feature_df_all = feature_df
X_COLS = list(tns_track['stage1']['split']['feature_cols'])
# v4's diagnostics were written for v4's fully imputed table; give them one.
feature_df_full = v4_impute(tns_diagnostic_frame(feature_df), X_COLS)
feature_df      = feature_df_full                  # v4's separability cell reads this name
split, stage1, stage2a = v4_adapters(tns_track)
print(f'TNS diagnostics on {len(feature_df_full)} real objects, {len(X_COLS)} features')

In [ ]:
# ---- 1. ablation: which feature GROUP is carrying the result --------------------
def evaluate_feature_set(cols, classes=None, n_repeats=15, seed=0, balanced_n=None):
    d = feature_df_full if classes is None else feature_df_full[feature_df_full['label'].isin(classes)]
    # v5 guard: a feature group that is entirely missing (e.g. the BTS catalogue
    # columns when no BTS-sourced object survived acquisition) cannot be binned,
    # and previously crashed the whole cell. Report it as unavailable instead.
    usable = [c for c in cols if c in d.columns and d[c].notna().any()
              and d[c].nunique(dropna=True) > 1]
    if len(usable) == 0:
        print('  (skipped: no usable columns in this group)')
        return dict(acc=np.nan, acc_sd=np.nan, bacc=np.nan, f1=np.nan)
    cols = usable
    accs, f1s, baccs = [], [], []
    for rep in range(n_repeats):
        rs = seed + rep * 97
        dd = d
        if balanced_n:
            dd = pd.concat([g.sample(min(balanced_n, len(g)), random_state=rs)
                            for _, g in d.groupby('label')])
        X, y = dd[cols].values, dd['label'].values
        Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=rs)
        sc = StandardScaler(); Xtr = sc.fit_transform(Xtr); Xte = sc.transform(Xte)
        m = HistGradientBoostingClassifier(max_iter=400, learning_rate=0.06, random_state=rs)
        m.fit(Xtr, ytr); p = m.predict(Xte)
        accs.append(accuracy_score(yte, p)); baccs.append(balanced_accuracy_score(yte, p))
        f1s.append(f1_score(yte, p, average='macro'))
    return dict(acc=np.mean(accs), acc_sd=np.std(accs), bacc=np.mean(baccs), f1=np.mean(f1s))

SHAPE_ONLY = [c for c in LIGHTCURVE_COLS if c in feature_df_full.columns]
PHYS       = [c for c in PHYSICAL_COLS if c in feature_df_full.columns]
CAT        = [c for c in CATALOGUE_COLS if c in feature_df_full.columns]

print('Stage 2, 5 SN subtypes - what each feature GROUP is worth\n')
rows = []
for tag, cols in [('light-curve shape only',              SHAPE_ONLY),
                  ('+ redshift and fitted absolute mag',  SHAPE_ONLY + PHYS),
                  ('+ BTS catalogue columns (v4)',        SHAPE_ONLY + PHYS + CAT),
                  ('catalogue columns ALONE',             CAT),
                  ('redshift ALONE',                      ['redshift'])]:
    cols = [c for c in cols if c in feature_df_full.columns]
    if not cols:
        continue
    r = evaluate_feature_set(cols, classes=SN_SUBTYPES)
    rows.append({'feature set': tag, 'n features': len(cols), **r})
    print(f'{tag:<38s} {len(cols):3d} feats  acc {r["acc"]:.3f}+/-{r["acc_sd"]:.3f}  '
          f'bal-acc {r["bacc"]:.3f}  macroF1 {r["f1"]:.3f}')
ablation = pd.DataFrame(rows)

print("""
If 'redshift ALONE' lands close to the full model, the classifier is largely reading
the survey selection function: a magnitude-limited survey finds SLSNe at z~0.15 and
Type II at z~0.035, so redshift is nearly a class label. On the benchmark that single
column reached 0.395 where all 22 features reached 0.469. The next cell measures how
much of your result survives when that crutch is removed.
""")

In [ ]:
# ---- 2. how much of Stage 2 is redshift, and how much is astrophysics? ----------
#
# Two controls:
#   (a) drop redshift and refit. A small drop means the light curves carry the signal.
#   (b) resample so every class has the SAME redshift distribution, then refit. This
#       removes the selection effect entirely. If accuracy collapses here, the model
#       was detecting the survey, not the transient.
#
# This is the single most important number to put in the report. A classifier that
# scores well only because SLSNe are far away will not transfer to any other survey.

d5 = feature_df_full[feature_df_full['label'].isin(SN_SUBTYPES)].copy()
NOZ = [c for c in X_COLS if c not in ('redshift',)]
NOPHYS = [c for c in X_COLS if c not in ('redshift', 'peak_abs_mag', 'bts_peakabs')]

n_bal = int(d5['label'].value_counts().min())
full = evaluate_feature_set(X_COLS, classes=SN_SUBTYPES, balanced_n=n_bal)
noz  = evaluate_feature_set(NOZ,    classes=SN_SUBTYPES, balanced_n=n_bal)
nop  = evaluate_feature_set(NOPHYS, classes=SN_SUBTYPES, balanced_n=n_bal)
print(f'all features          acc {full["acc"]:.3f}  macroF1 {full["f1"]:.3f}')
print(f'without redshift      acc {noz["acc"]:.3f}  macroF1 {noz["f1"]:.3f}')
print(f'without any luminosity/redshift information  acc {nop["acc"]:.3f}  macroF1 {nop["f1"]:.3f}')

# (b) redshift-matched control
print('\nredshift distribution per class:')
print(d5.groupby('label')['redshift'].describe()[['count', '25%', '50%', '75%']].round(4).to_string())

# Bin only over the redshift range where ALL classes actually have objects. Binning
# over the global range puts SLSN alone in the top bins and leaves nothing to match.
lo = max(g['redshift'].quantile(0.05) for _, g in d5.groupby('label'))
hi = min(g['redshift'].quantile(0.95) for _, g in d5.groupby('label'))
if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
    lo, hi = d5['redshift'].quantile(0.05), d5['redshift'].quantile(0.95)
    print(f'NOTE: class redshift ranges barely overlap; falling back to the global range.')
print(f'\ncommon redshift range across all five subtypes: {lo:.4f} - {hi:.4f}')
d5 = d5[(d5['redshift'] >= lo) & (d5['redshift'] <= hi)].copy()
edges = np.unique(np.quantile(d5['redshift'].dropna(), np.linspace(0, 1, 5))) if len(d5) else np.array([])
if len(edges) >= 3:
    d5['zbin'] = pd.cut(d5['redshift'], edges, labels=False, include_lowest=True)
    per_bin = d5.groupby(['label', 'zbin']).size().unstack(fill_value=0)
    print('\nobjects per class per redshift bin:'); print(per_bin.to_string())
    quota = per_bin.min(axis=0)
    matched = [g[g['zbin'] == zb].sample(min(int(q), int((g['zbin'] == zb).sum())), random_state=0)
               for _, g in d5.groupby('label') for zb, q in quota.items() if q > 0]
    matched = pd.concat(matched) if matched else pd.DataFrame()
    nm = int(matched['label'].value_counts().min()) if len(matched) else 0
    print(f'\nredshift-matched sample: {nm} per class')
    if nm >= 10:
        accs, f1s = [], []
        for rep in range(15):
            rs = rep * 97
            dd = pd.concat([g.sample(nm, random_state=rs) for _, g in matched.groupby('label')])
            Xtr, Xte, ytr, yte = train_test_split(dd[X_COLS].values, dd['label'].values,
                                                  test_size=0.25, stratify=dd['label'].values,
                                                  random_state=rs)
            sc = StandardScaler(); Xtr = sc.fit_transform(Xtr); Xte = sc.transform(Xte)
            m = HistGradientBoostingClassifier(max_iter=400, learning_rate=0.06, random_state=rs)
            m.fit(Xtr, ytr); p = m.predict(Xte)
            accs.append(accuracy_score(yte, p)); f1s.append(f1_score(yte, p, average='macro'))
        unm = evaluate_feature_set(X_COLS, classes=SN_SUBTYPES, balanced_n=nm)
        print(f'unmatched, {nm}/class : acc {unm["acc"]:.3f}  macroF1 {unm["f1"]:.3f}')
        print(f'z-MATCHED, {nm}/class : acc {np.mean(accs):.3f}  macroF1 {np.mean(f1s):.3f}')
        print(f'\nselection-driven component: {unm["acc"] - np.mean(accs):+.3f} accuracy')
    else:
        print('too few objects for a matched control at this sample size.')
        print('Either raise N_ACQUIRE_SN_SUBTYPE, or read this as the result itself:')
        print('if the classes barely share a redshift range, then redshift IS close to')
        print('a class label in this sample, and the headline accuracy is partly a')
        print('measurement of how ZTF selects targets rather than of the transients.')

In [ ]:
# ---- 3. the pairwise ceiling: which class pairs are actually separable? ---------
#
# For every pair of subtypes and every feature, the effect size |Cohen's d| between the
# two class distributions. This is the diagnostic that explains the confusion matrix.
#
# Rule of thumb for an ideal single-feature separator:
#   |d| = 2.0 -> ~84% pairwise accuracy;  1.0 -> ~69%;  0.4 -> ~58%;  0.15 -> ~53%.
import itertools

def cohens_d(a, b):
    a = np.asarray(a, float); b = np.asarray(b, float)
    a = a[np.isfinite(a)]; b = b[np.isfinite(b)]
    if len(a) < 3 or len(b) < 3:
        return np.nan
    sa, sb = a.std(ddof=1), b.std(ddof=1)
    pooled = np.sqrt((sa ** 2 + sb ** 2) / 2)
    return abs(a.mean() - b.mean()) / pooled if pooled > 0 else np.nan

d5 = feature_df_full[feature_df_full['label'].isin(SN_SUBTYPES)]
rows = []
for A, B in itertools.combinations(SN_SUBTYPES, 2):
    ds = {c: cohens_d(d5[d5['label'] == A][c], d5[d5['label'] == B][c]) for c in X_COLS}
    finite = {k: v for k, v in ds.items() if np.isfinite(v)}
    if not finite:
        continue
    bf = max(finite, key=finite.get)
    rows.append({'pair': f'{A} vs {B}', 'best feature': bf, 'max |d|': finite[bf],
                 'median |d| over all features': float(np.nanmedian(list(finite.values())))})
pairs = pd.DataFrame(rows).sort_values('max |d|', ascending=False)
print(pairs.round(2).to_string(index=False))

worst = pairs.iloc[-1]
print(f"\nHardest pair on your data: {worst['pair']} - best single feature "
      f"'{worst['best feature']}' at |d| = {worst['max |d|']:.2f}")
print("""
On the full ZTF BTS catalogue the answer is SN_Ib vs SN_Ic at |d| = 0.39, with every
other catalogue axis below 0.15 - median z 0.0355 vs 0.0346, peak absolute magnitude
-17.50 vs -17.79, rise 11.5 vs 11.0 d, fade 11.5 vs 12.1 d, duration 24.5 vs 23.1 d.
Two classes that overlap that completely cap the 5-way scheme in the low-to-mid 70s
no matter what model is used, because what separates them - a helium absorption line -
is spectroscopic and is not present in g and r photometry.

That is why PLAsTiCC ships a single SNIbc class and Superphot+ uses SN Ib/c. The
scheme comparison below quantifies exactly what merging them buys on YOUR data.
""")

fig, ax = plt.subplots(figsize=(7, 4))
p = pairs.iloc[::-1]
ax.barh(p['pair'], p['max |d|'], color=['#E15759' if v < 0.5 else '#4C78A8' for v in p['max |d|']])
for x, lbl in [(0.4, '0.4 ~ 58%'), (1.0, '1.0 ~ 69%'), (2.0, '2.0 ~ 84%')]:
    ax.axvline(x, ls='--', c='grey', lw=0.8)
    ax.text(x, -0.7, lbl, fontsize=7, ha='center', color='grey')
ax.set_xlabel("best single-feature |Cohen's d|")
ax.set_title('Which subtype pairs are separable at all')
plt.tight_layout(); plt.show()

In [ ]:
# ---- 2. which class scheme reaches the target ----------------------------------
SCHEMES = {
 '5-way  Ia / Ib / Ic / II / SLSN': {'SN_Ia':'SN_Ia','SN_Ib':'SN_Ib','SN_Ic':'SN_Ic',
                                     'SN_II':'SN_II','SLSN':'SLSN'},
 '4-way  Ia / Ib-c / II / SLSN':    {'SN_Ia':'SN_Ia','SN_Ib':'SN_Ibc','SN_Ic':'SN_Ibc',
                                     'SN_II':'SN_II','SLSN':'SLSN'},
 '3-way  Ia / CCSN / SLSN':         {'SN_Ia':'SN_Ia','SN_Ib':'CCSN','SN_Ic':'CCSN',
                                     'SN_II':'CCSN','SLSN':'SLSN'},
 '2-way  Ia / non-Ia':              {'SN_Ia':'SN_Ia','SN_Ib':'non_Ia','SN_Ic':'non_Ia',
                                     'SN_II':'non_Ia','SLSN':'non_Ia'},
}

def evaluate_scheme(mapping, n_repeats=15, seed=0):
    d = feature_df_full[feature_df_full['label'].isin(mapping)].copy()
    d['y'] = d['label'].map(mapping)
    accs, f1s, baccs = [], [], []
    for rep in range(n_repeats):
        rs = seed + rep * 97
        Xtr, Xte, ytr, yte = train_test_split(d[X_COLS].values, d['y'].values,
                                              test_size=0.25, stratify=d['y'].values,
                                              random_state=rs)
        sc = StandardScaler(); Xtr = sc.fit_transform(Xtr); Xte = sc.transform(Xte)
        m = HistGradientBoostingClassifier(max_iter=400, learning_rate=0.06, random_state=rs)
        m.fit(Xtr, ytr); p = m.predict(Xte)
        accs.append(accuracy_score(yte, p)); baccs.append(balanced_accuracy_score(yte, p))
        f1s.append(f1_score(yte, p, average='macro'))
    return np.mean(accs), np.std(accs), np.mean(baccs), np.mean(f1s)

rows = []
for name, mp in SCHEMES.items():
    a, sd, ba, f1 = evaluate_scheme(mp)
    rows.append({'scheme': name, 'n classes': len(set(mp.values())),
                 'accuracy': a, 'acc sd': sd, 'balanced acc': ba, 'macro F1': f1})
scheme_table = pd.DataFrame(rows)
print(scheme_table.round(3).to_string(index=False))
print('\nEvery row uses the SAME objects, the SAME features and the SAME model.')
print('The only thing that changes is how many distinctions the label set asks for.')

fig, ax = plt.subplots(figsize=(7, 3.6))
ax.barh(scheme_table['scheme'], scheme_table['accuracy'],
        xerr=scheme_table['acc sd'], color='#4C78A8')
ax.axvspan(0.85, 0.90, color='#59A14F', alpha=0.18, label='85-90% target')
ax.set_xlabel('accuracy'); ax.set_xlim(0, 1); ax.legend(); ax.invert_yaxis()
ax.set_title('Accuracy vs how fine the subtype question is')
plt.tight_layout(); plt.show()

In [ ]:
# ---- 3. learning curve on your own data ---------------------------------------
def learning_curve_balanced(classes, sizes, n_repeats=10, seed=0):
    d = feature_df_full[feature_df_full['label'].isin(classes)]
    avail = d['label'].value_counts().min()
    out = []
    for n in sizes:
        if n > avail:
            break
        accs, f1s = [], []
        for rep in range(n_repeats):
            rs = seed + rep * 97
            dd = pd.concat([d[d['label'] == c].sample(n, random_state=rs) for c in classes])
            Xtr, Xte, ytr, yte = train_test_split(dd[X_COLS].values, dd['label'].values,
                                                  test_size=0.25, stratify=dd['label'].values,
                                                  random_state=rs)
            sc = StandardScaler(); Xtr = sc.fit_transform(Xtr); Xte = sc.transform(Xte)
            m = HistGradientBoostingClassifier(max_iter=400, learning_rate=0.06, random_state=rs)
            m.fit(Xtr, ytr); p = m.predict(Xte)
            accs.append(accuracy_score(yte, p)); f1s.append(f1_score(yte, p, average='macro'))
        out.append({'n_per_class': n, 'accuracy': np.mean(accs), 'sd': np.std(accs),
                    'macro F1': np.mean(f1s)})
    return pd.DataFrame(out)

# 15 is the smallest size where a 75/25 split still leaves >=3 training objects
# per class; below that the curve reports fold noise, not learning.
sizes = [15, 20, 25, 30, 50, 80, 120, 200, 300]
lc5 = learning_curve_balanced(SN_SUBTYPES, sizes)
print('5-way learning curve on this dataset:'); print(lc5.round(3).to_string(index=False))

if len(lc5) >= 3:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.errorbar(lc5['n_per_class'], lc5['accuracy'], yerr=lc5['sd'],
                marker='o', capsize=3, label='5-way accuracy')
    ax.axhspan(0.85, 0.90, color='#59A14F', alpha=0.18, label='85-90% target')
    ax.axvline(N_HEADLINE_SN_SUBTYPE, ls='--', c='grey', label=f'headline ({N_HEADLINE_SN_SUBTYPE}/class)')
    ax.set_xlabel('objects per subtype'); ax.set_ylabel('accuracy'); ax.set_ylim(0, 1)
    ax.legend(); ax.set_title('How much does more data buy?')
    plt.tight_layout(); plt.show()

print("""
Reading the curve: if it is still rising at the right-hand edge, more data is the
binding constraint and raising N_PER_SN_SUBTYPE is worth the runtime. If it has
flattened well below the target band, the ceiling is the information content of
two-band photometry, and the honest move is to report the 4-way or 3-way scheme
alongside the 5-way one rather than to keep collecting.
""")

In [ ]:
# ---- does class imbalance actually cost anything here? ------------------------
# A like-for-like test. Same features, same model, same TOTAL number of objects.
# The only thing that changes is how those objects are distributed across the five
# subtypes. Anything else confounds sample size with sample balance, which is exactly
# the mistake that made imbalance look like the culprit in v2.
d = feature_df_full[feature_df_full['label'].isin(SN_SUBTYPES)]
nmin = int(d['label'].value_counts().min())
N_TOTAL = nmin * len(SN_SUBTYPES)

MIX = {'balanced':          {c: 1 / 5 for c in SN_SUBTYPES},
       'Ia-heavy (natural)': {'SN_Ia': 0.60, 'SN_II': 0.20, 'SN_Ib': 0.07,
                              'SN_Ic': 0.08, 'SLSN': 0.05}}

def sample_mix(d, weights, n_total, random_state):
    parts = []
    for c, w in weights.items():
        want = max(int(round(n_total * w)), 5)
        pool = d[d['label'] == c]
        parts.append(pool.sample(min(want, len(pool)), random_state=random_state))
    return pd.concat(parts)

print(f'Both rows use {N_TOTAL} objects in total. Only the class mix differs.\n')
for tag, w in MIX.items():
    accs, baccs, f1s = [], [], []
    for rep in range(15):
        rs = rep * 97
        dd = sample_mix(d, w, N_TOTAL, rs)
        Xtr, Xte, ytr, yte = train_test_split(dd[X_COLS].values, dd['label'].values,
                                              test_size=0.25, stratify=dd['label'].values,
                                              random_state=rs)
        sc = StandardScaler(); Xtr = sc.fit_transform(Xtr); Xte = sc.transform(Xte)
        m = HistGradientBoostingClassifier(max_iter=400, learning_rate=0.06, random_state=rs)
        m.fit(Xtr, ytr); p = m.predict(Xte)
        accs.append(accuracy_score(yte, p)); baccs.append(balanced_accuracy_score(yte, p))
        f1s.append(f1_score(yte, p, average='macro'))
    print(f'{tag:<20s} acc {np.mean(accs):.3f}  bal-acc {np.mean(baccs):.3f}  '
          f'macro-F1 {np.mean(f1s):.3f}')

print("""
What to take from this. Under a Ia-heavy mix, plain accuracy usually goes UP while
balanced accuracy and macro-F1 go DOWN: the model buys accuracy by getting good at the
majority class and giving up on the rest. That is the trap in the 85-90% figures quoted
in the literature — Superphot+ reports 0.88 accuracy and 0.71 macro-F1 on a sample that
is 75% SN Ia, and both numbers describe the same model.

It also means the reverse of v2's assumption is true. Stage 2 was balanced at 30 per
subtype, so imbalance was costing it nothing; rebalancing something already balanced
cannot recover accuracy. SMOTE, class weights and oversampling are the right tools for
Stage 1, where AGN and TDE genuinely run short, and they are the wrong tools for
Stage 2, where the binding constraints are feature quality and sample size.
""")

In [ ]:
def cascade_predict(m1, m2, X_raw, sc1, sc2, le1, le2, sn_coarse='SNe'):
    coarse = le1.inverse_transform(m1.predict(sc1.transform(X_raw)))
    final = np.array(coarse, dtype=object)
    mask = coarse == sn_coarse
    if mask.any():
        final[mask] = le2.inverse_transform(m2.predict(sc2.transform(X_raw[mask])))
    return {'coarse_pred': coarse, 'final_pred': final, 'sne_mask': mask}


def run_stage2_option_b(split, stage1, stage2a, sn_subtype_labels, verbose=True):
    df, X = split['feature_df'], split['X']
    ite = split['idx_test']; Xte = X[ite]
    true_fine = df['label'].iloc[ite].values
    is_sn = np.isin(true_fine, sn_subtype_labels)
    rows, results = [], {}
    for name in stage1['models']:
        r = cascade_predict(stage1['models'][name], stage2a['models'][name], Xte,
                            stage1['scaler'], stage2a['scaler'], stage1['le'], stage2a['le'])
        final, coarse, mask = r['final_pred'], r['coarse_pred'], r['sne_mask']
        e2e = float((final == true_fine).mean())
        routed = is_sn & (coarse == 'SNe')
        cond = float((final[routed] == true_fine[routed]).mean()) if routed.any() else np.nan
        comp = float((final[is_sn] == true_fine[is_sn]).mean()) if is_sn.any() else np.nan
        rows.append({'Model': name, 'End-to-end (all 8 classes)': e2e,
                     'Conditional (correctly-routed SNe)': cond,
                     'Option-A-comparable (true SNe)': comp,
                     'Non-SNe mis-routed in': int((~is_sn & mask).sum()),
                     'True SNe lost before Stage 2': int((is_sn & ~mask).sum())})
        results[name] = r
        if verbose:
            print(f'[Stage 2 / B] {name:<22} end-to-end {e2e:.3f} | conditional {cond:.3f} | '
                  f'comparable {comp:.3f}')
    return results, pd.DataFrame(rows)

stage2b, stage2b_results = run_stage2_option_b(split, stage1, stage2a, SN_SUBTYPES)
stage2b_results.round(3)

In [ ]:
comparison = pd.DataFrame({
    'Model': stage2a['results']['Model'],
    'Option A (ground-truth routing)': stage2a['results']['Test accuracy'].values,
    'Option A macro-F1': stage2a['results']['Macro F1'].values,
    'Option B (comparable)': stage2b_results['Option-A-comparable (true SNe)'].values,
    'Option B (conditional)': stage2b_results['Conditional (correctly-routed SNe)'].values,
    'Option B (end-to-end, 8 classes)': stage2b_results['End-to-end (all 8 classes)'].values,
})
comparison['Routing cost (A - B comparable)'] = (
    comparison['Option A (ground-truth routing)'] - comparison['Option B (comparable)'])
print(comparison.round(3).to_string(index=False))

In [ ]:
# Permutation importance — one method that spans all five model families, computed on
# held-out data, exactly as in v2.
from sklearn.inspection import permutation_importance

def perm_importance_table(stage, x_cols, n_repeats=20, random_state=RANDOM_STATE):
    out = {}
    for name, model in stage['models'].items():
        r = permutation_importance(model, stage['X_test'], stage['y_test'],
                                   n_repeats=n_repeats, random_state=random_state,
                                   scoring='accuracy')
        out[name] = pd.Series(r.importances_mean, index=x_cols)
    t = pd.DataFrame(out)
    t.index.name = 'feature'
    return t

imp2 = perm_importance_table(stage2a, X_COLS)
summary2 = pd.DataFrame({'mean_perm_importance': imp2.mean(axis=1),
                         'across_model_std': imp2.std(axis=1)}).sort_values(
                             'mean_perm_importance', ascending=False)
print('Stage 2 — most discriminative features, averaged across the headline models:')
print(summary2.round(4).head(12).to_string())

fig, ax = plt.subplots(figsize=(8, 5))
top = summary2.head(12).iloc[::-1]
ax.barh(top.index, top['mean_perm_importance'], xerr=top['across_model_std'], color='#4C78A8')
ax.set_xlabel('permutation importance (accuracy drop)')
ax.set_title('Stage 2: what the models actually use')
plt.tight_layout(); plt.show()

In [ ]:
# Feature separability, the diagnostic that exposed the v2 feature set.
def separability(df, label_col, cols):
    rows = []
    for c in cols:
        g = df.groupby(label_col)[c]
        between = float(np.var(g.mean().values))
        within = float(g.var().mean())
        rows.append({'feature': c, 'between_class_var': between,
                     'mean_within_class_var': within,
                     'separation_ratio': between / within if within > 0 else np.nan})
    return pd.DataFrame(rows).sort_values('separation_ratio', ascending=False)

print('Stage 2 feature separability (between-class var / within-class var):')
sep2 = separability(feature_df[feature_df['label'].isin(SN_SUBTYPES)], 'label', X_COLS)
print(sep2.round(3).head(12).to_string(index=False))
print('\nFor reference, v2 Stage 2 topped out at 0.399 (colour) with the next four '
      'features at 0.059, 0.031, 0.008 and 0.008.')

In [ ]:
feature_df = _feature_df_all      # restore the full table for the steps below

In [ ]:
# ALeRCE pass: the SAME v4 diagnostic source as the cells above (asserted
# byte-identical by tests5/test_notebook_structure.py), re-executed with the
# ALeRCE track bound in place of the TNS one, and everything restored afterwards.
_V4_DIAGNOSTIC_SOURCES = [(32, '# ---- 1. ablation: which feature GROUP is carrying the result --------------------\ndef evaluate_feature_set(cols, classes=None, n_repeats=15, seed=0, balanced_n=None):\n    d = feature_df_full if classes is None else feature_df_full[feature_df_full[\'label\'].isin(classes)]\n    # v5 guard: a feature group that is entirely missing (e.g. the BTS catalogue\n    # columns when no BTS-sourced object survived acquisition) cannot be binned,\n    # and previously crashed the whole cell. Report it as unavailable instead.\n    usable = [c for c in cols if c in d.columns and d[c].notna().any()\n              and d[c].nunique(dropna=True) > 1]\n    if len(usable) == 0:\n        print(\'  (skipped: no usable columns in this group)\')\n        return dict(acc=np.nan, acc_sd=np.nan, bacc=np.nan, f1=np.nan)\n    cols = usable\n    accs, f1s, baccs = [], [], []\n    for rep in range(n_repeats):\n        rs = seed + rep * 97\n        dd = d\n        if balanced_n:\n            dd = pd.concat([g.sample(min(balanced_n, len(g)), random_state=rs)\n                            for _, g in d.groupby(\'label\')])\n        X, y = dd[cols].values, dd[\'label\'].values\n        Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=rs)\n        sc = StandardScaler(); Xtr = sc.fit_transform(Xtr); Xte = sc.transform(Xte)\n        m = HistGradientBoostingClassifier(max_iter=400, learning_rate=0.06, random_state=rs)\n        m.fit(Xtr, ytr); p = m.predict(Xte)\n        accs.append(accuracy_score(yte, p)); baccs.append(balanced_accuracy_score(yte, p))\n        f1s.append(f1_score(yte, p, average=\'macro\'))\n    return dict(acc=np.mean(accs), acc_sd=np.std(accs), bacc=np.mean(baccs), f1=np.mean(f1s))\n\nSHAPE_ONLY = [c for c in LIGHTCURVE_COLS if c in feature_df_full.columns]\nPHYS       = [c for c in PHYSICAL_COLS if c in feature_df_full.columns]\nCAT        = [c for c in CATALOGUE_COLS if c in feature_df_full.columns]\n\nprint(\'Stage 2, 5 SN subtypes - what each feature GROUP is worth\\n\')\nrows = []\nfor tag, cols in [(\'light-curve shape only\',              SHAPE_ONLY),\n                  (\'+ redshift and fitted absolute mag\',  SHAPE_ONLY + PHYS),\n                  (\'+ BTS catalogue columns (v4)\',        SHAPE_ONLY + PHYS + CAT),\n                  (\'catalogue columns ALONE\',             CAT),\n                  (\'redshift ALONE\',                      [\'redshift\'])]:\n    cols = [c for c in cols if c in feature_df_full.columns]\n    if not cols:\n        continue\n    r = evaluate_feature_set(cols, classes=SN_SUBTYPES)\n    rows.append({\'feature set\': tag, \'n features\': len(cols), **r})\n    print(f\'{tag:<38s} {len(cols):3d} feats  acc {r["acc"]:.3f}+/-{r["acc_sd"]:.3f}  \'\n          f\'bal-acc {r["bacc"]:.3f}  macroF1 {r["f1"]:.3f}\')\nablation = pd.DataFrame(rows)\n\nprint("""\nIf \'redshift ALONE\' lands close to the full model, the classifier is largely reading\nthe survey selection function: a magnitude-limited survey finds SLSNe at z~0.15 and\nType II at z~0.035, so redshift is nearly a class label. On the benchmark that single\ncolumn reached 0.395 where all 22 features reached 0.469. The next cell measures how\nmuch of your result survives when that crutch is removed.\n""")'), (33, '# ---- 2. how much of Stage 2 is redshift, and how much is astrophysics? ----------\n#\n# Two controls:\n#   (a) drop redshift and refit. A small drop means the light curves carry the signal.\n#   (b) resample so every class has the SAME redshift distribution, then refit. This\n#       removes the selection effect entirely. If accuracy collapses here, the model\n#       was detecting the survey, not the transient.\n#\n# This is the single most important number to put in the report. A classifier that\n# scores well only because SLSNe are far away will not transfer to any other survey.\n\nd5 = feature_df_full[feature_df_full[\'label\'].isin(SN_SUBTYPES)].copy()\nNOZ = [c for c in X_COLS if c not in (\'redshift\',)]\nNOPHYS = [c for c in X_COLS if c not in (\'redshift\', \'peak_abs_mag\', \'bts_peakabs\')]\n\nn_bal = int(d5[\'label\'].value_counts().min())\nfull = evaluate_feature_set(X_COLS, classes=SN_SUBTYPES, balanced_n=n_bal)\nnoz  = evaluate_feature_set(NOZ,    classes=SN_SUBTYPES, balanced_n=n_bal)\nnop  = evaluate_feature_set(NOPHYS, classes=SN_SUBTYPES, balanced_n=n_bal)\nprint(f\'all features          acc {full["acc"]:.3f}  macroF1 {full["f1"]:.3f}\')\nprint(f\'without redshift      acc {noz["acc"]:.3f}  macroF1 {noz["f1"]:.3f}\')\nprint(f\'without any luminosity/redshift information  acc {nop["acc"]:.3f}  macroF1 {nop["f1"]:.3f}\')\n\n# (b) redshift-matched control\nprint(\'\\nredshift distribution per class:\')\nprint(d5.groupby(\'label\')[\'redshift\'].describe()[[\'count\', \'25%\', \'50%\', \'75%\']].round(4).to_string())\n\n# Bin only over the redshift range where ALL classes actually have objects. Binning\n# over the global range puts SLSN alone in the top bins and leaves nothing to match.\nlo = max(g[\'redshift\'].quantile(0.05) for _, g in d5.groupby(\'label\'))\nhi = min(g[\'redshift\'].quantile(0.95) for _, g in d5.groupby(\'label\'))\nif not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:\n    lo, hi = d5[\'redshift\'].quantile(0.05), d5[\'redshift\'].quantile(0.95)\n    print(f\'NOTE: class redshift ranges barely overlap; falling back to the global range.\')\nprint(f\'\\ncommon redshift range across all five subtypes: {lo:.4f} - {hi:.4f}\')\nd5 = d5[(d5[\'redshift\'] >= lo) & (d5[\'redshift\'] <= hi)].copy()\nedges = np.unique(np.quantile(d5[\'redshift\'].dropna(), np.linspace(0, 1, 5))) if len(d5) else np.array([])\nif len(edges) >= 3:\n    d5[\'zbin\'] = pd.cut(d5[\'redshift\'], edges, labels=False, include_lowest=True)\n    per_bin = d5.groupby([\'label\', \'zbin\']).size().unstack(fill_value=0)\n    print(\'\\nobjects per class per redshift bin:\'); print(per_bin.to_string())\n    quota = per_bin.min(axis=0)\n    matched = [g[g[\'zbin\'] == zb].sample(min(int(q), int((g[\'zbin\'] == zb).sum())), random_state=0)\n               for _, g in d5.groupby(\'label\') for zb, q in quota.items() if q > 0]\n    matched = pd.concat(matched) if matched else pd.DataFrame()\n    nm = int(matched[\'label\'].value_counts().min()) if len(matched) else 0\n    print(f\'\\nredshift-matched sample: {nm} per class\')\n    if nm >= 10:\n        accs, f1s = [], []\n        for rep in range(15):\n            rs = rep * 97\n            dd = pd.concat([g.sample(nm, random_state=rs) for _, g in matched.groupby(\'label\')])\n            Xtr, Xte, ytr, yte = train_test_split(dd[X_COLS].values, dd[\'label\'].values,\n                                                  test_size=0.25, stratify=dd[\'label\'].values,\n                                                  random_state=rs)\n            sc = StandardScaler(); Xtr = sc.fit_transform(Xtr); Xte = sc.transform(Xte)\n            m = HistGradientBoostingClassifier(max_iter=400, learning_rate=0.06, random_state=rs)\n            m.fit(Xtr, ytr); p = m.predict(Xte)\n            accs.append(accuracy_score(yte, p)); f1s.append(f1_score(yte, p, average=\'macro\'))\n        unm = evaluate_feature_set(X_COLS, classes=SN_SUBTYPES, balanced_n=nm)\n        print(f\'unmatched, {nm}/class : acc {unm["acc"]:.3f}  macroF1 {unm["f1"]:.3f}\')\n        print(f\'z-MATCHED, {nm}/class : acc {np.mean(accs):.3f}  macroF1 {np.mean(f1s):.3f}\')\n        print(f\'\\nselection-driven component: {unm["acc"] - np.mean(accs):+.3f} accuracy\')\n    else:\n        print(\'too few objects for a matched control at this sample size.\')\n        print(\'Either raise N_ACQUIRE_SN_SUBTYPE, or read this as the result itself:\')\n        print(\'if the classes barely share a redshift range, then redshift IS close to\')\n        print(\'a class label in this sample, and the headline accuracy is partly a\')\n        print(\'measurement of how ZTF selects targets rather than of the transients.\')'), (34, '# ---- 3. the pairwise ceiling: which class pairs are actually separable? ---------\n#\n# For every pair of subtypes and every feature, the effect size |Cohen\'s d| between the\n# two class distributions. This is the diagnostic that explains the confusion matrix.\n#\n# Rule of thumb for an ideal single-feature separator:\n#   |d| = 2.0 -> ~84% pairwise accuracy;  1.0 -> ~69%;  0.4 -> ~58%;  0.15 -> ~53%.\nimport itertools\n\ndef cohens_d(a, b):\n    a = np.asarray(a, float); b = np.asarray(b, float)\n    a = a[np.isfinite(a)]; b = b[np.isfinite(b)]\n    if len(a) < 3 or len(b) < 3:\n        return np.nan\n    sa, sb = a.std(ddof=1), b.std(ddof=1)\n    pooled = np.sqrt((sa ** 2 + sb ** 2) / 2)\n    return abs(a.mean() - b.mean()) / pooled if pooled > 0 else np.nan\n\nd5 = feature_df_full[feature_df_full[\'label\'].isin(SN_SUBTYPES)]\nrows = []\nfor A, B in itertools.combinations(SN_SUBTYPES, 2):\n    ds = {c: cohens_d(d5[d5[\'label\'] == A][c], d5[d5[\'label\'] == B][c]) for c in X_COLS}\n    finite = {k: v for k, v in ds.items() if np.isfinite(v)}\n    if not finite:\n        continue\n    bf = max(finite, key=finite.get)\n    rows.append({\'pair\': f\'{A} vs {B}\', \'best feature\': bf, \'max |d|\': finite[bf],\n                 \'median |d| over all features\': float(np.nanmedian(list(finite.values())))})\npairs = pd.DataFrame(rows).sort_values(\'max |d|\', ascending=False)\nprint(pairs.round(2).to_string(index=False))\n\nworst = pairs.iloc[-1]\nprint(f"\\nHardest pair on your data: {worst[\'pair\']} - best single feature "\n      f"\'{worst[\'best feature\']}\' at |d| = {worst[\'max |d|\']:.2f}")\nprint("""\nOn the full ZTF BTS catalogue the answer is SN_Ib vs SN_Ic at |d| = 0.39, with every\nother catalogue axis below 0.15 - median z 0.0355 vs 0.0346, peak absolute magnitude\n-17.50 vs -17.79, rise 11.5 vs 11.0 d, fade 11.5 vs 12.1 d, duration 24.5 vs 23.1 d.\nTwo classes that overlap that completely cap the 5-way scheme in the low-to-mid 70s\nno matter what model is used, because what separates them - a helium absorption line -\nis spectroscopic and is not present in g and r photometry.\n\nThat is why PLAsTiCC ships a single SNIbc class and Superphot+ uses SN Ib/c. The\nscheme comparison below quantifies exactly what merging them buys on YOUR data.\n""")\n\nfig, ax = plt.subplots(figsize=(7, 4))\np = pairs.iloc[::-1]\nax.barh(p[\'pair\'], p[\'max |d|\'], color=[\'#E15759\' if v < 0.5 else \'#4C78A8\' for v in p[\'max |d|\']])\nfor x, lbl in [(0.4, \'0.4 ~ 58%\'), (1.0, \'1.0 ~ 69%\'), (2.0, \'2.0 ~ 84%\')]:\n    ax.axvline(x, ls=\'--\', c=\'grey\', lw=0.8)\n    ax.text(x, -0.7, lbl, fontsize=7, ha=\'center\', color=\'grey\')\nax.set_xlabel("best single-feature |Cohen\'s d|")\nax.set_title(\'Which subtype pairs are separable at all\')\nplt.tight_layout(); plt.show()'), (36, '# ---- 3. learning curve on your own data ---------------------------------------\ndef learning_curve_balanced(classes, sizes, n_repeats=10, seed=0):\n    d = feature_df_full[feature_df_full[\'label\'].isin(classes)]\n    avail = d[\'label\'].value_counts().min()\n    out = []\n    for n in sizes:\n        if n > avail:\n            break\n        accs, f1s = [], []\n        for rep in range(n_repeats):\n            rs = seed + rep * 97\n            dd = pd.concat([d[d[\'label\'] == c].sample(n, random_state=rs) for c in classes])\n            Xtr, Xte, ytr, yte = train_test_split(dd[X_COLS].values, dd[\'label\'].values,\n                                                  test_size=0.25, stratify=dd[\'label\'].values,\n                                                  random_state=rs)\n            sc = StandardScaler(); Xtr = sc.fit_transform(Xtr); Xte = sc.transform(Xte)\n            m = HistGradientBoostingClassifier(max_iter=400, learning_rate=0.06, random_state=rs)\n            m.fit(Xtr, ytr); p = m.predict(Xte)\n            accs.append(accuracy_score(yte, p)); f1s.append(f1_score(yte, p, average=\'macro\'))\n        out.append({\'n_per_class\': n, \'accuracy\': np.mean(accs), \'sd\': np.std(accs),\n                    \'macro F1\': np.mean(f1s)})\n    return pd.DataFrame(out)\n\n# 15 is the smallest size where a 75/25 split still leaves >=3 training objects\n# per class; below that the curve reports fold noise, not learning.\nsizes = [15, 20, 25, 30, 50, 80, 120, 200, 300]\nlc5 = learning_curve_balanced(SN_SUBTYPES, sizes)\nprint(\'5-way learning curve on this dataset:\'); print(lc5.round(3).to_string(index=False))\n\nif len(lc5) >= 3:\n    fig, ax = plt.subplots(figsize=(7, 4))\n    ax.errorbar(lc5[\'n_per_class\'], lc5[\'accuracy\'], yerr=lc5[\'sd\'],\n                marker=\'o\', capsize=3, label=\'5-way accuracy\')\n    ax.axhspan(0.85, 0.90, color=\'#59A14F\', alpha=0.18, label=\'85-90% target\')\n    ax.axvline(N_HEADLINE_SN_SUBTYPE, ls=\'--\', c=\'grey\', label=f\'headline ({N_HEADLINE_SN_SUBTYPE}/class)\')\n    ax.set_xlabel(\'objects per subtype\'); ax.set_ylabel(\'accuracy\'); ax.set_ylim(0, 1)\n    ax.legend(); ax.set_title(\'How much does more data buy?\')\n    plt.tight_layout(); plt.show()\n\nprint("""\nReading the curve: if it is still rising at the right-hand edge, more data is the\nbinding constraint and raising N_PER_SN_SUBTYPE is worth the runtime. If it has\nflattened well below the target band, the ceiling is the information content of\ntwo-band photometry, and the honest move is to report the 4-way or 3-way scheme\nalongside the 5-way one rather than to keep collecting.\n""")'), (39, "def cascade_predict(m1, m2, X_raw, sc1, sc2, le1, le2, sn_coarse='SNe'):\n    coarse = le1.inverse_transform(m1.predict(sc1.transform(X_raw)))\n    final = np.array(coarse, dtype=object)\n    mask = coarse == sn_coarse\n    if mask.any():\n        final[mask] = le2.inverse_transform(m2.predict(sc2.transform(X_raw[mask])))\n    return {'coarse_pred': coarse, 'final_pred': final, 'sne_mask': mask}\n\n\ndef run_stage2_option_b(split, stage1, stage2a, sn_subtype_labels, verbose=True):\n    df, X = split['feature_df'], split['X']\n    ite = split['idx_test']; Xte = X[ite]\n    true_fine = df['label'].iloc[ite].values\n    is_sn = np.isin(true_fine, sn_subtype_labels)\n    rows, results = [], {}\n    for name in stage1['models']:\n        r = cascade_predict(stage1['models'][name], stage2a['models'][name], Xte,\n                            stage1['scaler'], stage2a['scaler'], stage1['le'], stage2a['le'])\n        final, coarse, mask = r['final_pred'], r['coarse_pred'], r['sne_mask']\n        e2e = float((final == true_fine).mean())\n        routed = is_sn & (coarse == 'SNe')\n        cond = float((final[routed] == true_fine[routed]).mean()) if routed.any() else np.nan\n        comp = float((final[is_sn] == true_fine[is_sn]).mean()) if is_sn.any() else np.nan\n        rows.append({'Model': name, 'End-to-end (all 8 classes)': e2e,\n                     'Conditional (correctly-routed SNe)': cond,\n                     'Option-A-comparable (true SNe)': comp,\n                     'Non-SNe mis-routed in': int((~is_sn & mask).sum()),\n                     'True SNe lost before Stage 2': int((is_sn & ~mask).sum())})\n        results[name] = r\n        if verbose:\n            print(f'[Stage 2 / B] {name:<22} end-to-end {e2e:.3f} | conditional {cond:.3f} | '\n                  f'comparable {comp:.3f}')\n    return results, pd.DataFrame(rows)\n\nstage2b, stage2b_results = run_stage2_option_b(split, stage1, stage2a, SN_SUBTYPES)\nstage2b_results.round(3)"), (40, "comparison = pd.DataFrame({\n    'Model': stage2a['results']['Model'],\n    'Option A (ground-truth routing)': stage2a['results']['Test accuracy'].values,\n    'Option A macro-F1': stage2a['results']['Macro F1'].values,\n    'Option B (comparable)': stage2b_results['Option-A-comparable (true SNe)'].values,\n    'Option B (conditional)': stage2b_results['Conditional (correctly-routed SNe)'].values,\n    'Option B (end-to-end, 8 classes)': stage2b_results['End-to-end (all 8 classes)'].values,\n})\ncomparison['Routing cost (A - B comparable)'] = (\n    comparison['Option A (ground-truth routing)'] - comparison['Option B (comparable)'])\nprint(comparison.round(3).to_string(index=False))"), (41, "# Permutation importance — one method that spans all five model families, computed on\n# held-out data, exactly as in v2.\nfrom sklearn.inspection import permutation_importance\n\ndef perm_importance_table(stage, x_cols, n_repeats=20, random_state=RANDOM_STATE):\n    out = {}\n    for name, model in stage['models'].items():\n        r = permutation_importance(model, stage['X_test'], stage['y_test'],\n                                   n_repeats=n_repeats, random_state=random_state,\n                                   scoring='accuracy')\n        out[name] = pd.Series(r.importances_mean, index=x_cols)\n    t = pd.DataFrame(out)\n    t.index.name = 'feature'\n    return t\n\nimp2 = perm_importance_table(stage2a, X_COLS)\nsummary2 = pd.DataFrame({'mean_perm_importance': imp2.mean(axis=1),\n                         'across_model_std': imp2.std(axis=1)}).sort_values(\n                             'mean_perm_importance', ascending=False)\nprint('Stage 2 — most discriminative features, averaged across the headline models:')\nprint(summary2.round(4).head(12).to_string())\n\nfig, ax = plt.subplots(figsize=(8, 5))\ntop = summary2.head(12).iloc[::-1]\nax.barh(top.index, top['mean_perm_importance'], xerr=top['across_model_std'], color='#4C78A8')\nax.set_xlabel('permutation importance (accuracy drop)')\nax.set_title('Stage 2: what the models actually use')\nplt.tight_layout(); plt.show()"), (42, "# Feature separability, the diagnostic that exposed the v2 feature set.\ndef separability(df, label_col, cols):\n    rows = []\n    for c in cols:\n        g = df.groupby(label_col)[c]\n        between = float(np.var(g.mean().values))\n        within = float(g.var().mean())\n        rows.append({'feature': c, 'between_class_var': between,\n                     'mean_within_class_var': within,\n                     'separation_ratio': between / within if within > 0 else np.nan})\n    return pd.DataFrame(rows).sort_values('separation_ratio', ascending=False)\n\nprint('Stage 2 feature separability (between-class var / within-class var):')\nsep2 = separability(feature_df[feature_df['label'].isin(SN_SUBTYPES)], 'label', X_COLS)\nprint(sep2.round(3).head(12).to_string(index=False))\nprint('\\nFor reference, v2 Stage 2 topped out at 0.399 (colour) with the next four '\n      'features at 0.059, 0.031, 0.008 and 0.008.')")]

_a_split, _a_stage1, _a_stage2a = v4_adapters(alerce_track)
if _a_stage2a is None:
    print('ALeRCE track has no Stage 2 model — diagnostics skipped on that track.')
else:
    _a_cols = list(alerce_track['stage1']['split']['feature_cols'])
    _a_view = v4_impute(alerce_track_frame(feature_df, real_only=True), _a_cols)
    run_v4_cells(_V4_DIAGNOSTIC_SOURCES, globals(), {
        'feature_df_full': _a_view,
        'feature_df': _a_view,
        'SN_SUBTYPES': ALERCE_SN_CLASSES,
        'X_COLS': _a_cols,
        'split': _a_split, 'stage1': _a_stage1, 'stage2a': _a_stage2a,
    }, label='ALeRCE track', min_per_class=8)
for _i, _why in TNS_SCHEME_ONLY.items():
    print(f'v4 cell {_i} not run on the ALeRCE track — {_why}')

# STEP 12 — Calibration  *(brief Phase 6B, Section 8)*

After de Soto et al. (2024, Fig. 11): does redshift make Stage 2 **more confident without
making it more correct**? Reported either way.

In [ ]:
# Section 8 — calibration diagnostic
# ---------------------------------------------------------------
# Inlined verbatim from btp5/calibration.py, which the repository's test suite
# (tests5/) exercises. Edit it there and rebuild this notebook with
# tools/build_notebook_v5.py rather than editing it here.
# ---------------------------------------------------------------

"""
Section 8 — prediction calibration, after de Soto et al. 2024 Figure 11.

The specific claim worth testing: adding redshift can make a model *more confident
without making it more correct*. v4 already has a redshift-ablation diagnostic; this
sharpens it by asking what redshift does to the confidence-vs-accuracy relationship
rather than only to the accuracy.
"""

import numpy as np
import pandas as pd


def calibration_curve(confidences, correct, n_bins=10):
    """
    Binned confidence vs observed fraction correct.

    Returns one row per non-empty bin. A perfectly calibrated model has
    fraction_correct == mean_confidence in every bin.
    """
    conf = np.asarray(confidences, float)
    ok = np.asarray(correct).astype(bool)
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    rows = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf >= lo) & (conf < hi if hi < 1.0 else conf <= hi)
        if not m.any():
            continue
        rows.append({'bin_low': lo, 'bin_high': hi, 'n': int(m.sum()),
                     'mean_confidence': float(conf[m].mean()),
                     'fraction_correct': float(ok[m].mean())})
    return pd.DataFrame(rows)


def expected_calibration_error(curve):
    """ECE: sample-weighted mean |confidence - accuracy| across bins."""
    if curve.empty:
        return float('nan')
    w = curve['n'] / curve['n'].sum()
    return float((w * (curve['mean_confidence'] - curve['fraction_correct']).abs()).sum())


def overconfidence(curve):
    """Signed mean (confidence - accuracy). Positive = overconfident."""
    if curve.empty:
        return float('nan')
    w = curve['n'] / curve['n'].sum()
    return float((w * (curve['mean_confidence'] - curve['fraction_correct'])).sum())


def calibration_for_model(model, X_test, y_test, n_bins=10):
    """Curve + summary stats for a fitted classifier that exposes predict_proba."""
    if not hasattr(model, 'predict_proba'):
        return None
    proba = model.predict_proba(X_test)
    conf = proba.max(axis=1)
    pred = model.classes_[proba.argmax(axis=1)]
    correct = (pred == np.asarray(y_test))
    curve = calibration_curve(conf, correct, n_bins=n_bins)
    return {'curve': curve,
            'ece': expected_calibration_error(curve),
            'overconfidence': overconfidence(curve),
            'mean_confidence': float(conf.mean()),
            'accuracy': float(correct.mean()),
            'n': int(len(conf))}


def compare_with_without_redshift(with_z, without_z, model_name):
    """
    The specific de Soto et al. claim, stated as a testable comparison and answered
    from the numbers rather than asserted.
    """
    if with_z is None or without_z is None:
        return f'{model_name}: calibration unavailable (no predict_proba).'
    d_acc = with_z['accuracy'] - without_z['accuracy']
    d_conf = with_z['mean_confidence'] - without_z['mean_confidence']
    msg = (f"**{model_name}.** With redshift: accuracy {with_z['accuracy']:.3f}, mean "
           f"confidence {with_z['mean_confidence']:.3f}, ECE {with_z['ece']:.3f}. "
           f"Without: accuracy {without_z['accuracy']:.3f}, confidence "
           f"{without_z['mean_confidence']:.3f}, ECE {without_z['ece']:.3f}. ")
    if d_conf > 0.02 and d_acc <= 0.02:
        msg += (f"Confidence rises {d_conf:+.3f} while accuracy moves only {d_acc:+.3f} "
                "— this reproduces de Soto et al. (2024, Fig. 11): redshift makes the "
                "model more confident without making it more correct.")
    elif d_conf > 0.02 and d_acc > 0.02:
        msg += (f"Both confidence ({d_conf:+.3f}) and accuracy ({d_acc:+.3f}) rise, so "
                "here redshift carries real information rather than only inflating "
                "confidence — the de Soto pattern does NOT reproduce in our data.")
    else:
        msg += ("Redshift does not materially change confidence, so the de Soto "
                "overconfidence pattern does not arise either way here.")
    return msg


def plot_calibration(curves, title, out_path=None):
    """Reliability diagram: one line per variant, diagonal = perfect calibration."""
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(5.5, 5.2))
    ax.plot([0, 1], [0, 1], ls='--', color='0.5', lw=1, label='perfectly calibrated')
    for name, c in curves.items():
        if c is None or c['curve'].empty:
            continue
        cv = c['curve']
        ax.plot(cv['mean_confidence'], cv['fraction_correct'], marker='o', label=name)
    ax.set_xlabel('mean predicted confidence')
    ax.set_ylabel('observed fraction correct')
    ax.set_title(title, fontsize=11)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.legend(fontsize=8)
    fig.tight_layout()
    if out_path:
        fig.savefig(out_path, dpi=140, bbox_inches='tight')
    return fig

In [ ]:
calibration_texts = []
s2 = tns_track['stage2']
if s2 is not None:
    sp2 = s2['split']
    z_idx = [sp2['feature_cols'].index(c) for c in ['redshift', 'peak_abs_mag']
             if c in sp2['feature_cols']]
    keep = [i for i in range(len(sp2['feature_cols'])) if i not in z_idx]
    curves = {}
    for name in HEADLINE_MODELS:
        with_z = calibration_for_model(s2['models'][name], sp2['X_test'], sp2['y_test'])
        m2 = build_models()[name]
        m2.fit(sp2['X_train'][:, keep], sp2['y_train'])
        without_z = calibration_for_model(m2, sp2['X_test'][:, keep], sp2['y_test'])
        curves[f'{name} (with z)'] = with_z
        curves[f'{name} (no z)'] = without_z
        calibration_texts.append(compare_with_without_redshift(with_z, without_z, name))
        print(calibration_texts[-1]); print()
    plot_calibration(curves, 'Stage 2 calibration, with and without redshift')
    plt.show()
else:
    print('Stage 2 unavailable — calibration skipped.')

# STEP 13 — Written summary  *(brief Phase 7)*

Generated from the objects computed above, so it cannot disagree with them.

In [ ]:
L = []
L.append('## Samples and pools\n')
L.append(feature_df['source'].value_counts().rename('rows').to_frame().to_markdown())
L.append('\n\n' + pool_counts.to_markdown(index=False))
if 'native_summary' in globals():
    L.append('\n\n**What ALeRCE offers above the confidence cut, per group:**\n\n'
             + native_summary.to_markdown(index=False))

L.append('\n\n## Headline results (Section 6.3)\n')
L.append(summary_table.round(3).to_markdown(index=False))
L.append('\n\nThese are scored against different truths on different samples and are not '
         'comparable as bare numbers; the agreement analysis below is the comparison.')

if aug_ablation is not None:
    L.append('\n\n## Augmentation (Section 5)\n')
    L.append(aug_ablation.round(3).to_markdown())
if augmentation_check:
    L.append(f"\n\nSynthetic copies fit {augmentation_check['median_residual_mag']:+.3f} mag "
             f"brighter than their parents predict (median; spread "
             f"{augmentation_check['residual_spread_mag']:.3f}). Negative values are the "
             'truncation bias of near-limit detection, which faint real objects share.')

L.append('\n\n## Section 3.1 — TDE cross-check\n')
L.append(tde_text)
if tde_table is not None:
    L.append('\n\n' + tde_table.round(2).to_markdown())

L.append('\n\n## Agreement (Section 6)\n')
for stage, reps in (('Stage 1', stage1_reports), ('Stage 2', stage2_reports)):
    L.append(f'\n### {stage}\n')
    for name, rep in reps.items():
        for sub, r in rep.get('subsets', {}).items():
            if 'text' in r:
                L.append('- ' + r['text'])
        L.append('- ' + contrast_subsets(rep))

if calibration_texts:
    L.append('\n\n## Calibration (Section 8)\n')
    L.extend('- ' + t for t in calibration_texts)

L.append('\n\n## External corroboration (Section 6.4)\n')
L.append(EXTERNAL_RESULTS.to_markdown(index=False))
L.append('\n\n' + TDE_SLSN_STATEMENT)

L.append('\n\n## Ground-truth caveat (Section 6.5)\n')
L.append(TNS_LABEL_NOISE_CAVEAT)

L.append('\n\n## Deferred items — limitations, with their reasons\n')
L.append('- **AllWISE infrared colour (Section 4.2) — not built.** ALeRCE\'s feature set '
         'includes AllWISE infrared colours, which this pipeline does not; this is a plausible '
         'avenue for closing part of the remaining Stage-1 TDE/AGN confusion, left for future '
         'work. It needs a separate cross-match and lacks host offset\'s direct link to that '
         'boundary.')
L.append('- **Colour variability over time (Section 4.3) — not built.** It would need feature '
         'extraction to track colour across epochs rather than at one. It is specifically why '
         'ALeRCE\'s TDE update works: TDEs stay blue while SNe redden. A bigger lift than 4.1.')
L.append('- **ParSNIP-style non-parametric features (Section 4.4) — deliberately not built.** '
         'The most strongly evidenced accuracy lever in the research log: across three groups '
         'and two surveys, ParSNIP/SuperRAENN beat Villar-family fits by 8-10 F1 points every '
         'time tested. Not built because it requires training a variational autoencoder, a far '
         'heavier task than anything here, and the evidence comes from samples 15-1000x larger '
         'than ours (6,061 for Superphot+, 6,123 for ParSNIP, against our low hundreds). For '
         'scale: Superphot+ reached 83% / 0.61 F1 on 5-way SNe with N=6061; SuperRAENN 87% / '
         '0.66 purity with N~2885.')
L.append('- **SNANA simulation (Section 5) — deferred.** Aleo et al. 2023 / YSE-DR1 use '
         '~60,000 objects per class. Mature templates exist for Ia/II/Ibc but not obviously '
         'for SLSN or TDE, exactly the classes that need help.')
L.append('- **ALeRCE\'s total feature count** is cited in the research log as unverified. It '
         'is not quoted anywhere in this notebook, and no comparison here depends on it.')

L.append('\n\n## Structural limits found while building v5\n')
L.append('- ALeRCE\'s TDE class exists only in lc_classifier_BHRF_forced_phot 2.1.0; the '
         'classic lc_classifier has none. Verified live.')
L.append('- The BHRF transient branch separates SNIIn and calls the stripped-envelope class '
         'SESN. The IIn merge here rests on our own sample size, not on ALeRCE\'s taxonomy.')
L.append('- The ALeRCE track cannot contain stellar flares (TESS objects have no ZTF alerts). '
         'Its CV/Nova + YSO objects stand in for that class and are not the same population.')
L.append('- ALeRCE-native objects without a TNS spectrum have no redshift; they were acquired '
         'with only that part of v4\'s gate relaxed, and are excluded from the TNS track.')
L.append('- Ib and Ic stay separate on the TNS side and are merged only to compare with '
         'ALeRCE, which never separates them — the pair the literature finds least separable.')
L.append('- v4\'s peak_abs_mag is derived from the Villar A parameter, which is degenerate with '
         't0 and tau_rise: under an identical noise realization it scattered 0.255 mag against '
         '0.035 mag for the fitted curve\'s actual peak. Preserved, as v4 behaviour; flagged as '
         'the cheapest improvement available to v4\'s features.')

summary_md = '\n'.join(L)
with open(os.path.join(FEATURE_DIR, 'v5_results_summary.md'), 'w') as f:
    f.write(summary_md)
print('written to', os.path.join(FEATURE_DIR, 'v5_results_summary.md'))

from IPython.display import Markdown
Markdown(summary_md)

In [ ]:
# Everything needed to reproduce a number without re-running acquisition.
feature_df.to_csv(os.path.join(FEATURE_DIR, 'v5_feature_table.csv'), index=False)
summary_table.to_csv(os.path.join(FEATURE_DIR, 'v5_headline_results.csv'), index=False)
pool_counts.to_csv(os.path.join(FEATURE_DIR, 'v5_pool_counts.csv'), index=False)
for stage, reps in (('stage1', stage1_reports), ('stage2', stage2_reports)):
    for name, rep in reps.items():
        tag = name.replace(' ', '_')
        if 'expected_matrix' in rep:
            rep['expected_matrix'].to_csv(os.path.join(FEATURE_DIR, f'v5_{stage}_{tag}_expected.csv'))
        for sub, r in rep.get('subsets', {}).items():
            if 'actual_matrix' in r:
                s = 'spec' if sub == SUBSET_SPEC else 'phot'
                r['actual_matrix'].to_csv(os.path.join(FEATURE_DIR, f'v5_{stage}_{tag}_actual_{s}.csv'))
print('saved to', FEATURE_DIR)